#Cargar bases de datos y crear las de actividades económicas
IGAE y las actividades van de Ene/2002 a Dic/2024


In [1]:
import pandas as pd

In [2]:
pip install dcor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 9.5 MB/s eta 0:00:00


In [3]:
Temperaturas = pd.read_csv("Temperaturas promedio.csv", skiprows = 2)

In [4]:
IGAE = pd.read_excel(
    "IGAE_2.xlsx",
    sheet_name=0
)

In [5]:
IGAE_serie = IGAE.iloc[6]

In [6]:
len(IGAE_serie)

443

In [7]:
IGAE_2002 = IGAE_serie.iloc[118:]

In [8]:
IGAE_2002 = IGAE_2002.iloc[:-10]

In [9]:
len(IGAE_2002)

315

In [10]:
Primarias = IGAE.iloc[7]


In [11]:
Primarias = Primarias[:-10]

In [12]:
Primarias = Primarias[118:]

In [13]:
Secundarias = IGAE.iloc[10]

In [14]:
Secundarias = Secundarias[118:]

In [15]:
Secundarias = Secundarias[:-10]

In [16]:
Terciarias = IGAE.iloc[15]

In [17]:
Terciarias = Terciarias[118:]

In [18]:
Terciarias = Terciarias[:-10]

#Procesamiento de IGAE y actividades económicas

In [19]:
import pandas as pd

def drop_every_13th(series: pd.Series) -> pd.Series:
    """
    Elimina cada 13° valor (13, 26, 39, ...) de una Series,
    devolviendo la Series corrida sin esos elementos.
    """
    # posiciones que queremos conservar: aquellas cuyo índice entero no es múltiplo de 13
    mask = [(i + 1) % 13 != 0 for i in range(len(series))]
    return series.iloc[mask].reset_index(drop=True)


In [20]:
Primarias_clean   = drop_every_13th(Primarias)
Secundarias_clean = drop_every_13th(Secundarias)
Terciarias_clean  = drop_every_13th(Terciarias)
IGAE_2002_clean   = drop_every_13th(IGAE_2002)

# Preprocesamiento Temperatura

In [21]:
Temperaturas_Serie = Temperaturas.iloc[0]

In [22]:
Temperaturas_Serie = Temperaturas_Serie[1:]

In [23]:
Temperaturas_Serie = Temperaturas_Serie.dropna()

#Expandir Exógenas

In [24]:
print("Temperaturas_Serie :", len(Temperaturas_Serie))
print("Primarias_clean    :", len(Primarias_clean))
print("Secundarias_clean  :", len(Secundarias_clean))
print("Terciarias_clean   :", len(Terciarias_clean))
print("IGAE_2002_clean    :", len(IGAE_2002_clean))

Temperaturas_Serie : 291
Primarias_clean    : 291
Secundarias_clean  : 291
Terciarias_clean   : 291
IGAE_2002_clean    : 291


In [25]:
import pandas as pd

FECHA_INICIO = "2002-01-01"

series = {
    "Temperaturas": Temperaturas_Serie,
    "Primarias": Primarias_clean,
    "Secundarias": Secundarias_clean,
    "Terciarias": Terciarias_clean,
    "IGAE": IGAE_2002_clean
}

for nombre, serie in series.items():

    fechas = pd.date_range(
        start=FECHA_INICIO,
        periods=len(serie),
        freq="MS"
    )

    print(
        f"{nombre:15s} | "
        f"len={len(serie):3d} | "
        f"{fechas.min().date()} -> {fechas.max().date()}"
    )

Temperaturas    | len=291 | 2002-01-01 -> 2026-03-01
Primarias       | len=291 | 2002-01-01 -> 2026-03-01
Secundarias     | len=291 | 2002-01-01 -> 2026-03-01
Terciarias      | len=291 | 2002-01-01 -> 2026-03-01
IGAE            | len=291 | 2002-01-01 -> 2026-03-01


In [26]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================

FECHA_INICIO = "2002-01-01"

FECHA_CORTE_INICIO = "2019-01-01"
FECHA_CORTE_FIN    = "2026-03-01" #Aquí lo moví

# =========================================================
# FUNCION
# =========================================================

def expandir_exogena(serie_original, nombre="serie"):

    serie_original = pd.to_numeric(
        pd.Series(serie_original),
        errors="coerce"
    )

    fechas = pd.date_range(
        start=FECHA_INICIO,
        periods=len(serie_original),
        freq="MS"
    )

    df = pd.DataFrame({
        "fecha_mensual": fechas,
        "valor": serie_original.values
    })

    # ----------------------------------
    # CORTE
    # ----------------------------------

    df = df[
        (df["fecha_mensual"] >= FECHA_CORTE_INICIO)
        &
        (df["fecha_mensual"] <= FECHA_CORTE_FIN)
    ].copy()

    df = df.dropna(subset=["valor"])

    # ----------------------------------
    # EXPANSION HORARIA
    # ----------------------------------

    filas = []

    for _, row in df.iterrows():

        fecha_mes = row["fecha_mensual"]
        valor = row["valor"]

        for dia in range(fecha_mes.days_in_month):

            fecha_actual = fecha_mes + pd.Timedelta(days=dia)

            for hora in range(1, 25):

                filas.append({
                    "fecha": fecha_actual.normalize(),
                    "hora": hora,
                    "valor": valor
                })

    df_horario = pd.DataFrame(filas)

    print(
        f"{nombre:15s} | "
        f"{df_horario['fecha'].min().date()} -> "
        f"{df_horario['fecha'].max().date()} | "
        f"{len(df_horario):,} filas"
    )

    return df_horario

In [27]:
Temperaturas_H = expandir_exogena(
    Temperaturas_Serie,
    "Temperaturas"
)

Primarias_H = expandir_exogena(
    Primarias_clean,
    "Primarias"
)

Secundarias_H = expandir_exogena(
    Secundarias_clean,
    "Secundarias"
)

Terciarias_H = expandir_exogena(
    Terciarias_clean,
    "Terciarias"
)

IGAE_H = expandir_exogena(
    IGAE_2002_clean,
    "IGAE"
)

Temperaturas    | 2019-01-01 -> 2026-03-31 | 63,528 filas
Primarias       | 2019-01-01 -> 2026-03-31 | 63,528 filas
Secundarias     | 2019-01-01 -> 2026-03-31 | 63,528 filas
Terciarias      | 2019-01-01 -> 2026-03-31 | 63,528 filas
IGAE            | 2019-01-01 -> 2026-03-31 | 63,528 filas


In [28]:
for nombre, df in {
    "Temperaturas": Temperaturas_H,
    "Primarias": Primarias_H,
    "Secundarias": Secundarias_H,
    "Terciarias": Terciarias_H,
    "IGAE": IGAE_H
}.items():

    print(
        f"{nombre:15s} | "
        f"{len(df):,} filas"
    )

Temperaturas    | 63,528 filas
Primarias       | 63,528 filas
Secundarias     | 63,528 filas
Terciarias      | 63,528 filas
IGAE            | 63,528 filas


#Quitar horas extras (Yei!)

In [29]:
import os
import pandas as pd

# =========================================================
# CONFIG
# =========================================================

CARPETA = "/content"

REGIONES = [
    "BCA",
    "CEN",
    "NES",
    "NOR",
    "NTE",
    "OCC",
    "ORI",
    "PEN",
]

TIPOS_EXOGENAS = [
    "GEN",
    "IMP",
    "EXP",
]

# Si es True, reemplaza los CSV originales.
# Si es False, guarda archivos con sufijo "_limpio".
REEMPLAZAR_ORIGINALES = True


# =========================================================
# DETECTAR COLUMNA HORA
# =========================================================

def detectar_columna_hora(df):

    columnas = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    if "hora" not in columnas:
        raise ValueError(
            "No se encontró una columna llamada "
            "'hora' o 'Hora'."
        )

    return columnas["hora"]


# =========================================================
# LIMPIAR HORA 25
# =========================================================

def eliminar_hora_25(
    df,
    nombre_archivo
):

    df = df.copy()

    hora_col = detectar_columna_hora(
        df
    )

    df[hora_col] = pd.to_numeric(
        df[hora_col],
        errors="coerce"
    )

    filas_originales = len(df)

    filas_hora_25 = int(
        (df[hora_col] == 25).sum()
    )

    # Conservar únicamente horas 1–24.
    # También elimina horas inválidas o NaN.
    df_limpio = df[
        df[hora_col].between(
            1,
            24,
            inclusive="both"
        )
    ].copy()

    filas_finales = len(
        df_limpio
    )

    eliminadas_totales = (
        filas_originales
        - filas_finales
    )

    print(
        f"{nombre_archivo:25s} | "
        f"originales: {filas_originales:,} | "
        f"hora 25: {filas_hora_25:,} | "
        f"eliminadas totales: {eliminadas_totales:,} | "
        f"finales: {filas_finales:,}"
    )

    return df_limpio


# =========================================================
# PROCESAR UN ARCHIVO
# =========================================================

def procesar_archivo(
    ruta
):

    if not os.path.exists(
        ruta
    ):

        print(
            f"⚠️ No se encontró: {ruta}"
        )

        return

    df = pd.read_csv(
        ruta,
        encoding="utf-8-sig"
    )

    df_limpio = eliminar_hora_25(
        df,
        os.path.basename(
            ruta
        )
    )

    if REEMPLAZAR_ORIGINALES:

        ruta_salida = ruta

    else:

        nombre_base, extension = (
            os.path.splitext(
                ruta
            )
        )

        ruta_salida = (
            f"{nombre_base}"
            f"_limpio"
            f"{extension}"
        )

    df_limpio.to_csv(
        ruta_salida,
        index=False,
        encoding="utf-8-sig"
    )


# =========================================================
# ARCHIVOS REGIONALES LONG
# =========================================================

print(
    "=" * 90
)

print(
    "LIMPIANDO ARCHIVOS REGIONALES LONG"
)

print(
    "=" * 90
)

for region in REGIONES:

    ruta = os.path.join(
        CARPETA,
        f"{region}_long.csv"
    )

    procesar_archivo(
        ruta
    )


# =========================================================
# ARCHIVOS GEN / IMP / EXP
# =========================================================

print(
    "\n"
    + "=" * 90
)

print(
    "LIMPIANDO EXOGENAS REGIONALES"
)

print(
    "=" * 90
)

for region in REGIONES:

    for tipo in TIPOS_EXOGENAS:

        ruta = os.path.join(
            CARPETA,
            f"{region}_{tipo}.csv"
        )

        procesar_archivo(
            ruta
        )


# =========================================================
# VALIDACION FINAL
# =========================================================

print(
    "\n"
    + "=" * 90
)

print(
    "VALIDACION FINAL"
)

print(
    "=" * 90
)

archivos_validar = []

for region in REGIONES:

    archivos_validar.append(
        os.path.join(
            CARPETA,
            f"{region}_long.csv"
        )
    )

    for tipo in TIPOS_EXOGENAS:

        archivos_validar.append(
            os.path.join(
                CARPETA,
                f"{region}_{tipo}.csv"
            )
        )


problemas = []

for ruta in archivos_validar:

    if not os.path.exists(
        ruta
    ):
        continue

    df = pd.read_csv(
        ruta,
        encoding="utf-8-sig"
    )

    try:

        hora_col = detectar_columna_hora(
            df
        )

        hora = pd.to_numeric(
            df[hora_col],
            errors="coerce"
        )

        valores_invalidos = sorted(
            hora[
                ~hora.between(
                    1,
                    24,
                    inclusive="both"
                )
            ]
            .dropna()
            .unique()
            .tolist()
        )

        if valores_invalidos:

            problemas.append(
                (
                    os.path.basename(
                        ruta
                    ),
                    valores_invalidos
                )
            )

    except Exception as e:

        problemas.append(
            (
                os.path.basename(
                    ruta
                ),
                str(e)
            )
        )


if len(problemas) == 0:

    print(
        "✅ Todos los archivos existentes "
        "tienen únicamente horas 1–24."
    )

else:

    print(
        "⚠️ Se encontraron problemas:"
    )

    for archivo, detalle in problemas:

        print(
            f"   {archivo}: "
            f"{detalle}"
        )

LIMPIANDO ARCHIVOS REGIONALES LONG
BCA_long.csv              | originales: 64,799 | hora 25: 7 | eliminadas totales: 7 | finales: 64,792
CEN_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
NES_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
NOR_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
NTE_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
OCC_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
ORI_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796
PEN_long.csv              | originales: 64,800 | hora 25: 4 | eliminadas totales: 4 | finales: 64,796

LIMPIANDO EXOGENAS REGIONALES
⚠️ No se encontró: /content/BCA_GEN.csv
⚠️ No se encontró: /content/BCA_IMP.csv
⚠️ No se encontró: /content/BCA_EXP.cs

#Mochar CEN y medir correlaciones

In [ ]:
import pandas as pd
import numpy as np

# ============================================
# CONFIG
# ============================================

WINDOW = 30 * 24  # 30 dias horarios

# ============================================
# RECORTAR CEN
# ============================================

CEN_corr = CEN.iloc[:-1008].copy()

print("Filas originales CEN:", len(CEN))
print("Filas CEN recortado :", len(CEN_corr))

# ============================================
# EXOGENAS
# ============================================

EXOGENAS = {

    "Temperaturas": Temperaturas_H,
    "Primarias": Primarias_H,
    "Secundarias": Secundarias_H,
    "Terciarias": Terciarias_H,
    "IGAE": IGAE_H

}

# ============================================
# FUNCION
# ============================================

def analizar_objetivo(
    serie_objetivo,
    nombre_objetivo
):

    rows_spear = []
    rows_kend = []
    rows_roll = []

    serie_objetivo = pd.to_numeric(
        serie_objetivo,
        errors="coerce"
    )

    print("\n")
    print("=" * 60)
    print(nombre_objetivo)
    print("=" * 60)

    for nombre_exogena, exogena in EXOGENAS.items():

        x = pd.to_numeric(
            exogena["valor"],
            errors="coerce"
        )

        print(
            f"{nombre_exogena}: "
            f"{len(serie_objetivo)} vs {len(x)}"
        )

        # ----------------------------------
        # VERIFICAR LONGITUD
        # ----------------------------------

        n = min(
            len(serie_objetivo),
            len(x)
        )

        y = serie_objetivo.iloc[:n].reset_index(drop=True)
        x = x.iloc[:n].reset_index(drop=True)

        mask = y.notna() & x.notna()

        y = y[mask]
        x = x[mask]

        if len(y) < WINDOW + 100:
            print("Muy corta")
            continue

        # ----------------------------------
        # SPEARMAN
        # ----------------------------------

        spear = y.corr(
            x,
            method="spearman"
        )

        # ----------------------------------
        # KENDALL
        # ----------------------------------

        kend = y.corr(
            x,
            method="kendall"
        )

        rows_spear.append([

            nombre_objetivo,
            nombre_exogena,
            spear

        ])

        rows_kend.append([

            nombre_objetivo,
            nombre_exogena,
            kend

        ])

        # ----------------------------------
        # ROLLING
        # ----------------------------------

        rolling_corr = (
            y.rolling(WINDOW)
             .corr(x)
        )

        for i, valor in enumerate(rolling_corr):

            if pd.notna(valor):

                rows_roll.append([

                    i,
                    nombre_objetivo,
                    nombre_exogena,
                    valor,
                    WINDOW

                ])

        print(
            f"OK {nombre_exogena} | "
            f"Spearman={spear:.4f} | "
            f"Kendall={kend:.4f}"
        )

    # =======================================
    # DATAFRAMES
    # =======================================

    df_spear = pd.DataFrame(

        rows_spear,

        columns=[
            "objetivo",
            "variable",
            "valor"
        ]

    )

    df_kend = pd.DataFrame(

        rows_kend,

        columns=[
            "objetivo",
            "variable",
            "valor"
        ]

    )

    df_roll = pd.DataFrame(

        rows_roll,

        columns=[
            "indice",
            "objetivo",
            "variable",
            "valor",
            "ventana"
        ]

    )

    # =======================================
    # SAVE
    # =======================================

    prefix = nombre_objetivo.lower()

    df_spear.to_csv(
        f"{prefix}_spearman.csv",
        index=False
    )

    df_kend.to_csv(
        f"{prefix}_kendall.csv",
        index=False
    )

    df_roll.to_csv(
        f"{prefix}_rolling.csv",
        index=False
    )

    print("\nCSV GENERADOS")

    print(f"{prefix}_spearman.csv")
    print(f"{prefix}_kendall.csv")
    print(f"{prefix}_rolling.csv")

    return (
        df_spear,
        df_kend,
        df_roll
    )

# ============================================
# DEMANDA
# ============================================

dem_spear, dem_kend, dem_roll = analizar_objetivo(

    CEN_corr[
        "Estimacion de Demanda por Balance (MWh)"
    ],

    "Demanda"

)

# ============================================
# GENERACION
# ============================================

gen_spear, gen_kend, gen_roll = analizar_objetivo(

    CEN_corr[
        "Generacion (MWh)"
    ],

    "Generacion"

)

NameError: name 'CEN' is not defined

#Long

In [30]:
import pandas as pd
import os

# ==========================================
# CARGAR REGIONES
# ==========================================

ruta = "/content/"

regiones = {}

for region in [
    "BCA",
    "CEN",
    "NES",
    "NOR",
    "NTE",
    "OCC",
    "ORI",
    "PEN"
]:

    archivo = os.path.join(
        ruta,
        f"{region}_long.csv"
    )

    df = pd.read_csv(archivo)

    df["fecha"] = pd.to_datetime(
        df["fecha"],
        errors="coerce"
    )

    regiones[region] = df

# ==========================================
# RESUMEN
# ==========================================

resumen = []

for region, df in regiones.items():

    fecha_min = df["fecha"].min()
    fecha_max = df["fecha"].max()

    filas = len(df)

    fechas_unicas = (
        df["fecha"]
        .dt.normalize()
        .nunique()
    )

    filas_esperadas = (
        fechas_unicas * 24
    )

    resumen.append([
        region,
        fecha_min,
        fecha_max,
        filas,
        fechas_unicas,
        filas_esperadas,
        filas - filas_esperadas
    ])

resumen = pd.DataFrame(
    resumen,
    columns=[
        "region",
        "fecha_inicio",
        "fecha_fin",
        "filas",
        "fechas_unicas",
        "filas_esperadas_24h",
        "diferencia"
    ]
)

resumen = resumen.sort_values(
    "region"
)

print(resumen)

  region fecha_inicio  fecha_fin  filas  fechas_unicas  filas_esperadas_24h  \
0    BCA   2019-01-01 2026-05-23  64792           2700                64800   
1    CEN   2019-01-01 2026-05-23  64796           2700                64800   
2    NES   2019-01-01 2026-05-23  64796           2700                64800   
3    NOR   2019-01-01 2026-05-23  64796           2700                64800   
4    NTE   2019-01-01 2026-05-23  64796           2700                64800   
5    OCC   2019-01-01 2026-05-23  64796           2700                64800   
6    ORI   2019-01-01 2026-05-23  64796           2700                64800   
7    PEN   2019-01-01 2026-05-23  64796           2700                64800   

   diferencia  
0          -8  
1          -4  
2          -4  
3          -4  
4          -4  
5          -4  
6          -4  
7          -4  


In [31]:
import pandas as pd
import os

# ==========================================
# CARPETA DE SALIDA
# ==========================================

OUTPUT_DIR = "/content"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# FUNCION
# ==========================================

def crear_series(df, nombre):

    df = df.copy()

    # ------------------------------
    # detectar nombres de columnas
    # ------------------------------

    fecha_col = next(
        c for c in df.columns
        if c.lower() == "fecha"
    )

    hora_col = next(
        c for c in df.columns
        if c.lower() == "hora"
    )

    gen_col = next(
        c for c in df.columns
        if "generacion" in c.lower()
    )

    exp_col = next(
        c for c in df.columns
        if "exportacion" in c.lower()
    )

    imp_col = next(
        c for c in df.columns
        if "importacion" in c.lower()
    )

    dem_col = next(
        c for c in df.columns
        if "estimacion de demanda" in c.lower()
    )

    # ------------------------------
    # limpieza
    # ------------------------------

    df[fecha_col] = pd.to_datetime(
        df[fecha_col],
        errors="coerce"
    )

    df[hora_col] = pd.to_numeric(
        df[hora_col],
        errors="coerce"
    )

    # ------------------------------
    # series
    # ------------------------------

    generacion = (
        df.groupby(
            [fecha_col, hora_col],
            as_index=False
        )[gen_col]
        .max()
        .rename(
            columns={
                fecha_col: "fecha",
                hora_col: "hora",
                gen_col: "valor"
            }
        )
    )

    exportacion = (
        df.groupby(
            [fecha_col, hora_col],
            as_index=False
        )[exp_col]
        .max()
        .rename(
            columns={
                fecha_col: "fecha",
                hora_col: "hora",
                exp_col: "valor"
            }
        )
    )

    importacion = (
        df.groupby(
            [fecha_col, hora_col],
            as_index=False
        )[imp_col]
        .max()
        .rename(
            columns={
                fecha_col: "fecha",
                hora_col: "hora",
                imp_col: "valor"
            }
        )
    )

    demanda = (
        df.groupby(
            [fecha_col, hora_col],
            as_index=False
        )[dem_col]
        .max()
        .rename(
            columns={
                fecha_col: "fecha",
                hora_col: "hora",
                dem_col: "valor"
            }
        )
    )

    return (
        generacion,
        exportacion,
        importacion,
        demanda
    )


# ==========================================
# CREAR SERIES
# ==========================================

series = {}

for region, df in regiones.items():

    gen, exp, imp, dem = crear_series(
        df,
        region
    )

    series[f"{region}_GEN"] = gen
    series[f"{region}_EXP"] = exp
    series[f"{region}_IMP"] = imp
    series[f"{region}_DEM"] = dem


# ==========================================
# GUARDAR CSVs EN /content
# ==========================================

for nombre, df in series.items():

    salida = os.path.join(
        OUTPUT_DIR,
        f"{nombre}.csv"
    )

    df.to_csv(
        salida,
        index=False,
        encoding="utf-8-sig"
    )


# ==========================================
# RESUMEN
# ==========================================

print("\nSeries generadas:\n")

for nombre, df in sorted(series.items()):

    print(
        f"{nombre:10s} | "
        f"{len(df):,} filas"
    )

print("\nArchivos guardados en:")

for nombre in sorted(series.keys()):
    print(f"/content/{nombre}.csv")


Series generadas:

BCA_DEM    | 64,792 filas
BCA_EXP    | 64,792 filas
BCA_GEN    | 64,792 filas
BCA_IMP    | 64,792 filas
CEN_DEM    | 64,796 filas
CEN_EXP    | 64,796 filas
CEN_GEN    | 64,796 filas
CEN_IMP    | 64,796 filas
NES_DEM    | 64,796 filas
NES_EXP    | 64,796 filas
NES_GEN    | 64,796 filas
NES_IMP    | 64,796 filas
NOR_DEM    | 64,796 filas
NOR_EXP    | 64,796 filas
NOR_GEN    | 64,796 filas
NOR_IMP    | 64,796 filas
NTE_DEM    | 64,796 filas
NTE_EXP    | 64,796 filas
NTE_GEN    | 64,796 filas
NTE_IMP    | 64,796 filas
OCC_DEM    | 64,796 filas
OCC_EXP    | 64,796 filas
OCC_GEN    | 64,796 filas
OCC_IMP    | 64,796 filas
ORI_DEM    | 64,796 filas
ORI_EXP    | 64,796 filas
ORI_GEN    | 64,796 filas
ORI_IMP    | 64,796 filas
PEN_DEM    | 64,796 filas
PEN_EXP    | 64,796 filas
PEN_GEN    | 64,796 filas
PEN_IMP    | 64,796 filas

Archivos guardados en:
/content/BCA_DEM.csv
/content/BCA_EXP.csv
/content/BCA_GEN.csv
/content/BCA_IMP.csv
/content/CEN_DEM.csv
/content/CEN_EXP.cs

#Correlaciones

In [ ]:
import pandas as pd
import os

FECHA_CORTE = pd.Timestamp("2026-03-31")

for archivo in [

    "BCA_long.csv",
    "BCS_long.csv",
    "CEN_long.csv",
    "NES_long.csv",
    "NOR_long.csv",
    "NTE_long.csv",
    "OCC_long.csv",
    "ORI_long.csv",
    "PEN_long.csv"

]:

    ruta = f"/content/{archivo}"

    if not os.path.exists(ruta):

        print(f"❌ No existe: {archivo}")
        continue

    df = pd.read_csv(
        ruta,
        low_memory=False
    )

    df["fecha"] = pd.to_datetime(
        df["fecha"],
        errors="coerce"
    )

    antes = len(df)

    df = df[
        df["fecha"] <= FECHA_CORTE
    ].copy()

    despues = len(df)

    df.to_csv(
        ruta,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✅ {archivo}: "
        f"{antes:,} -> {despues:,}"
    )

❌ No existe: BCA_long.csv
❌ No existe: BCS_long.csv
✅ CEN_long.csv: 64,800 -> 63,528
✅ NES_long.csv: 64,800 -> 63,528
✅ NOR_long.csv: 64,800 -> 63,528
✅ NTE_long.csv: 64,800 -> 63,528
✅ OCC_long.csv: 64,800 -> 63,528
✅ ORI_long.csv: 64,800 -> 63,528
✅ PEN_long.csv: 64,800 -> 63,528


In [ ]:
import pandas as pd

CEN = pd.read_csv(
    "/content/CEN_long.csv",
    low_memory=False
)

NES = pd.read_csv(
    "/content/NES_long.csv",
    low_memory=False
)

NOR = pd.read_csv(
    "/content/NOR_long.csv",
    low_memory=False
)

NTE = pd.read_csv(
    "/content/NTE_long.csv",
    low_memory=False
)

OCC = pd.read_csv(
    "/content/OCC_long.csv",
    low_memory=False
)

ORI = pd.read_csv(
    "/content/ORI_long.csv",
    low_memory=False
)

PEN = pd.read_csv(
    "/content/PEN_long.csv",
    low_memory=False
)

print("CEN:", len(CEN))
print("NES:", len(NES))
print("NOR:", len(NOR))
print("NTE:", len(NTE))
print("OCC:", len(OCC))
print("ORI:", len(ORI))
print("PEN:", len(PEN))

CEN: 64800
NES: 64800
NOR: 64800
NTE: 64800
OCC: 64800
ORI: 64800
PEN: 64800


In [ ]:
import pandas as pd
import numpy as np

# =====================================================
# CARGA DE CSVs
# =====================================================

archivos = {
    "CEN": "/content/CEN_long.csv",
    "NES": "/content/NES_long.csv",
    "NOR": "/content/NOR_long.csv",
    "NTE": "/content/NTE_long.csv",
    "OCC": "/content/OCC_long.csv",
    "ORI": "/content/ORI_long.csv",
    "PEN": "/content/PEN_long.csv",
}

dataframes = []

for region, path in archivos.items():
    df = pd.read_csv(path, low_memory=False)

    # agregar columna de región para rastrear origen
    df["REGION"] = region

    dataframes.append(df)

    print(f"{region}: {len(df)} filas")

# =====================================================
# UNIFICAR TODO EN UNO SOLO
# =====================================================

CEN_long_full = pd.concat(dataframes, ignore_index=True)

print("\nTotal filas combinado:", len(CEN_long_full))

# =====================================================
# GUARDAR CSV FINAL
# =====================================================

output_path = "/content/CEN_LONG_UNIFICADO.csv"

CEN_long_full.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV generado:")
print(output_path)

CEN: 64800 filas
NES: 64800 filas
NOR: 64800 filas
NTE: 64800 filas
OCC: 64800 filas
ORI: 64800 filas
PEN: 64800 filas

Total filas combinado: 453600

CSV generado:
/content/CEN_LONG_UNIFICADO.csv


#Modelos univariados

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 14.0 MB/s eta 0:00:00


In [ ]:
# =========================================================
# PIPELINE HORARIO REGIONES
# XGBoost/LightGBM: serie horaria completa
# LSTM: solo último año horario, split 90/10
# Sin ARIMA / SARIMA
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import numpy as np
import pandas as pd
import optuna

from optuna.samplers import TPESampler

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping


# =========================================================
# CONFIG
# =========================================================

MIN_OBS = 24 * 30

WINDOW_DEFAULT = 168          # 1 semana horaria
VAL_GAP = 24 * 7              # 1 semana de gap
VAL_SPLITS = 3

N_TRIALS_OPTUNA = 10          # XGBoost / LightGBM
N_TRIALS_LSTM = 3             # LSTM rápida

EPOCHS_LSTM = 5
PATIENCE_LSTM = 2

LSTM_LAST_HOURS = 24 * 365    # último año

OPTUNA_DB = "optuna_regiones_horario.db"
SAVE_PREFIX = "regiones_horario"

ARCHIVOS_REGIONES = {
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = "Estimacion de Demanda por Balance (MWh)"
COL_GENERACION = "Generacion (MWh)"


# =========================================================
# UTILIDADES
# =========================================================

def cleanup():
    K.clear_session()
    gc.collect()


def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return np.mean(
        np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
    ) * 100


def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return np.mean(
        np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]
    ) * 100


def calcular_metricas(y_true, y_pred):
    n = min(len(y_true), len(y_pred))

    y_c = np.array(y_true[:n], dtype=float)
    p_c = np.array(y_pred[:n], dtype=float)

    mask = (~np.isnan(y_c)) & (~np.isnan(p_c))

    if mask.sum() == 0:
        return None

    y_c = y_c[mask]
    p_c = p_c[mask]

    return {
        "MAE": mean_absolute_error(y_c, p_c),
        "RMSE": np.sqrt(mean_squared_error(y_c, p_c)),
        "MAPE": mape(y_c, p_c),
        "sMAPE": smape(y_c, p_c)
    }


# =========================================================
# LECTURA
# =========================================================

def cargar_regiones():
    regiones = {}

    for region, archivo in ARCHIVOS_REGIONES.items():

        if not os.path.exists(archivo):
            print(f"⚠️ No encontré {archivo}, salto {region}")
            continue

        df = pd.read_csv(archivo)

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
        )

        regiones[region] = df

        print(f"✅ {region}: {archivo} cargado con shape {df.shape}")

    return regiones


def extraer_serie_horaria(df, columna, nombre_serie):

    if columna not in df.columns:
        raise ValueError(
            f"No existe columna {columna} en {nombre_serie}. "
            f"Columnas: {list(df.columns)}"
        )

    if COL_FECHA not in df.columns:
        raise ValueError(f"No existe columna '{COL_FECHA}' en {nombre_serie}")

    if COL_HORA not in df.columns:
        raise ValueError(f"No existe columna '{COL_HORA}' en {nombre_serie}")

    aux = df[[COL_FECHA, COL_HORA, columna]].copy()

    aux[COL_FECHA] = pd.to_datetime(
        aux[COL_FECHA],
        errors="coerce"
    )

    aux[COL_HORA] = pd.to_numeric(
        aux[COL_HORA],
        errors="coerce"
    )

    aux[columna] = pd.to_numeric(
        aux[columna],
        errors="coerce"
    )

    aux = aux.dropna(subset=[COL_FECHA, COL_HORA, columna])

    # Si Hora viene 1-24, la convertimos a 0-23
    aux["hora_0_23"] = aux[COL_HORA].astype(int) - 1

    aux["datetime"] = aux[COL_FECHA] + pd.to_timedelta(
        aux["hora_0_23"],
        unit="h"
    )

    aux = aux.sort_values("datetime")

    fechas = aux["datetime"].values
    serie = aux[columna].values.astype(float)

    return serie, fechas


# =========================================================
# SPLIT TEMPORAL
# =========================================================

def temporal_validation_split(train_y, n_splits=3, gap=168):
    splits = []
    total_len = len(train_y)

    val_size = max(24 * 7, total_len // (n_splits + 2))

    for i in range(n_splits):
        val_end = total_len - (i * val_size) - gap
        val_start = max(val_end - val_size, gap)
        train_end = val_start - gap

        if train_end < MIN_OBS or val_end - val_start < 24:
            continue

        splits.append({
            "train_y": train_y[:train_end],
            "val_y": train_y[val_start:val_end],
            "name": f"split_{i}"
        })

    return splits


# =========================================================
# FEATURES HORARIAS ML
# =========================================================

def create_feature_df(y, window=168):
    df = pd.DataFrame({"y": y})

    for lag in range(1, window + 1):
        df[f"lag_{lag}"] = df["y"].shift(lag)

    df["rolling_mean_24"] = df["y"].rolling(24).mean()
    df["rolling_std_24"] = df["y"].rolling(24).std()

    df["rolling_mean_168"] = df["y"].rolling(168).mean()
    df["rolling_std_168"] = df["y"].rolling(168).std()

    df["trend"] = np.arange(len(df))

    return df.dropna()


def create_features_from_history(hist_y, window=168):
    features = {}

    for lag in range(1, window + 1):
        features[f"lag_{lag}"] = (
            hist_y[-lag]
            if len(hist_y) >= lag
            else hist_y[0]
        )

    features["rolling_mean_24"] = np.mean(hist_y[-24:])
    features["rolling_std_24"] = np.std(hist_y[-24:])

    features["rolling_mean_168"] = np.mean(hist_y[-168:])
    features["rolling_std_168"] = np.std(hist_y[-168:])

    features["trend"] = len(hist_y)

    return pd.DataFrame([features])


# =========================================================
# BASELINES
# =========================================================

def forecast_naive(train, horizon):
    return np.repeat(train[-1], horizon)


def forecast_naive_trend(train, horizon):
    x = np.arange(len(train))
    trend = np.polyfit(x, train, 1)

    x_future = np.arange(
        len(train),
        len(train) + horizon
    )

    return trend[0] * x_future + trend[1]


# =========================================================
# XGBOOST
# =========================================================

def objective_xgboost(trial, train_y, val_y, window):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.25),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42
    }

    df_train = create_feature_df(train_y, window)

    if len(df_train) < 200:
        return float("inf")

    X_train = df_train.drop(columns=["y"])
    y_train = df_train["y"]

    val_context = np.concatenate([train_y, val_y])[-len(val_y)-window:]

    df_val = create_feature_df(val_context, window)

    if len(df_val) < len(val_y):
        return float("inf")

    X_val = df_val.drop(columns=["y"]).iloc[-len(val_y):]
    y_val = df_val["y"].iloc[-len(val_y):]

    model = XGBRegressor(
        **params,
        objective="reg:squarederror",
        n_jobs=1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    score = smape(y_val.values, preds)

    return score if not np.isnan(score) else float("inf")


def tune_xgboost(train_y, val_y, nombre_serie, window=168):

    study_name = f"{nombre_serie}_xgboost"
    storage = f"sqlite:///{OPTUNA_DB}"

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=study_name,
        storage=storage,
        load_if_exists=True
    )

    study.optimize(
        lambda trial: objective_xgboost(
            trial,
            train_y,
            val_y,
            window
        ),
        n_trials=N_TRIALS_OPTUNA,
        show_progress_bar=False
    )

    return study.best_params, study.trials_dataframe()


def forecast_xgboost_tuned(train_y, horizon, best_params, window=168):

    try:
        df_train = create_feature_df(train_y, window)

        X_train = df_train.drop(columns=["y"])
        y_train = df_train["y"]

        model = XGBRegressor(
            **best_params,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=1
        )

        model.fit(X_train, y_train)

        preds = []
        hist = list(train_y)

        for _ in range(horizon):
            X_future = create_features_from_history(hist, window)
            pred = model.predict(X_future)[0]

            preds.append(pred)
            hist.append(pred)

        return np.array(preds)

    except Exception as e:
        print(f"         Error forecast XGBoost: {str(e)[:80]}")
        return np.full(horizon, np.nan)


# =========================================================
# LIGHTGBM
# =========================================================

def objective_lightgbm(trial, train_y, val_y, window):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.25),
        "num_leaves": trial.suggest_int("num_leaves", 16, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "verbose": -1
    }

    df_train = create_feature_df(train_y, window)

    if len(df_train) < 200:
        return float("inf")

    X_train = df_train.drop(columns=["y"])
    y_train = df_train["y"]

    val_context = np.concatenate([train_y, val_y])[-len(val_y)-window:]

    df_val = create_feature_df(val_context, window)

    if len(df_val) < len(val_y):
        return float("inf")

    X_val = df_val.drop(columns=["y"]).iloc[-len(val_y):]
    y_val = df_val["y"].iloc[-len(val_y):]

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    score = smape(y_val.values, preds)

    return score if not np.isnan(score) else float("inf")


def tune_lightgbm(train_y, val_y, nombre_serie, window=168):

    study_name = f"{nombre_serie}_lightgbm"
    storage = f"sqlite:///{OPTUNA_DB}"

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=study_name,
        storage=storage,
        load_if_exists=True
    )

    study.optimize(
        lambda trial: objective_lightgbm(
            trial,
            train_y,
            val_y,
            window
        ),
        n_trials=N_TRIALS_OPTUNA,
        show_progress_bar=False
    )

    return study.best_params, study.trials_dataframe()


def forecast_lightgbm_tuned(train_y, horizon, best_params, window=168):

    try:
        df_train = create_feature_df(train_y, window)

        X_train = df_train.drop(columns=["y"])
        y_train = df_train["y"]

        model = LGBMRegressor(
            **best_params,
            random_state=42,
            verbose=-1
        )

        model.fit(X_train, y_train)

        preds = []
        hist = list(train_y)

        for _ in range(horizon):
            X_future = create_features_from_history(hist, window)
            pred = model.predict(X_future)[0]

            preds.append(pred)
            hist.append(pred)

        return np.array(preds)

    except Exception as e:
        print(f"         Error forecast LightGBM: {str(e)[:80]}")
        return np.full(horizon, np.nan)


# =========================================================
# LSTM RÁPIDA ÚLTIMO AÑO
# =========================================================

def crear_secuencias(y, window):
    Xs = []
    ys = []

    for i in range(window, len(y)):
        Xs.append(y[i-window:i])
        ys.append(y[i])

    return np.array(Xs), np.array(ys)


def objective_lstm(trial, train_y, val_y):

    params = {
        "units": trial.suggest_categorical("units", [16, 32]),
        "dropout": trial.suggest_categorical("dropout", [0.1, 0.2]),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "window": trial.suggest_categorical("window", [24, 48, 168]),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            [0.001, 0.003]
        )
    }

    window = params["window"]

    if len(train_y) < window + 200:
        return float("inf")

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_y.reshape(-1, 1)
    ).flatten()

    X_seq, y_seq = crear_secuencias(
        train_scaled,
        window
    )

    if len(X_seq) < 200:
        return float("inf")

    X_seq = X_seq.reshape(
        X_seq.shape[0],
        X_seq.shape[1],
        1
    )

    split = int(len(X_seq) * 0.8)

    X_train = X_seq[:split]
    X_val = X_seq[split:]

    y_train = y_seq[:split]
    y_val = y_seq[split:]

    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(
            params["units"],
            dropout=params["dropout"]
        ),
        Dense(1)
    ])

    model.compile(
        loss="mse",
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=params["learning_rate"]
        )
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_LSTM,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_LSTM,
        batch_size=params["batch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    val_scaled = scaler.transform(
        val_y.reshape(-1, 1)
    ).flatten()

    preds = []
    hist = list(train_scaled)

    for _ in range(len(val_scaled)):
        seq = np.array(hist[-window:]).reshape(1, window, 1)

        pred_s = model.predict(seq, verbose=0)[0][0]

        preds.append(pred_s)
        hist.append(pred_s)

    preds_original = scaler.inverse_transform(
        np.array(preds).reshape(-1, 1)
    ).flatten()

    score = smape(val_y, preds_original)

    cleanup()

    return score if not np.isnan(score) else float("inf")


def tune_lstm(train_y, val_y, nombre_serie):

    study_name = f"{nombre_serie}_lstm_fast_1y"
    storage = f"sqlite:///{OPTUNA_DB}"

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=study_name,
        storage=storage,
        load_if_exists=True
    )

    study.optimize(
        lambda trial: objective_lstm(
            trial,
            train_y,
            val_y
        ),
        n_trials=N_TRIALS_LSTM,
        show_progress_bar=False
    )

    return study.best_params, study.trials_dataframe()


def forecast_lstm_tuned(train_y, horizon, best_params):

    try:
        window = best_params.get("window", 24)

        scaler = StandardScaler()

        train_scaled = scaler.fit_transform(
            train_y.reshape(-1, 1)
        ).flatten()

        X_seq, y_seq = crear_secuencias(
            train_scaled,
            window
        )

        if len(X_seq) < 200:
            return np.full(horizon, np.nan)

        X_seq = X_seq.reshape(
            X_seq.shape[0],
            X_seq.shape[1],
            1
        )

        split = int(len(X_seq) * 0.8)

        X_train = X_seq[:split]
        X_val = X_seq[split:]

        y_train = y_seq[:split]
        y_val = y_seq[split:]

        model = Sequential([
            Input(shape=(window, 1)),
            LSTM(
                best_params["units"],
                dropout=best_params["dropout"]
            ),
            Dense(1)
        ])

        model.compile(
            loss="mse",
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=best_params["learning_rate"]
            )
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_LSTM,
            restore_best_weights=True
        )

        model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS_LSTM,
            batch_size=best_params["batch_size"],
            callbacks=[early_stop],
            verbose=0
        )

        preds = []
        hist = list(train_scaled)

        for _ in range(horizon):
            seq = np.array(hist[-window:]).reshape(1, window, 1)

            pred_s = model.predict(seq, verbose=0)[0][0]

            preds.append(pred_s)
            hist.append(pred_s)

        preds_original = scaler.inverse_transform(
            np.array(preds).reshape(-1, 1)
        ).flatten()

        return preds_original

    except Exception as e:
        print(f"         Error forecast LSTM: {str(e)[:80]}")
        return np.full(horizon, np.nan)

    finally:
        cleanup()


# =========================================================
# RESULTADOS
# =========================================================

RESULTS_SERIES = []
RESULTS_METRICS = []
RESULTS_TRIALS = []
RESULTS_CONFIG_USADA = []


def guardar_predicciones(nombre_serie, fechas_test, pred, modelo):

    for j, pred_val in enumerate(pred):
        if j < len(fechas_test):
            RESULTS_SERIES.append({
                "serie": nombre_serie,
                "fecha": fechas_test[j],
                "tipo": "prediccion",
                "subset": "test",
                "modelo": modelo,
                "valor": pred_val
            })


def guardar_metricas(nombre_serie, modelo, tuneado, metricas, horizonte_usado):

    RESULTS_METRICS.append({
        "serie": nombre_serie,
        "modelo": modelo,
        "tuneado": tuneado,
        "horizonte_usado": horizonte_usado,
        "MAPE": metricas["MAPE"],
        "sMAPE": metricas["sMAPE"],
        "MAE": metricas["MAE"],
        "RMSE": metricas["RMSE"]
    })


# =========================================================
# EVALUAR SERIE
# =========================================================

def evaluar_serie(nombre_serie, serie, fechas):

    global RESULTS_SERIES
    global RESULTS_METRICS
    global RESULTS_TRIALS
    global RESULTS_CONFIG_USADA

    if len(serie) < MIN_OBS:
        print(f"   ⚠️ Serie insuficiente: {nombre_serie}")
        return

    for j, fecha in enumerate(fechas):
        RESULTS_SERIES.append({
            "serie": nombre_serie,
            "fecha": fecha,
            "tipo": "real",
            "subset": "completo",
            "modelo": "real",
            "valor": serie[j]
        })

    # =====================================================
    # SPLIT GENERAL PARA NAIVE / XGB / LGBM
    # =====================================================

    test_size = max(24 * 30, int(len(serie) * 0.10))
    test_size = min(test_size, len(serie) // 3)

    train = serie[:-test_size]
    test = serie[-test_size:]

    fechas_test = fechas[-test_size:]

    horizon = len(test)

    print(f"\n   📊 Split general")
    print(f"      Train: {len(train)} obs")
    print(f"      Test:  {len(test)} obs")

    splits = temporal_validation_split(
        train,
        n_splits=VAL_SPLITS,
        gap=VAL_GAP
    )

    if len(splits) == 0:
        val_size = max(24 * 30, len(train) // 5)

        splits = [{
            "train_y": train[:-val_size],
            "val_y": train[-val_size:],
            "name": "split_0"
        }]

    tune_split = splits[0]
    window_tune = WINDOW_DEFAULT

    # =====================================================
    # NAIVE
    # =====================================================

    try:
        pred = forecast_naive(train, horizon)
        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(
                nombre_serie,
                "Naive",
                False,
                metricas,
                "serie_completa"
            )

            guardar_predicciones(
                nombre_serie,
                fechas_test,
                pred,
                "Naive"
            )

            print(f"      Naive: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      Naive Error: {str(e)[:80]}")

    # =====================================================
    # NAIVE TREND
    # =====================================================

    try:
        pred = forecast_naive_trend(train, horizon)
        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(
                nombre_serie,
                "Naive_Trend",
                False,
                metricas,
                "serie_completa"
            )

            guardar_predicciones(
                nombre_serie,
                fechas_test,
                pred,
                "Naive_Trend"
            )

            print(f"      Naive_Trend: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      Naive_Trend Error: {str(e)[:80]}")

    # =====================================================
    # XGBOOST
    # =====================================================

    try:
        print("      XGBoost tuning...")

        best_params, trials_df = tune_xgboost(
            tune_split["train_y"],
            tune_split["val_y"],
            nombre_serie,
            window_tune
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "XGBoost"
        RESULTS_TRIALS.append(trials_df)

        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "XGBoost",
            "parametros": str(best_params),
            "horizonte_usado": "serie_completa"
        })

        pred = forecast_xgboost_tuned(
            train,
            horizon,
            best_params,
            window_tune
        )

        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(
                nombre_serie,
                "XGBoost_Tuned",
                True,
                metricas,
                "serie_completa"
            )

            guardar_predicciones(
                nombre_serie,
                fechas_test,
                pred,
                "XGBoost_Tuned"
            )

            print(f"      XGBoost_Tuned: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      XGBoost_Tuned Error: {str(e)[:80]}")

    # =====================================================
    # LIGHTGBM
    # =====================================================

    try:
        print("      LightGBM tuning...")

        best_params, trials_df = tune_lightgbm(
            tune_split["train_y"],
            tune_split["val_y"],
            nombre_serie,
            window_tune
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "LightGBM"
        RESULTS_TRIALS.append(trials_df)

        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "LightGBM",
            "parametros": str(best_params),
            "horizonte_usado": "serie_completa"
        })

        pred = forecast_lightgbm_tuned(
            train,
            horizon,
            best_params,
            window_tune
        )

        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(
                nombre_serie,
                "LightGBM_Tuned",
                True,
                metricas,
                "serie_completa"
            )

            guardar_predicciones(
                nombre_serie,
                fechas_test,
                pred,
                "LightGBM_Tuned"
            )

            print(f"      LightGBM_Tuned: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      LightGBM_Tuned Error: {str(e)[:80]}")

    # =====================================================
    # LSTM ÚLTIMO AÑO
    # =====================================================

    try:
        print("      LSTM último año tuning...")

        serie_lstm = serie[-LSTM_LAST_HOURS:]
        fechas_lstm = fechas[-LSTM_LAST_HOURS:]

        test_size_lstm = max(24 * 7, int(len(serie_lstm) * 0.10))

        train_lstm = serie_lstm[:-test_size_lstm]
        test_lstm = serie_lstm[-test_size_lstm:]

        fechas_test_lstm = fechas_lstm[-test_size_lstm:]

        print(f"         LSTM train: {len(train_lstm)} obs")
        print(f"         LSTM test:  {len(test_lstm)} obs")

        val_size_lstm = max(24 * 7, int(len(train_lstm) * 0.10))

        train_lstm_tune = train_lstm[:-val_size_lstm]
        val_lstm_tune = train_lstm[-val_size_lstm:]

        best_params, trials_df = tune_lstm(
            train_lstm_tune,
            val_lstm_tune,
            nombre_serie
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "LSTM"
        RESULTS_TRIALS.append(trials_df)

        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "LSTM",
            "parametros": str(best_params),
            "horizonte_usado": "ultimo_anio"
        })

        pred = forecast_lstm_tuned(
            train_lstm,
            len(test_lstm),
            best_params
        )

        metricas = calcular_metricas(test_lstm, pred)

        if metricas:
            guardar_metricas(
                nombre_serie,
                "LSTM_Tuned_1Y",
                True,
                metricas,
                "ultimo_anio"
            )

            guardar_predicciones(
                nombre_serie,
                fechas_test_lstm,
                pred,
                "LSTM_Tuned_1Y"
            )

            print(f"      LSTM_Tuned_1Y: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      LSTM_Tuned_1Y Error: {str(e)[:80]}")


# =========================================================
# GUARDAR CSVs
# =========================================================

def guardar_todos_csv():

    df_metrics = None

    if RESULTS_SERIES:
        df_series = pd.DataFrame(RESULTS_SERIES)

        df_series["fecha"] = pd.to_datetime(
            df_series["fecha"],
            errors="coerce"
        )

        df_series = df_series.sort_values(
            ["serie", "fecha", "modelo"]
        )

        df_series.to_csv(
            f"{SAVE_PREFIX}_series.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\n✅ Series guardadas: {len(df_series):,} registros")

    if RESULTS_METRICS:
        df_metrics = pd.DataFrame(RESULTS_METRICS)

        df_metrics = df_metrics.sort_values(
            ["serie", "MAPE"]
        )

        df_metrics.to_csv(
            f"{SAVE_PREFIX}_metricas.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"✅ Métricas guardadas: {len(df_metrics):,} registros")

    if RESULTS_TRIALS:
        df_trials = pd.concat(
            RESULTS_TRIALS,
            ignore_index=True
        )

        df_trials.to_csv(
            f"{SAVE_PREFIX}_optuna_trials.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"✅ Trials Optuna guardados: {len(df_trials):,} registros")

    if RESULTS_CONFIG_USADA:
        df_config = pd.DataFrame(RESULTS_CONFIG_USADA)

        df_config.to_csv(
            f"{SAVE_PREFIX}_config_usada.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(f"✅ Configuración usada guardada: {len(df_config):,} registros")

    return df_metrics


def imprimir_resultados(df_metricas):

    if df_metricas is None or len(df_metricas) == 0:
        print("\n❌ No hay resultados para mostrar")
        return

    print("\n" + "=" * 100)
    print("📊 MEJOR MODELO POR SERIE")
    print("=" * 100)

    mejor_por_serie = df_metricas.loc[
        df_metricas.groupby("serie")["MAPE"].idxmin()
    ]

    print(
        mejor_por_serie[
            [
                "serie",
                "modelo",
                "horizonte_usado",
                "MAPE",
                "sMAPE",
                "MAE",
                "RMSE"
            ]
        ].to_string(index=False)
    )

    print("\n" + "=" * 100)
    print("🏆 RANKING GLOBAL DE MODELOS")
    print("=" * 100)

    ranking = (
        df_metricas
        .groupby("modelo")["MAPE"]
        .agg(["mean", "std"])
        .round(2)
        .sort_values("mean")
    )

    print(ranking.to_string())


# =========================================================
# PIPELINE PRINCIPAL
# =========================================================

def ejecutar_pipeline():

    global RESULTS_SERIES
    global RESULTS_METRICS
    global RESULTS_TRIALS
    global RESULTS_CONFIG_USADA

    RESULTS_SERIES = []
    RESULTS_METRICS = []
    RESULTS_TRIALS = []
    RESULTS_CONFIG_USADA = []

    print("=" * 80)
    print("🔋 PIPELINE HORARIO REGIONES")
    print("=" * 80)

    regiones = cargar_regiones()

    for region, df in regiones.items():

        objetivos = {
            f"{region}_DEMANDA": COL_DEMANDA,
            f"{region}_GENERACION": COL_GENERACION
        }

        for nombre_serie, columna in objetivos.items():

            print(f"\n{'=' * 80}")
            print(f"🔥 Serie: {nombre_serie}")
            print(f"{'=' * 80}")

            serie, fechas = extraer_serie_horaria(
                df,
                columna,
                nombre_serie
            )

            print(f"\n   📈 Serie horaria:")
            print(f"      Longitud: {len(serie):,} observaciones")
            print(f"      Rango: {fechas[0]} a {fechas[-1]}")
            print(f"      Mínimo: {np.nanmin(serie):.2f}")
            print(f"      Máximo: {np.nanmax(serie):.2f}")
            print(f"      Media: {np.nanmean(serie):.2f}")

            evaluar_serie(
                nombre_serie,
                serie,
                fechas
            )

    df_metricas = guardar_todos_csv()

    imprimir_resultados(df_metricas)

    print("\n" + "=" * 80)
    print("✅ PIPELINE COMPLETADO")
    print(f"📁 {SAVE_PREFIX}_series.csv")
    print(f"📁 {SAVE_PREFIX}_metricas.csv")
    print(f"📁 {SAVE_PREFIX}_optuna_trials.csv")
    print(f"📁 {SAVE_PREFIX}_config_usada.csv")
    print("=" * 80)


# =========================================================
# EJECUTAR
# =========================================================

ejecutar_pipeline()

🔋 PIPELINE HORARIO REGIONES
✅ CEN: CEN_long.csv cargado con shape (64800, 11)
✅ NES: NES_long.csv cargado con shape (64800, 11)
✅ NOR: NOR_long.csv cargado con shape (64800, 11)
✅ NTE: NTE_long.csv cargado con shape (64800, 11)
✅ OCC: OCC_long.csv cargado con shape (64800, 11)
✅ ORI: ORI_long.csv cargado con shape (64800, 11)
✅ PEN: PEN_long.csv cargado con shape (64800, 11)

🔥 Serie: CEN_DEMANDA

   📈 Serie horaria:
      Longitud: 64,800 observaciones
      Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000
      Mínimo: 1431.15
      Máximo: 8918.56
      Media: 6231.83

   📊 Split general
      Train: 58320 obs
      Test:  6480 obs
      Naive: MAPE=11.97%
      Naive_Trend: MAPE=14.29%
      XGBoost tuning...


[I 2026-06-08 21:56:44,513] A new study created in RDB with name: CEN_DEMANDA_xgboost
[I 2026-06-08 21:57:07,920] Trial 0 finished with value: 1.1878461768167738 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 1.1878461768167738.
[I 2026-06-08 21:57:25,497] Trial 1 finished with value: 1.203452561021069 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 1.1878461768167738.
[I 2026-06-08 21:57:33,641] Trial 2 finished with value: 1.2000407600097835 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790

      XGBoost_Tuned: MAPE=5.88%
      LightGBM tuning...


[I 2026-06-08 22:02:05,973] Trial 0 finished with value: 1.1531782920243812 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'num_leaves': 54, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 1.1531782920243812.
[I 2026-06-08 22:02:18,208] Trial 1 finished with value: 1.2152687032356368 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'num_leaves': 79, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 1.1531782920243812.
[I 2026-06-08 22:02:24,894] Trial 2 finished with value: 1.1838776683555359 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'num_leaves': 34, 'subsample': 0.8835558684167139, 'colsample_bytree': 0.74184815819

      LightGBM_Tuned: MAPE=4.62%
      LSTM último año tuning...
         LSTM train: 7884 obs
         LSTM test:  876 obs


...

      XGBoost_Tuned: MAPE=47.92%
      LightGBM tuning...


[I 2026-06-09 01:02:50,822] Trial 0 finished with value: 4.1455137850039705 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'num_leaves': 54, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 4.1455137850039705.
[I 2026-06-09 01:03:05,397] Trial 1 finished with value: 4.208185469991386 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'num_leaves': 79, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 4.1455137850039705.
[I 2026-06-09 01:03:12,526] Trial 2 finished with value: 4.093460028399319 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'num_leaves': 34, 'subsample': 0.8835558684167139, 'colsample_bytree': 0.7418481581956

      LightGBM_Tuned: MAPE=52.49%
      LSTM último año tuning...
         LSTM train: 7884 obs
         LSTM test:  876 obs


[I 2026-06-09 01:06:37,247] Trial 0 finished with value: 29.391050824915936 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 29.391050824915936.
[I 2026-06-09 01:08:12,265] Trial 1 finished with value: 26.280067424740494 and parameters: {'units': 16, 'dropout': 0.1, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 26.280067424740494.
[I 2026-06-09 01:10:12,389] Trial 2 finished with value: 21.019473382498976 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 128, 'window': 168, 'learning_rate': 0.001}. Best is trial 2 with value: 21.019473382498976.


      LSTM_Tuned_1Y: MAPE=29.45%

✅ Series guardadas: 1,282,344 registros
✅ Métricas guardadas: 70 registros
✅ Trials Optuna guardados: 322 registros
✅ Configuración usada guardada: 42 registros

📊 MEJOR MODELO POR SERIE
         serie         modelo horizonte_usado      MAPE     sMAPE         MAE        RMSE
   CEN_DEMANDA LightGBM_Tuned  serie_completa  4.618660  4.599332  306.255201  418.942429
CEN_GENERACION  XGBoost_Tuned  serie_completa 14.536346 14.491128  545.311957  682.719139
   NES_DEMANDA  LSTM_Tuned_1Y     ultimo_anio 13.207601 12.608799  945.473800 1176.169993
NES_GENERACION  LSTM_Tuned_1Y     ultimo_anio 13.810232 14.661440 1611.028442 1949.857788
   NOR_DEMANDA  LSTM_Tuned_1Y     ultimo_anio 16.209481 14.750979  508.348172  618.872459
NOR_GENERACION  LSTM_Tuned_1Y     ultimo_anio 13.510374 14.718822  568.454960  698.770456
   NTE_DEMANDA  LSTM_Tuned_1Y     ultimo_anio 10.128244 10.433085  401.977339  494.887445
NTE_GENERACION  LSTM_Tuned_1Y     ultimo_anio 17.143746 19.

In [ ]:
# =========================================================
# CELDA COLAB - PIPELINE HORARIO MULTIVARIADO
# Asume que YA corriste:
# Temperaturas_H, Primarias_H, Secundarias_H, Terciarias_H, IGAE_H
# cada dataframe con columnas: fecha, hora, valor
#
# Incluye BCA_long.csv
# Guarda avances por region en Google Drive
# =========================================================

import os
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna

from optuna.samplers import TPESampler

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping

from google.colab import drive
drive.mount("/content/drive")


# =========================================================
# CONFIG
# =========================================================

MIN_OBS = 24 * 30

WINDOW_DEFAULT = 168
VAL_GAP = 24 * 7
VAL_SPLITS = 3

N_TRIALS_OPTUNA = 10
N_TRIALS_LSTM = 3

EPOCHS_LSTM = 5
PATIENCE_LSTM = 2

LSTM_LAST_HOURS = 24 * 365

DRIVE_OUT_DIR = "/content/drive/MyDrive/resultados_regiones_multivariado"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

OPTUNA_DB = os.path.join(DRIVE_OUT_DIR, "optuna_regiones_horario_multivariado.db")
SAVE_PREFIX = "regiones_horario_multivariado"

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = "Estimacion de Demanda por Balance (MWh)"
COL_GENERACION = "Generacion (MWh)"


# =========================================================
# UTILIDADES
# =========================================================

def cleanup():
    K.clear_session()
    gc.collect()


def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100


def calcular_metricas(y_true, y_pred):
    n = min(len(y_true), len(y_pred))
    y_c = np.array(y_true[:n], dtype=float)
    p_c = np.array(y_pred[:n], dtype=float)
    mask = (~np.isnan(y_c)) & (~np.isnan(p_c))

    if mask.sum() == 0:
        return None

    y_c = y_c[mask]
    p_c = p_c[mask]

    return {
        "MAE": mean_absolute_error(y_c, p_c),
        "RMSE": np.sqrt(mean_squared_error(y_c, p_c)),
        "MAPE": mape(y_c, p_c),
        "sMAPE": smape(y_c, p_c),
    }


def crear_datetime_desde_fecha_hora(df, col_fecha="fecha", col_hora="hora"):
    out = df.copy()
    out[col_fecha] = pd.to_datetime(out[col_fecha], errors="coerce")
    out[col_hora] = pd.to_numeric(out[col_hora], errors="coerce")
    out = out.dropna(subset=[col_fecha, col_hora])
    out["hora_0_23"] = out[col_hora].astype(int) - 1
    out["datetime"] = out[col_fecha] + pd.to_timedelta(out["hora_0_23"], unit="h")
    return out


# =========================================================
# EXOGENAS YA EXPANDIDAS
# =========================================================

def preparar_exogena_expandida(df, nombre):
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    col_hora = "hora" if "hora" in out.columns else "Hora"

    if "valor" not in out.columns:
        raise ValueError(f"{nombre}_H debe tener columna 'valor'")

    out = crear_datetime_desde_fecha_hora(out, col_fecha="fecha", col_hora=col_hora)
    out = out[["datetime", "valor"]].rename(columns={"valor": nombre})
    out[nombre] = pd.to_numeric(out[nombre], errors="coerce")
    out = out.dropna(subset=["datetime"])
    out = out.sort_values("datetime")
    out = out.drop_duplicates(subset=["datetime"], keep="last")

    return out


def construir_exogenas_horarias_desde_colab():
    exogs = {
        "Temperaturas": Temperaturas_H,
        "Primarias": Primarias_H,
        "Secundarias": Secundarias_H,
        "Terciarias": Terciarias_H,
        "IGAE": IGAE_H,
    }

    dfs = []

    for nombre, df in exogs.items():
        tmp = preparar_exogena_expandida(df, nombre)
        print(
            f"{nombre:15s} | "
            f"{tmp['datetime'].min()} -> {tmp['datetime'].max()} | "
            f"{len(tmp):,} filas"
        )
        dfs.append(tmp)

    df_exog = dfs[0]

    for df_next in dfs[1:]:
        df_exog = df_exog.merge(df_next, on="datetime", how="outer")

    exog_cols = [c for c in df_exog.columns if c != "datetime"]
    df_exog = df_exog.sort_values("datetime")
    df_exog[exog_cols] = df_exog[exog_cols].ffill().bfill()

    print("\nExogenas combinadas:")
    print(df_exog.head())
    print(df_exog.tail())

    return df_exog


def alinear_exogenas(fechas, df_exog):
    base = pd.DataFrame({"datetime": pd.to_datetime(fechas)})
    out = base.merge(df_exog, on="datetime", how="left")

    exog_cols = [c for c in out.columns if c != "datetime"]
    out[exog_cols] = out[exog_cols].ffill().bfill()

    return out


# =========================================================
# LECTURA SERIES REGIONALES
# =========================================================

def cargar_regiones():
    regiones = {}

    for region, archivo in ARCHIVOS_REGIONES.items():
        if not os.path.exists(archivo):
            print(f"AVISO: no encontre {archivo}, salto {region}")
            continue

        df = pd.read_csv(archivo)
        df.columns = df.columns.astype(str).str.strip()
        regiones[region] = df
        print(f"OK {region}: {archivo} cargado con shape {df.shape}")

    return regiones


def extraer_serie_horaria(df, columna, nombre_serie):
    if columna not in df.columns:
        raise ValueError(
            f"No existe columna {columna} en {nombre_serie}. "
            f"Columnas: {list(df.columns)}"
        )

    aux = df[[COL_FECHA, COL_HORA, columna]].copy()
    aux = crear_datetime_desde_fecha_hora(aux, col_fecha=COL_FECHA, col_hora=COL_HORA)
    aux[columna] = pd.to_numeric(aux[columna], errors="coerce")
    aux = aux.dropna(subset=["datetime", columna])
    aux = aux.sort_values("datetime")

    fechas = aux["datetime"].values
    serie = aux[columna].values.astype(float)

    return serie, fechas


# =========================================================
# FEATURES MULTIVARIADAS
# =========================================================

def agregar_calendario(df):
    out = df.copy()
    dt = pd.to_datetime(out["datetime"])

    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["month"] = dt.dt.month
    out["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)

    return out


def preparar_exog_features(exog_df):
    out = agregar_calendario(exog_df)
    out = out.drop(columns=["datetime"], errors="ignore")
    out = out.apply(pd.to_numeric, errors="coerce")
    out = out.ffill().bfill()
    return out


def create_feature_df_multivar(y, exog_df, window=168):
    df = pd.DataFrame({"y": y})
    exog_features = preparar_exog_features(exog_df).reset_index(drop=True)
    df = pd.concat([df, exog_features], axis=1)

    for lag in range(1, window + 1):
        df[f"lag_{lag}"] = df["y"].shift(lag)

    df["rolling_mean_24"] = df["y"].rolling(24).mean()
    df["rolling_std_24"] = df["y"].rolling(24).std()
    df["rolling_mean_168"] = df["y"].rolling(168).mean()
    df["rolling_std_168"] = df["y"].rolling(168).std()
    df["trend"] = np.arange(len(df))

    return df.dropna()


def create_features_from_history_multivar(hist_y, exog_row, train_columns, window=168):
    features = {}

    exog_features = preparar_exog_features(exog_row).reset_index(drop=True)

    for col in exog_features.columns:
        features[col] = exog_features.loc[0, col]

    for lag in range(1, window + 1):
        features[f"lag_{lag}"] = hist_y[-lag] if len(hist_y) >= lag else hist_y[0]

    features["rolling_mean_24"] = np.mean(hist_y[-24:])
    features["rolling_std_24"] = np.std(hist_y[-24:])
    features["rolling_mean_168"] = np.mean(hist_y[-168:])
    features["rolling_std_168"] = np.std(hist_y[-168:])
    features["trend"] = len(hist_y)

    return pd.DataFrame([features]).reindex(columns=train_columns, fill_value=0)


# =========================================================
# SPLITS
# =========================================================

def temporal_validation_split(train_y, train_exog, n_splits=3, gap=168):
    splits = []
    total_len = len(train_y)
    val_size = max(24 * 7, total_len // (n_splits + 2))

    for i in range(n_splits):
        val_end = total_len - (i * val_size) - gap
        val_start = max(val_end - val_size, gap)
        train_end = val_start - gap

        if train_end < MIN_OBS or val_end - val_start < 24:
            continue

        splits.append({
            "train_y": train_y[:train_end],
            "train_exog": train_exog.iloc[:train_end].reset_index(drop=True),
            "val_y": train_y[val_start:val_end],
            "val_exog": train_exog.iloc[val_start:val_end].reset_index(drop=True),
        })

    return splits


# =========================================================
# MODELOS BASE
# =========================================================

def forecast_naive(train, horizon):
    return np.repeat(train[-1], horizon)


def forecast_naive_trend(train, horizon):
    x = np.arange(len(train))
    trend = np.polyfit(x, train, 1)
    x_future = np.arange(len(train), len(train) + horizon)
    return trend[0] * x_future + trend[1]


# =========================================================
# XGBOOST / LIGHTGBM
# =========================================================

def objective_xgboost(trial, train_y, train_exog, val_y, val_exog, window):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.25),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
    }

    df_train = create_feature_df_multivar(train_y, train_exog, window)

    if len(df_train) < 200:
        return float("inf")

    X_train = df_train.drop(columns=["y"])
    y_train = df_train["y"]

    val_context_y = np.concatenate([train_y, val_y])[-len(val_y) - window:]
    val_context_exog = pd.concat([train_exog, val_exog], ignore_index=True).iloc[
        -len(val_y) - window:
    ].reset_index(drop=True)

    df_val = create_feature_df_multivar(val_context_y, val_context_exog, window)

    if len(df_val) < len(val_y):
        return float("inf")

    X_val = df_val.drop(columns=["y"]).iloc[-len(val_y):]
    y_val = df_val["y"].iloc[-len(val_y):]

    model = XGBRegressor(**params, objective="reg:squarederror", n_jobs=1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    score = smape(y_val.values, preds)
    return score if not np.isnan(score) else float("inf")


def objective_lightgbm(trial, train_y, train_exog, val_y, val_exog, window):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.25),
        "num_leaves": trial.suggest_int("num_leaves", 16, 80),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "verbose": -1,
    }

    df_train = create_feature_df_multivar(train_y, train_exog, window)

    if len(df_train) < 200:
        return float("inf")

    X_train = df_train.drop(columns=["y"])
    y_train = df_train["y"]

    val_context_y = np.concatenate([train_y, val_y])[-len(val_y) - window:]
    val_context_exog = pd.concat([train_exog, val_exog], ignore_index=True).iloc[
        -len(val_y) - window:
    ].reset_index(drop=True)

    df_val = create_feature_df_multivar(val_context_y, val_context_exog, window)

    if len(df_val) < len(val_y):
        return float("inf")

    X_val = df_val.drop(columns=["y"]).iloc[-len(val_y):]
    y_val = df_val["y"].iloc[-len(val_y):]

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    score = smape(y_val.values, preds)
    return score if not np.isnan(score) else float("inf")


def tune_model(objective_fn, train_y, train_exog, val_y, val_exog, nombre_serie, modelo, window=168):
    storage = f"sqlite:///{OPTUNA_DB}"
    study_name = f"{nombre_serie}_{modelo}_multivariado"

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
    )

    study.optimize(
        lambda trial: objective_fn(trial, train_y, train_exog, val_y, val_exog, window),
        n_trials=N_TRIALS_OPTUNA,
        show_progress_bar=False,
    )

    return study.best_params, study.trials_dataframe()


def forecast_tree_tuned(model_class, train_y, train_exog, future_exog, horizon, best_params, window=168):
    try:
        df_train = create_feature_df_multivar(train_y, train_exog, window)
        X_train = df_train.drop(columns=["y"])
        y_train = df_train["y"]

        if model_class == XGBRegressor:
            model = model_class(
                **best_params,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=1,
            )
        else:
            model = model_class(**best_params, random_state=42, verbose=-1)

        model.fit(X_train, y_train)

        preds = []
        hist = list(train_y)

        for i in range(horizon):
            exog_row = future_exog.iloc[[i]].reset_index(drop=True)
            X_future = create_features_from_history_multivar(
                hist,
                exog_row,
                X_train.columns,
                window,
            )
            pred = model.predict(X_future)[0]
            preds.append(pred)
            hist.append(pred)

        return np.array(preds)

    except Exception as e:
        print(f"         Error forecast {model_class.__name__}: {str(e)[:120]}")
        return np.full(horizon, np.nan)


# =========================================================
# LSTM MULTIVARIADA
# =========================================================

def crear_secuencias_multivar(y, exog_df, window):
    exog_features = preparar_exog_features(exog_df).reset_index(drop=True)
    matriz = pd.concat([pd.Series(y, name="y"), exog_features], axis=1)
    matriz = matriz.apply(pd.to_numeric, errors="coerce").ffill().bfill()

    values = matriz.values.astype(float)
    Xs = []
    ys = []

    for i in range(window, len(values)):
        Xs.append(values[i - window:i, :])
        ys.append(values[i, 0])

    return np.array(Xs), np.array(ys), matriz.columns.tolist()


def preparar_lstm_data(train_y, train_exog, window):
    X_raw, y_raw, feature_cols = crear_secuencias_multivar(train_y, train_exog, window)

    if len(X_raw) == 0:
        return None

    n_features = X_raw.shape[2]
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    flat = X_raw.reshape(-1, n_features)
    X_scaled = x_scaler.fit_transform(flat).reshape(X_raw.shape[0], X_raw.shape[1], n_features)
    y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).flatten()

    return X_scaled, y_scaled, x_scaler, y_scaler, feature_cols


def forecast_lstm_con_modelo(model, train_y, train_exog, future_exog, horizon, window, x_scaler, y_scaler, feature_cols):
    train_features = preparar_exog_features(train_exog).reset_index(drop=True)
    hist_df = pd.concat([pd.Series(train_y, name="y"), train_features], axis=1)
    hist_df = hist_df.reindex(columns=feature_cols)
    hist_df = hist_df.apply(pd.to_numeric, errors="coerce").ffill().bfill()

    preds = []

    for i in range(horizon):
        seq_raw = hist_df.iloc[-window:].values.astype(float)
        seq_scaled = x_scaler.transform(seq_raw).reshape(1, window, len(feature_cols))
        pred_scaled = model.predict(seq_scaled, verbose=0)[0][0]
        pred_original = y_scaler.inverse_transform([[pred_scaled]])[0][0]
        preds.append(pred_original)

        exog_row = preparar_exog_features(future_exog.iloc[[i]]).reset_index(drop=True)
        new_row = pd.concat([pd.DataFrame({"y": [pred_original]}), exog_row], axis=1)
        new_row = new_row.reindex(columns=feature_cols)
        hist_df = pd.concat([hist_df, new_row], ignore_index=True)
        hist_df = hist_df.apply(pd.to_numeric, errors="coerce").ffill().bfill()

    return np.array(preds)


def objective_lstm(trial, train_y, train_exog, val_y, val_exog):
    params = {
        "units": trial.suggest_categorical("units", [16, 32]),
        "dropout": trial.suggest_categorical("dropout", [0.1, 0.2]),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "window": trial.suggest_categorical("window", [24, 48, 168]),
        "learning_rate": trial.suggest_categorical("learning_rate", [0.001, 0.003]),
    }

    window = params["window"]

    if len(train_y) < window + 200:
        return float("inf")

    prepared = preparar_lstm_data(train_y, train_exog, window)

    if prepared is None:
        return float("inf")

    X_seq, y_seq, x_scaler, y_scaler, feature_cols = prepared

    if len(X_seq) < 200:
        return float("inf")

    split = int(len(X_seq) * 0.8)

    model = Sequential([
        Input(shape=(window, X_seq.shape[2])),
        LSTM(params["units"], dropout=params["dropout"]),
        Dense(1),
    ])

    model.compile(
        loss="mse",
        optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]),
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_LSTM,
        restore_best_weights=True,
    )

    model.fit(
        X_seq[:split],
        y_seq[:split],
        validation_data=(X_seq[split:], y_seq[split:]),
        epochs=EPOCHS_LSTM,
        batch_size=params["batch_size"],
        callbacks=[early_stop],
        verbose=0,
    )

    preds = forecast_lstm_con_modelo(
        model,
        train_y,
        train_exog,
        val_exog,
        len(val_y),
        window,
        x_scaler,
        y_scaler,
        feature_cols,
    )

    score = smape(val_y, preds)
    cleanup()

    return score if not np.isnan(score) else float("inf")


def tune_lstm(train_y, train_exog, val_y, val_exog, nombre_serie):
    storage = f"sqlite:///{OPTUNA_DB}"
    study_name = f"{nombre_serie}_lstm_multivariado"

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
    )

    study.optimize(
        lambda trial: objective_lstm(trial, train_y, train_exog, val_y, val_exog),
        n_trials=N_TRIALS_LSTM,
        show_progress_bar=False,
    )

    return study.best_params, study.trials_dataframe()


def forecast_lstm_tuned(train_y, train_exog, future_exog, horizon, best_params):
    try:
        window = best_params.get("window", 24)
        prepared = preparar_lstm_data(train_y, train_exog, window)

        if prepared is None:
            return np.full(horizon, np.nan)

        X_seq, y_seq, x_scaler, y_scaler, feature_cols = prepared

        if len(X_seq) < 200:
            return np.full(horizon, np.nan)

        split = int(len(X_seq) * 0.8)

        model = Sequential([
            Input(shape=(window, X_seq.shape[2])),
            LSTM(best_params["units"], dropout=best_params["dropout"]),
            Dense(1),
        ])

        model.compile(
            loss="mse",
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]),
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE_LSTM,
            restore_best_weights=True,
        )

        model.fit(
            X_seq[:split],
            y_seq[:split],
            validation_data=(X_seq[split:], y_seq[split:]),
            epochs=EPOCHS_LSTM,
            batch_size=best_params["batch_size"],
            callbacks=[early_stop],
            verbose=0,
        )

        return forecast_lstm_con_modelo(
            model,
            train_y,
            train_exog,
            future_exog,
            horizon,
            window,
            x_scaler,
            y_scaler,
            feature_cols,
        )

    except Exception as e:
        print(f"         Error forecast LSTM: {str(e)[:120]}")
        return np.full(horizon, np.nan)

    finally:
        cleanup()


# =========================================================
# GUARDADO PROGRESIVO
# =========================================================

RESULTS_SERIES = []
RESULTS_METRICS = []
RESULTS_TRIALS = []
RESULTS_CONFIG_USADA = []


def guardar_predicciones(nombre_serie, fechas_test, pred, modelo):
    for j, pred_val in enumerate(pred):
        if j < len(fechas_test):
            RESULTS_SERIES.append({
                "serie": nombre_serie,
                "fecha": fechas_test[j],
                "tipo": "prediccion",
                "subset": "test",
                "modelo": modelo,
                "valor": pred_val,
            })


def guardar_metricas(nombre_serie, modelo, tuneado, metricas, horizonte_usado):
    RESULTS_METRICS.append({
        "serie": nombre_serie,
        "modelo": modelo,
        "tuneado": tuneado,
        "horizonte_usado": horizonte_usado,
        "MAPE": metricas["MAPE"],
        "sMAPE": metricas["sMAPE"],
        "MAE": metricas["MAE"],
        "RMSE": metricas["RMSE"],
    })


def guardar_real(nombre_serie, fechas, serie):
    for j, fecha in enumerate(fechas):
        RESULTS_SERIES.append({
            "serie": nombre_serie,
            "fecha": fecha,
            "tipo": "real",
            "subset": "completo",
            "modelo": "real",
            "valor": serie[j],
        })


def guardar_snapshot(region):
    if RESULTS_SERIES:
        pd.DataFrame(RESULTS_SERIES).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_series_HASTA_{region}.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_METRICS:
        pd.DataFrame(RESULTS_METRICS).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_metricas_HASTA_{region}.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_TRIALS:
        pd.concat(RESULTS_TRIALS, ignore_index=True).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_optuna_trials_HASTA_{region}.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_CONFIG_USADA:
        pd.DataFrame(RESULTS_CONFIG_USADA).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_config_usada_HASTA_{region}.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    print(f"Snapshot guardado en Drive hasta region {region}")


def guardar_final():
    df_metrics = None

    if RESULTS_SERIES:
        df_series = pd.DataFrame(RESULTS_SERIES)
        df_series["fecha"] = pd.to_datetime(df_series["fecha"], errors="coerce")
        df_series = df_series.sort_values(["serie", "fecha", "modelo"])
        df_series.to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_series_FINAL.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_METRICS:
        df_metrics = pd.DataFrame(RESULTS_METRICS)
        df_metrics = df_metrics.sort_values(["serie", "MAPE"])
        df_metrics.to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_metricas_FINAL.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_TRIALS:
        pd.concat(RESULTS_TRIALS, ignore_index=True).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_optuna_trials_FINAL.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    if RESULTS_CONFIG_USADA:
        pd.DataFrame(RESULTS_CONFIG_USADA).to_csv(
            os.path.join(DRIVE_OUT_DIR, f"{SAVE_PREFIX}_config_usada_FINAL.csv"),
            index=False,
            encoding="utf-8-sig",
        )

    return df_metrics


# =========================================================
# EVALUACION
# =========================================================

def evaluar_serie(nombre_serie, serie, fechas, exog_alineada):
    if len(serie) < MIN_OBS:
        print(f"   AVISO: serie insuficiente: {nombre_serie}")
        return

    guardar_real(nombre_serie, fechas, serie)

    test_size = max(24 * 30, int(len(serie) * 0.10))
    test_size = min(test_size, len(serie) // 3)

    train = serie[:-test_size]
    test = serie[-test_size:]

    train_exog = exog_alineada.iloc[:-test_size].reset_index(drop=True)
    test_exog = exog_alineada.iloc[-test_size:].reset_index(drop=True)

    fechas_test = fechas[-test_size:]
    horizon = len(test)

    print("\n   Split general")
    print(f"      Train: {len(train):,} obs")
    print(f"      Test:  {len(test):,} obs")
    print(f"      Exogenas base: {max(0, train_exog.shape[1] - 1)}")

    splits = temporal_validation_split(train, train_exog, n_splits=VAL_SPLITS, gap=VAL_GAP)

    if len(splits) == 0:
        val_size = max(24 * 30, len(train) // 5)
        splits = [{
            "train_y": train[:-val_size],
            "train_exog": train_exog.iloc[:-val_size].reset_index(drop=True),
            "val_y": train[-val_size:],
            "val_exog": train_exog.iloc[-val_size:].reset_index(drop=True),
        }]

    tune_split = splits[0]

    for modelo, pred in [
        ("Naive", forecast_naive(train, horizon)),
        ("Naive_Trend", forecast_naive_trend(train, horizon)),
    ]:
        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(nombre_serie, modelo, False, metricas, "serie_completa")
            guardar_predicciones(nombre_serie, fechas_test, pred, modelo)
            print(f"      {modelo}: MAPE={metricas['MAPE']:.2f}%")

    try:
        print("      XGBoost multivariado tuning...")
        best_params, trials_df = tune_model(
            objective_xgboost,
            tune_split["train_y"],
            tune_split["train_exog"],
            tune_split["val_y"],
            tune_split["val_exog"],
            nombre_serie,
            "xgboost",
            WINDOW_DEFAULT,
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "XGBoost_Multivar"
        RESULTS_TRIALS.append(trials_df)
        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "XGBoost_Multivar",
            "parametros": str(best_params),
            "horizonte_usado": "serie_completa",
        })

        pred = forecast_tree_tuned(
            XGBRegressor,
            train,
            train_exog,
            test_exog,
            horizon,
            best_params,
            WINDOW_DEFAULT,
        )
        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(nombre_serie, "XGBoost_Multivar_Tuned", True, metricas, "serie_completa")
            guardar_predicciones(nombre_serie, fechas_test, pred, "XGBoost_Multivar_Tuned")
            print(f"      XGBoost_Multivar_Tuned: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      XGBoost_Multivar_Tuned Error: {str(e)[:120]}")

    try:
        print("      LightGBM multivariado tuning...")
        best_params, trials_df = tune_model(
            objective_lightgbm,
            tune_split["train_y"],
            tune_split["train_exog"],
            tune_split["val_y"],
            tune_split["val_exog"],
            nombre_serie,
            "lightgbm",
            WINDOW_DEFAULT,
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "LightGBM_Multivar"
        RESULTS_TRIALS.append(trials_df)
        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "LightGBM_Multivar",
            "parametros": str(best_params),
            "horizonte_usado": "serie_completa",
        })

        pred = forecast_tree_tuned(
            LGBMRegressor,
            train,
            train_exog,
            test_exog,
            horizon,
            best_params,
            WINDOW_DEFAULT,
        )
        metricas = calcular_metricas(test, pred)

        if metricas:
            guardar_metricas(nombre_serie, "LightGBM_Multivar_Tuned", True, metricas, "serie_completa")
            guardar_predicciones(nombre_serie, fechas_test, pred, "LightGBM_Multivar_Tuned")
            print(f"      LightGBM_Multivar_Tuned: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      LightGBM_Multivar_Tuned Error: {str(e)[:120]}")

    try:
        print("      LSTM multivariada ultimo anio tuning...")
        serie_lstm = serie[-LSTM_LAST_HOURS:]
        fechas_lstm = fechas[-LSTM_LAST_HOURS:]
        exog_lstm = exog_alineada.iloc[-LSTM_LAST_HOURS:].reset_index(drop=True)

        test_size_lstm = max(24 * 7, int(len(serie_lstm) * 0.10))

        train_lstm = serie_lstm[:-test_size_lstm]
        test_lstm = serie_lstm[-test_size_lstm:]
        train_exog_lstm = exog_lstm.iloc[:-test_size_lstm].reset_index(drop=True)
        test_exog_lstm = exog_lstm.iloc[-test_size_lstm:].reset_index(drop=True)
        fechas_test_lstm = fechas_lstm[-test_size_lstm:]

        val_size_lstm = max(24 * 7, int(len(train_lstm) * 0.10))

        best_params, trials_df = tune_lstm(
            train_lstm[:-val_size_lstm],
            train_exog_lstm.iloc[:-val_size_lstm].reset_index(drop=True),
            train_lstm[-val_size_lstm:],
            train_exog_lstm.iloc[-val_size_lstm:].reset_index(drop=True),
            nombre_serie,
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "LSTM_Multivar"
        RESULTS_TRIALS.append(trials_df)
        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "LSTM_Multivar",
            "parametros": str(best_params),
            "horizonte_usado": "ultimo_anio",
        })

        pred = forecast_lstm_tuned(
            train_lstm,
            train_exog_lstm,
            test_exog_lstm,
            len(test_lstm),
            best_params,
        )
        metricas = calcular_metricas(test_lstm, pred)

        if metricas:
            guardar_metricas(nombre_serie, "LSTM_Multivar_Tuned_1Y", True, metricas, "ultimo_anio")
            guardar_predicciones(nombre_serie, fechas_test_lstm, pred, "LSTM_Multivar_Tuned_1Y")
            print(f"      LSTM_Multivar_Tuned_1Y: MAPE={metricas['MAPE']:.2f}%")

    except Exception as e:
        print(f"      LSTM_Multivar_Tuned_1Y Error: {str(e)[:120]}")


def imprimir_resultados(df_metricas):
    if df_metricas is None or len(df_metricas) == 0:
        print("\nNo hay resultados para mostrar")
        return

    print("\n" + "=" * 100)
    print("MEJOR MODELO POR SERIE")
    print("=" * 100)
    mejor_por_serie = df_metricas.loc[df_metricas.groupby("serie")["MAPE"].idxmin()]
    print(mejor_por_serie[["serie", "modelo", "horizonte_usado", "MAPE", "sMAPE", "MAE", "RMSE"]].to_string(index=False))

    print("\n" + "=" * 100)
    print("RANKING GLOBAL DE MODELOS")
    print("=" * 100)
    ranking = (
        df_metricas
        .groupby("modelo")["MAPE"]
        .agg(["mean", "std"])
        .round(2)
        .sort_values("mean")
    )
    print(ranking.to_string())


# =========================================================
# EJECUCION PRINCIPAL
# =========================================================

def ejecutar_pipeline():
    print("=" * 80)
    print("PIPELINE HORARIO MULTIVARIADO DESDE EXOGENAS YA EXPANDIDAS")
    print("=" * 80)

    df_exog = construir_exogenas_horarias_desde_colab()
    regiones = cargar_regiones()

    for region, df in regiones.items():
        try:
            objetivos = {
                f"{region}_DEMANDA": COL_DEMANDA,
                f"{region}_GENERACION": COL_GENERACION,
            }

            for nombre_serie, columna in objetivos.items():
                print(f"\n{'=' * 80}")
                print(f"Serie: {nombre_serie}")
                print(f"{'=' * 80}")

                serie, fechas = extraer_serie_horaria(df, columna, nombre_serie)
                exog_alineada = alinear_exogenas(fechas, df_exog)

                print("\n   Serie horaria:")
                print(f"      Longitud: {len(serie):,} observaciones")
                print(f"      Rango: {fechas[0]} a {fechas[-1]}")
                print(f"      Minimo: {np.nanmin(serie):.2f}")
                print(f"      Maximo: {np.nanmax(serie):.2f}")
                print(f"      Media: {np.nanmean(serie):.2f}")

                evaluar_serie(nombre_serie, serie, fechas, exog_alineada)

            guardar_snapshot(region)

        except Exception as e:
            print(f"ERROR en region {region}: {str(e)[:200]}")
            guardar_snapshot(f"{region}_CON_ERROR")

    df_metricas = guardar_final()
    imprimir_resultados(df_metricas)

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETADO")
    print(f"Archivos guardados en: {DRIVE_OUT_DIR}")
    print("=" * 80)


ejecutar_pipeline()


Mounted at /content/drive
PIPELINE HORARIO MULTIVARIADO DESDE EXOGENAS YA EXPANDIDAS
Temperaturas    | 2019-01-01 00:00:00 -> 2026-03-31 23:00:00 | 63,528 filas
Primarias       | 2019-01-01 00:00:00 -> 2026-03-31 23:00:00 | 63,528 filas
Secundarias     | 2019-01-01 00:00:00 -> 2026-03-31 23:00:00 | 63,528 filas
Terciarias      | 2019-01-01 00:00:00 -> 2026-03-31 23:00:00 | 63,528 filas
IGAE            | 2019-01-01 00:00:00 -> 2026-03-31 23:00:00 | 63,528 filas

Exogenas combinadas:
             datetime  Temperaturas   Primarias  Secundarias  Terciarias  \
0 2019-01-01 00:00:00          16.6  112.521216    97.336063   98.419566   
1 2019-01-01 01:00:00          16.6  112.521216    97.336063   98.419566   
2 2019-01-01 02:00:00          16.6  112.521216    97.336063   98.419566   
3 2019-01-01 03:00:00          16.6  112.521216    97.336063   98.419566   
4 2019-01-01 04:00:00          16.6  112.521216    97.336063   98.419566   

        IGAE  
0  98.524359  
1  98.524359  
2  98.52435

[I 2026-07-04 21:16:06,212] A new study created in RDB with name: BCA_DEMANDA_xgboost_multivariado
[I 2026-07-04 21:16:23,762] Trial 0 finished with value: 0.8969048868312265 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.8969048868312265.
[I 2026-07-04 21:16:39,670] Trial 1 finished with value: 1.0144587084291514 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 0.8969048868312265.
[I 2026-07-04 21:16:45,791] Trial 2 finished with value: 0.9336722249608251 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rat

      XGBoost_Multivar_Tuned: MAPE=60.26%
      LightGBM multivariado tuning...


[I 2026-07-04 21:21:01,717] Trial 0 finished with value: 0.9527413473817357 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'num_leaves': 54, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.9527413473817357.
[I 2026-07-04 21:21:12,107] Trial 1 finished with value: 1.1156312602071121 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'num_leaves': 79, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 0.9527413473817357.
[I 2026-07-04 21:21:17,135] Trial 2 finished with value: 1.033255171309354 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'num_leaves': 34, 'subsample': 0.8835558684167139, 'colsample_bytree': 0.741848158195

      LightGBM_Multivar_Tuned: MAPE=51.93%
      LSTM multivariada ultimo anio tuning...


...

      XGBoost_Multivar_Tuned: MAPE=44.34%
      LightGBM multivariado tuning...


[I 2026-07-05 00:23:45,378] Trial 0 finished with value: 4.07902554102423 and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'num_leaves': 54, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 4.07902554102423.
[I 2026-07-05 00:23:55,437] Trial 1 finished with value: 4.239327171065857 and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'num_leaves': 79, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 4.07902554102423.
[I 2026-07-05 00:24:01,935] Trial 2 finished with value: 4.116095681349939 and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'num_leaves': 34, 'subsample': 0.8835558684167139, 'colsample_bytree': 0.7418481581956126, '

      LightGBM_Multivar_Tuned: MAPE=49.39%
      LSTM multivariada ultimo anio tuning...


[I 2026-07-05 00:27:28,731] Trial 0 finished with value: 29.537542408036856 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 29.537542408036856.
[I 2026-07-05 00:28:49,818] Trial 1 finished with value: 17.940322455125198 and parameters: {'units': 16, 'dropout': 0.1, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 17.940322455125198.
[I 2026-07-05 00:30:14,028] Trial 2 finished with value: 22.16399178251636 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 128, 'window': 168, 'learning_rate': 0.001}. Best is trial 1 with value: 17.940322455125198.


      LSTM_Multivar_Tuned_1Y: MAPE=11.07%
Snapshot guardado en Drive hasta region PEN

MEJOR MODELO POR SERIE
         serie                  modelo horizonte_usado      MAPE     sMAPE         MAE        RMSE
   BCA_DEMANDA  LSTM_Multivar_Tuned_1Y     ultimo_anio 24.758923 21.560635  436.571196  471.964256
BCA_GENERACION  XGBoost_Multivar_Tuned  serie_completa 12.418524 13.449754  218.182625  278.803606
   CEN_DEMANDA  XGBoost_Multivar_Tuned  serie_completa  6.047267  6.170283  410.213728  494.106173
CEN_GENERACION  LSTM_Multivar_Tuned_1Y     ultimo_anio 11.203185 10.647725  455.241582  569.522634
   NES_DEMANDA  LSTM_Multivar_Tuned_1Y     ultimo_anio 13.305539 14.227750 1051.509319 1277.794474
NES_GENERACION  XGBoost_Multivar_Tuned  serie_completa 11.110315 12.006253 1108.850925 1410.613710
   NOR_DEMANDA  LSTM_Multivar_Tuned_1Y     ultimo_anio  8.150624  7.716526  269.059830  353.301133
NOR_GENERACION  LSTM_Multivar_Tuned_1Y     ultimo_anio  7.954110  7.488695  293.978889  390.394376

#Ahora sí multivariado a 1 semana XGBoost

In [32]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 29.7 MB/s eta 0:00:00


In [36]:
# =========================================================
# PIPELINE XGBOOST DEMANDA
# Multivariado con exogenas seleccionables
#
# Train:
#   - Ultimas 2 semanas = 336 horas
#
# Test:
#   - Siguiente semana = 168 horas
#
# Exogenas conocidas en horizonte:
#   - Temperatura
#   - IGAE
#
# Exogenas NO conocidas en horizonte:
#   - Primarias
#   - Secundarias
#   - Terciarias
#   - Generacion
#   - Importacion
#   - Exportacion
#
# Para estas ultimas, el horizonte futuro se aproxima
# usando exclusivamente informacion anterior al test.
# =========================================================


# =========================================================
# IMPORTS
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import numpy as np
import pandas as pd
import optuna

from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


# =========================================================
# CONFIG GENERAL
# =========================================================

MIN_OBS = 24 * 14

FORECAST_HORIZON = 24 * 7          # 168 horas
TRAIN_LAST_HOURS = 24 * 14         # 336 horas = 2 semanas

WINDOW_DEFAULT = 168               # una semana de lags

VAL_GAP = 24 * 7
VAL_SPLITS = 3

DRIVE_OUTPUT_DIR = (
    "/content/drive/MyDrive/Pipeline_Resultados"
)

os.makedirs(
    DRIVE_OUTPUT_DIR,
    exist_ok=True
)


# =========================================================
# ARCHIVOS DE REGIONES
# =========================================================

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}


# =========================================================
# COLUMNAS
# =========================================================

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = (
    "Estimacion de Demanda por Balance (MWh)"
)


# =========================================================
# EXOGENAS
#
# COMENTA CUALQUIER LINEA PARA EXCLUIR ESA VARIABLE.
# POR DEFECTO ENTRAN TODAS.
# =========================================================

EXOG_COLS = [
    "Temperatura",
    "Primarias",
    "Secundarias",
    "Terciarias",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]


# =========================================================
# EXOGENAS GLOBALES
# =========================================================

EXOG_SOURCE_MAP = {
    "Temperatura": "Temperaturas_H",
    "Primarias": "Primarias_H",
    "Secundarias": "Secundarias_H",
    "Terciarias": "Terciarias_H",
    "IGAE": "IGAE_H",
}


# =========================================================
# EXOGENAS QUE SI SE CONOCEN EN EL HORIZONTE
# =========================================================

EXOG_CONOCIDAS_FUTURO = [
    "Temperatura",
    "IGAE",
]


# =========================================================
# EXOGENAS QUE NO SE CONOCEN EN EL HORIZONTE
#
# Para estas NO se utilizaran los valores reales
# correspondientes a las 168 horas de test.
# =========================================================

EXOG_NO_CONOCIDAS_FUTURO = [
    "Primarias",
    "Secundarias",
    "Terciarias",
    "Generacion",
    "Importacion",
    "Exportacion",
]


# =========================================================
# LAGS PARA APROXIMAR EXOGENAS FUTURAS
#
# Misma hora:
#   - hace 1 semana
#   - hace 2 semanas
# =========================================================

LAG_SEMANA_1 = 24 * 7      # 168
LAG_SEMANA_2 = 24 * 14     # 336


# =========================================================
# OPTUNA / SALIDA
# =========================================================

N_TRIALS_OPTUNA = 10

OPTUNA_DB = os.path.join(
    DRIVE_OUTPUT_DIR,
    "optuna_xgboost_demanda_2semanas.db"
)

SAVE_PREFIX = (
    "xgboost_demanda_horario_2semanas"
)

PIPELINE_NAME = (
    "PIPELINE XGBOOST DEMANDA - TRAIN 2 SEMANAS"
)


# =========================================================
# UTILIDADES
# =========================================================

def cleanup():

    gc.collect()


def mape(y_true, y_pred):

    y_true = np.array(
        y_true,
        dtype=float
    )

    y_pred = np.array(
        y_pred,
        dtype=float
    )

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                / y_true[mask]
            )
        )
        * 100
    )


def smape(y_true, y_pred):

    y_true = np.array(
        y_true,
        dtype=float
    )

    y_pred = np.array(
        y_pred,
        dtype=float
    )

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                y_true[mask]
                - y_pred[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def calcular_metricas(
    y_true,
    y_pred
):

    n = min(
        len(y_true),
        len(y_pred)
    )

    y_c = np.array(
        y_true[:n],
        dtype=float
    )

    p_c = np.array(
        y_pred[:n],
        dtype=float
    )

    mask = (
        (~np.isnan(y_c))
        & (~np.isnan(p_c))
    )

    if mask.sum() == 0:
        return None

    y_c = y_c[mask]
    p_c = p_c[mask]

    return {
        "MAE": mean_absolute_error(
            y_c,
            p_c
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_c,
                p_c
            )
        ),

        "MAPE": mape(
            y_c,
            p_c
        ),

        "sMAPE": smape(
            y_c,
            p_c
        ),
    }


def _hora_a_0_23(hora):

    hora = pd.to_numeric(
        hora,
        errors="coerce"
    )

    if hora.dropna().empty:
        return hora

    if (
        hora.min() >= 1
        and hora.max() <= 24
    ):

        return (
            hora.astype(float)
            - 1
        )

    return hora.astype(float)


# =========================================================
# NORMALIZAR EXOGENA
# =========================================================

def _normalizar_exogena(
    df,
    nombre_variable
):

    aux = df.copy()

    aux.columns = (
        aux.columns
        .astype(str)
        .str.strip()
    )

    cols = {
        c.lower(): c
        for c in aux.columns
    }

    for requerida in [
        "fecha",
        "hora",
        "valor"
    ]:

        if requerida not in cols:

            raise ValueError(
                f"No existe columna "
                f"{requerida} "
                f"en {nombre_variable}"
            )

    aux = aux[
        [
            cols["fecha"],
            cols["hora"],
            cols["valor"]
        ]
    ].copy()

    aux[
        cols["fecha"]
    ] = pd.to_datetime(
        aux[cols["fecha"]],
        errors="coerce"
    )

    aux[
        "hora_0_23"
    ] = _hora_a_0_23(
        aux[cols["hora"]]
    )

    aux[
        nombre_variable
    ] = pd.to_numeric(
        aux[cols["valor"]],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            cols["fecha"],
            "hora_0_23",
            nombre_variable
        ]
    )

    aux[
        "hora_0_23"
    ] = aux[
        "hora_0_23"
    ].astype(int)

    if not aux[
        "hora_0_23"
    ].between(
        0,
        23
    ).all():

        raise ValueError(
            f"La columna hora de "
            f"{nombre_variable} "
            "debe estar en 0-23 o 1-24"
        )

    aux[
        "datetime"
    ] = (
        aux[cols["fecha"]]
        + pd.to_timedelta(
            aux["hora_0_23"],
            unit="h"
        )
    )

    return (
        aux[
            [
                "datetime",
                nombre_variable
            ]
        ]
        .groupby(
            "datetime",
            as_index=False
        )
        .mean()
        .sort_values(
            "datetime"
        )
    )


# =========================================================
# CONSTRUIR EXOGENAS POR REGION
# =========================================================

def merge_exogenas(region):

    exog_df = None

    # -----------------------------------------------------
    # EXOGENAS GLOBALES
    # -----------------------------------------------------

    for (
        nombre_variable,
        nombre_df
    ) in EXOG_SOURCE_MAP.items():

        # Si esta comentada en EXOG_COLS,
        # simplemente no entra.
        if (
            nombre_variable
            not in EXOG_COLS
        ):
            continue

        if (
            nombre_df
            not in globals()
        ):

            raise ValueError(
                f"No existe el DataFrame "
                f"global {nombre_df}."
            )

        aux = _normalizar_exogena(
            globals()[nombre_df],
            nombre_variable
        )

        if exog_df is None:

            exog_df = aux

        else:

            exog_df = exog_df.merge(
                aux,
                on="datetime",
                how="outer"
            )

    # -----------------------------------------------------
    # EXOGENAS ESPECIFICAS DE LA REGION
    #
    # Se leen directamente desde /content:
    #
    #   /content/CEN_GEN.csv
    #   /content/CEN_IMP.csv
    #   /content/CEN_EXP.csv
    #
    # El nombre cambia automaticamente segun la region.
    # -----------------------------------------------------

    REGION_EXOG_MAP = {
        "Generacion": f"{region}_GEN.csv",
        "Importacion": f"{region}_IMP.csv",
        "Exportacion": f"{region}_EXP.csv",
    }

    for (
        nombre_variable,
        nombre_archivo
    ) in REGION_EXOG_MAP.items():

        if (
            nombre_variable
            not in EXOG_COLS
        ):
            continue

        ruta_exog = os.path.join(
            "/content",
            nombre_archivo
        )

        if not os.path.exists(
            ruta_exog
        ):

            raise FileNotFoundError(
                f"No existe el archivo "
                f"{ruta_exog}"
            )

        df_exog_region = pd.read_csv(
            ruta_exog
        )

        aux = _normalizar_exogena(
            df_exog_region,
            nombre_variable
        )

        if exog_df is None:

            exog_df = aux

        else:

            exog_df = exog_df.merge(
                aux,
                on="datetime",
                how="outer"
            )

    if exog_df is None:

        raise ValueError(
            "No hay ninguna exogena "
            "activa en EXOG_COLS."
        )

    exog_df = (
        exog_df
        .sort_values(
            "datetime"
        )
        .reset_index(
            drop=True
        )
    )

    # Completar huecos de las exogenas.
    exog_df[
        EXOG_COLS
    ] = (
        exog_df[
            EXOG_COLS
        ]
        .ffill()
        .bfill()
    )

    print(
        f"\nExogenas cargadas para "
        f"{region}:"
    )

    for col in EXOG_COLS:

        print(
            f"   {col:15s} | "
            f"{exog_df[col].notna().sum():,} "
            "valores"
        )

    return exog_df[
        ["datetime"]
        + EXOG_COLS
    ]


# =========================================================
# ALINEAR EXOGENAS A LA SERIE OBJETIVO
# =========================================================

def alinear_exogenas_a_fechas(
    fechas,
    exogenas_df
):

    if exogenas_df is None:

        raise ValueError(
            "exogenas_df "
            "no puede ser None"
        )

    base = pd.DataFrame({
        "datetime": pd.to_datetime(
            fechas,
            errors="coerce"
        )
    })

    aux = exogenas_df.copy()

    aux[
        "datetime"
    ] = pd.to_datetime(
        aux["datetime"],
        errors="coerce"
    )

    aux = (
        aux
        .dropna(
            subset=["datetime"]
        )
        .sort_values(
            "datetime"
        )
    )

    out = base.merge(
        aux[
            ["datetime"]
            + EXOG_COLS
        ],
        on="datetime",
        how="left"
    )

    # Solo las exogenas se rellenan.
    # La demanda nunca se modifica aquí.

    out[
        EXOG_COLS
    ] = (
        out[
            EXOG_COLS
        ]
        .ffill()
        .bfill()
    )

    if out[
        EXOG_COLS
    ].isna().any().any():

        raise ValueError(
            "No hay exogenas "
            "suficientes para "
            "las fechas de la serie"
        )

    return (
        out[
            EXOG_COLS
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# CONSTRUIR EXOGENAS DEL HORIZONTE
# SIN UTILIZAR EL FUTURO REAL DE LAS VARIABLES OPERATIVAS
# =========================================================

def construir_future_exog(
    exog_serie,
    train_end,
    horizon
):

    """
    Para Temperatura e IGAE:
        utiliza el valor correspondiente
        al horizonte.

    Para las demas exogenas:
        NO utiliza el valor real futuro.

        Cada hora se estima con:

        promedio(
            misma hora hace 1 semana,
            misma hora hace 2 semanas
        )
    """

    future = (
        exog_serie
        .iloc[
            train_end:
            train_end + horizon
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    for variable in (
        EXOG_NO_CONOCIDAS_FUTURO
    ):

        if (
            variable
            not in EXOG_COLS
        ):
            continue

        valores_estimados = []

        for h in range(
            horizon
        ):

            indice_futuro = (
                train_end
                + h
            )

            idx_semana_1 = (
                indice_futuro
                - LAG_SEMANA_1
            )

            idx_semana_2 = (
                indice_futuro
                - LAG_SEMANA_2
            )

            if idx_semana_2 < 0:

                raise ValueError(
                    f"No existen "
                    f"2 semanas previas "
                    f"para estimar "
                    f"{variable}"
                )

            valor_1 = float(
                exog_serie
                .iloc[
                    idx_semana_1
                ][variable]
            )

            valor_2 = float(
                exog_serie
                .iloc[
                    idx_semana_2
                ][variable]
            )

            estimado = float(
                np.mean(
                    [
                        valor_1,
                        valor_2
                    ]
                )
            )

            valores_estimados.append(
                estimado
            )

        future[
            variable
        ] = valores_estimados

    return (
        future[
            EXOG_COLS
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# VALIDAR HORIZONTE
# =========================================================

def validar_horizonte(
    modelo,
    pred
):

    if len(pred) != (
        FORECAST_HORIZON
    ):

        raise ValueError(
            f"{modelo}: "
            f"prediccion con "
            f"{len(pred)} horas; "
            f"se esperaban "
            f"{FORECAST_HORIZON}"
        )


# =========================================================
# CARGAR REGIONES
# =========================================================

def cargar_regiones():

    regiones = {}

    for (
        region,
        archivo
    ) in ARCHIVOS_REGIONES.items():

        if not os.path.exists(
            archivo
        ):

            print(
                f"No encontre "
                f"{archivo}, "
                f"salto {region}"
            )

            continue

        df = pd.read_csv(
            archivo
        )

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
        )

        regiones[
            region
        ] = df

        print(
            f"OK {region}: "
            f"{archivo} "
            f"cargado con "
            f"shape {df.shape}"
        )

    return regiones


# =========================================================
# EXTRAER SERIE HORARIA
# =========================================================

def extraer_serie_horaria(
    df,
    columna,
    nombre_serie
):

    if columna not in df.columns:

        raise ValueError(
            f"No existe columna "
            f"{columna} "
            f"en {nombre_serie}"
        )

    if COL_FECHA not in df.columns:

        raise ValueError(
            f"No existe columna "
            f"'{COL_FECHA}' "
            f"en {nombre_serie}"
        )

    if COL_HORA not in df.columns:

        raise ValueError(
            f"No existe columna "
            f"'{COL_HORA}' "
            f"en {nombre_serie}"
        )

    aux = df[
        [
            COL_FECHA,
            COL_HORA,
            columna
        ]
    ].copy()

    aux[
        COL_FECHA
    ] = pd.to_datetime(
        aux[COL_FECHA],
        errors="coerce"
    )

    aux[
        COL_HORA
    ] = pd.to_numeric(
        aux[COL_HORA],
        errors="coerce"
    )

    aux[
        columna
    ] = pd.to_numeric(
        aux[columna],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            COL_FECHA,
            COL_HORA
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            COL_HORA
        ]
        .astype(int)
        - 1
    )

    aux[
        "datetime"
    ] = (
        aux[
            COL_FECHA
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = aux.sort_values(
        "datetime"
    )

    return (
        aux[
            columna
        ].values.astype(float),

        aux[
            "datetime"
        ].values
    )


# =========================================================
# FEATURES HORARIAS
# =========================================================

def create_feature_df(
    y,
    window=168,
    exog=None
):

    df = pd.DataFrame({
        "y": np.asarray(
            y,
            dtype=float
        )
    })

    if exog is not None:

        exog_df = (
            pd.DataFrame(
                exog
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            exog_df
        ) != len(df):

            raise ValueError(
                "La longitud de exog "
                "debe coincidir con y"
            )

        for col in EXOG_COLS:

            if col not in (
                exog_df.columns
            ):

                raise ValueError(
                    f"Falta variable "
                    f"exogena {col}"
                )

            df[col] = pd.to_numeric(
                exog_df[col],
                errors="coerce"
            )

    # -----------------------------------------------------
    # LAGS DE DEMANDA
    # -----------------------------------------------------

    for lag in range(
        1,
        window + 1
    ):

        df[
            f"lag_{lag}"
        ] = df[
            "y"
        ].shift(
            lag
        )

    # -----------------------------------------------------
    # ROLLING FEATURES
    # -----------------------------------------------------

    y_past = df[
        "y"
    ].shift(1)

    df[
        "rolling_mean_24"
    ] = (
        y_past
        .rolling(24)
        .mean()
    )

    df[
        "rolling_std_24"
    ] = (
        y_past
        .rolling(24)
        .std()
    )

    df[
        "rolling_mean_168"
    ] = (
        y_past
        .rolling(168)
        .mean()
    )

    df[
        "rolling_std_168"
    ] = (
        y_past
        .rolling(168)
        .std()
    )

    df[
        "trend"
    ] = np.arange(
        len(df)
    )

    return df.dropna()


def create_features_from_history(
    hist_y,
    window=168,
    exog_row=None
):

    features = {}

    if exog_row is not None:

        if isinstance(
            exog_row,
            pd.Series
        ):

            exog_values = (
                exog_row.to_dict()
            )

        else:

            exog_values = dict(
                exog_row
            )

        for col in EXOG_COLS:

            if col not in (
                exog_values
            ):

                raise ValueError(
                    f"Falta variable "
                    f"exogena futura "
                    f"{col}"
                )

            features[
                col
            ] = float(
                exog_values[col]
            )

    # -----------------------------------------------------
    # LAGS
    # -----------------------------------------------------

    for lag in range(
        1,
        window + 1
    ):

        features[
            f"lag_{lag}"
        ] = (
            hist_y[-lag]
            if len(hist_y) >= lag
            else hist_y[0]
        )

    # -----------------------------------------------------
    # ROLLING
    # -----------------------------------------------------

    features[
        "rolling_mean_24"
    ] = np.mean(
        hist_y[-24:]
    )

    features[
        "rolling_std_24"
    ] = np.std(
        hist_y[-24:]
    )

    features[
        "rolling_mean_168"
    ] = np.mean(
        hist_y[-168:]
    )

    features[
        "rolling_std_168"
    ] = np.std(
        hist_y[-168:]
    )

    features[
        "trend"
    ] = len(
        hist_y
    )

    return pd.DataFrame(
        [features]
    )


# =========================================================
# XGBOOST - OBJECTIVE OPTUNA
# =========================================================

def objective_xgboost(
    trial,
    train_y,
    val_y,
    window,
    train_exog=None,
    val_exog=None
):

    params = {

        "n_estimators":
            trial.suggest_int(
                "n_estimators",
                50,
                200
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                2,
                8
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.03,
                0.25
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

        "min_child_weight":
            trial.suggest_int(
                "min_child_weight",
                1,
                8
            ),

        "reg_alpha":
            trial.suggest_float(
                "reg_alpha",
                0.0,
                5.0
            ),

        "reg_lambda":
            trial.suggest_float(
                "reg_lambda",
                0.0,
                5.0
            ),

        "random_state": 42,
    }

    df_train = create_feature_df(
        train_y,
        window,
        train_exog
    )

    if len(df_train) < 50:

        return float(
            "inf"
        )

    X_train = df_train.drop(
        columns=["y"]
    )

    y_train = df_train[
        "y"
    ]

    # -----------------------------------------------------
    # VALIDACION
    # -----------------------------------------------------

    val_context_y = np.concatenate(
        [
            train_y,
            val_y
        ]
    )

    val_context_exog = pd.concat(
        [
            pd.DataFrame(
                train_exog
            ),
            pd.DataFrame(
                val_exog
            )
        ],
        ignore_index=True
    )

    df_val_all = create_feature_df(
        val_context_y,
        window,
        val_context_exog
    )

    if len(
        df_val_all
    ) < len(val_y):

        return float(
            "inf"
        )

    X_val = (
        df_val_all
        .drop(
            columns=["y"]
        )
        .iloc[
            -len(val_y):
        ]
    )

    y_val = (
        df_val_all[
            "y"
        ]
        .iloc[
            -len(val_y):
        ]
    )

    model = XGBRegressor(
        **params,
        objective=(
            "reg:squarederror"
        ),
        n_jobs=1
    )

    model.fit(
        X_train,
        y_train
    )

    preds = model.predict(
        X_val
    )

    score = smape(
        y_val.values,
        preds
    )

    return (
        score
        if not np.isnan(score)
        else float("inf")
    )


# =========================================================
# OPTUNA
# =========================================================

def tune_xgboost(
    train_y,
    val_y,
    nombre_serie,
    window=168,
    train_exog=None,
    val_exog=None
):

    study = optuna.create_study(

        direction="minimize",

        sampler=TPESampler(
            seed=42
        ),

        study_name=(
            f"{nombre_serie}"
            "_xgboost_2semanas"
        ),

        storage=(
            f"sqlite:///"
            f"{OPTUNA_DB}"
        ),

        load_if_exists=True,
    )

    study.optimize(

        lambda trial:
            objective_xgboost(
                trial,
                train_y,
                val_y,
                window,
                train_exog,
                val_exog
            ),

        n_trials=(
            N_TRIALS_OPTUNA
        ),

        show_progress_bar=False,
    )

    return (
        study.best_params,
        study.trials_dataframe()
    )


# =========================================================
# FORECAST RECURSIVO
# =========================================================

def forecast_xgboost_tuned(
    train_y,
    horizon,
    best_params,
    window=168,
    train_exog=None,
    future_exog=None
):

    try:

        if (
            future_exog is None
            or len(
                future_exog
            ) != horizon
        ):

            raise ValueError(
                "future_exog debe "
                "tener exactamente "
                "horizon filas"
            )

        df_train = create_feature_df(
            train_y,
            window,
            train_exog
        )

        X_train = (
            df_train
            .drop(
                columns=["y"]
            )
        )

        y_train = df_train[
            "y"
        ]

        model = XGBRegressor(

            **best_params,

            objective=(
                "reg:squarederror"
            ),

            random_state=42,

            n_jobs=1
        )

        model.fit(
            X_train,
            y_train
        )

        preds = []

        hist = list(
            train_y
        )

        future_exog = (
            pd.DataFrame(
                future_exog
            )
            .reset_index(
                drop=True
            )
        )

        for step in range(
            horizon
        ):

            X_future = (
                create_features_from_history(
                    hist,
                    window,
                    future_exog.iloc[
                        step
                    ]
                )
            )

            X_future = (
                X_future
                .reindex(
                    columns=(
                        X_train.columns
                    )
                )
            )

            pred = model.predict(
                X_future
            )[0]

            preds.append(
                pred
            )

            # Forecast recursivo:
            # la prediccion pasa a formar
            # parte de la historia.

            hist.append(
                pred
            )

        return np.array(
            preds
        )

    except Exception as e:

        print(
            "         Error forecast "
            f"XGBoost: "
            f"{str(e)[:120]}"
        )

        return np.full(
            horizon,
            np.nan
        )


# =========================================================
# RESULTADOS
# =========================================================

RESULTS_SERIES = []
RESULTS_METRICS = []
RESULTS_TRIALS = []
RESULTS_CONFIG_USADA = []


def guardar_predicciones(
    nombre_serie,
    fechas_test,
    pred,
    modelo
):

    for j, pred_val in enumerate(
        pred
    ):

        if j < len(
            fechas_test
        ):

            RESULTS_SERIES.append({

                "serie":
                    nombre_serie,

                "fecha":
                    fechas_test[j],

                "tipo":
                    "prediccion",

                "subset":
                    "test",

                "modelo":
                    modelo,

                "valor":
                    pred_val,
            })


def guardar_metricas(
    nombre_serie,
    modelo,
    tuneado,
    metricas,
    horizonte_usado
):

    RESULTS_METRICS.append({

        "serie":
            nombre_serie,

        "modelo":
            modelo,

        "tuneado":
            tuneado,

        "horizonte_usado":
            horizonte_usado,

        "MAPE":
            metricas["MAPE"],

        "sMAPE":
            metricas["sMAPE"],

        "MAE":
            metricas["MAE"],

        "RMSE":
            metricas["RMSE"],
    })


# =========================================================
# GUARDAR TODOS LOS CSV
# =========================================================

def guardar_todos_csv():

    os.makedirs(
        DRIVE_OUTPUT_DIR,
        exist_ok=True
    )

    df_metrics = None

    # -----------------------------------------------------
    # SERIES
    # -----------------------------------------------------

    if RESULTS_SERIES:

        df_series = pd.DataFrame(
            RESULTS_SERIES
        )

        df_series[
            "fecha"
        ] = pd.to_datetime(
            df_series["fecha"],
            errors="coerce"
        )

        df_series = (
            df_series
            .sort_values(
                [
                    "serie",
                    "fecha",
                    "modelo"
                ]
            )
        )

        df_series.to_csv(

            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_series.csv"
            ),

            index=False,
            encoding="utf-8-sig"
        )

        print(
            "\nOK Series guardadas: "
            f"{len(df_series):,} "
            "registros"
        )

    # -----------------------------------------------------
    # METRICAS
    # -----------------------------------------------------

    if RESULTS_METRICS:

        df_metrics = pd.DataFrame(
            RESULTS_METRICS
        )

        df_metrics = (
            df_metrics
            .sort_values(
                [
                    "serie",
                    "MAPE"
                ]
            )
        )

        df_metrics.to_csv(

            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_metricas.csv"
            ),

            index=False,
            encoding="utf-8-sig"
        )

        print(
            "OK Metricas guardadas: "
            f"{len(df_metrics):,} "
            "registros"
        )

    # -----------------------------------------------------
    # TRIALS
    # -----------------------------------------------------

    if RESULTS_TRIALS:

        df_trials = pd.concat(
            RESULTS_TRIALS,
            ignore_index=True
        )

        df_trials.to_csv(

            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_optuna_trials.csv"
            ),

            index=False,
            encoding="utf-8-sig"
        )

        print(
            "OK Trials Optuna guardados: "
            f"{len(df_trials):,} "
            "registros"
        )

    # -----------------------------------------------------
    # CONFIG
    # -----------------------------------------------------

    if RESULTS_CONFIG_USADA:

        df_config = pd.DataFrame(
            RESULTS_CONFIG_USADA
        )

        df_config.to_csv(

            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_config_usada.csv"
            ),

            index=False,
            encoding="utf-8-sig"
        )

        print(
            "OK Configuracion usada guardada: "
            f"{len(df_config):,} "
            "registros"
        )

    return df_metrics


# =========================================================
# IMPRIMIR RESULTADOS
# =========================================================

def imprimir_resultados(
    df_metricas
):

    if (
        df_metricas is None
        or len(df_metricas) == 0
    ):

        print(
            "\nNo hay resultados "
            "para mostrar"
        )

        return

    print(
        "\n"
        + "=" * 100
    )

    print(
        "MEJOR MODELO POR SERIE"
    )

    print(
        "=" * 100
    )

    mejor_por_serie = (
        df_metricas.loc[
            df_metricas
            .groupby(
                "serie"
            )[
                "MAPE"
            ]
            .idxmin()
        ]
    )

    print(
        mejor_por_serie[
            [
                "serie",
                "modelo",
                "horizonte_usado",
                "MAPE",
                "sMAPE",
                "MAE",
                "RMSE"
            ]
        ]
        .to_string(
            index=False
        )
    )

    print(
        "\n"
        + "=" * 100
    )

    print(
        "RANKING GLOBAL DE MODELOS"
    )

    print(
        "=" * 100
    )

    ranking = (
        df_metricas
        .groupby(
            "modelo"
        )[
            "MAPE"
        ]
        .agg(
            [
                "mean",
                "std"
            ]
        )
        .round(2)
        .sort_values(
            "mean"
        )
    )

    print(
        ranking.to_string()
    )


# =========================================================
# PREPARAR TRAIN / TEST
# =========================================================

def _preparar_serie(
    nombre_serie,
    serie,
    fechas,
    exogenas_df
):

    required_hours = (
        TRAIN_LAST_HOURS
        + FORECAST_HORIZON
    )

    if len(serie) < (
        required_hours
    ):

        print(
            f"   Serie insuficiente: "
            f"{nombre_serie}"
        )

        return None

    # -----------------------------------------------------
    # IMPORTANTE:
    # Se conserva SOLO:
    #
    # 336 horas train
    # +168 horas test
    # -----------------------------------------------------

    start = (
        len(serie)
        - required_hours
    )

    serie_reciente = (
        serie[
            start:
        ]
    )

    fechas_recientes = (
        fechas[
            start:
        ]
    )

    # -----------------------------------------------------
    # ALINEAR EXOGENAS A TODO EL PERIODO ORIGINAL
    # -----------------------------------------------------

    exog_completa = (
        alinear_exogenas_a_fechas(
            fechas,
            exogenas_df
        )
    )

    # Necesitamos conservar tambien
    # historia anterior para poder obtener
    # t-336 al construir las exogenas futuras.

    exog_inicio = max(
        0,
        start - LAG_SEMANA_2
    )

    exog_contexto = (
        exog_completa
        .iloc[
            exog_inicio:
        ]
        .reset_index(
            drop=True
        )
    )

    # Posicion donde empieza
    # nuestro train de 336 horas
    # dentro de exog_contexto

    offset_train = (
        start
        - exog_inicio
    )

    # -----------------------------------------------------
    # TRAIN / TEST OBJETIVO
    # -----------------------------------------------------

    train = (
        serie_reciente[
            :TRAIN_LAST_HOURS
        ]
    )

    test = (
        serie_reciente[
            TRAIN_LAST_HOURS:
        ]
    )

    fechas_test = (
        fechas_recientes[
            TRAIN_LAST_HOURS:
        ]
    )

    # -----------------------------------------------------
    # TRAIN EXOG
    # -----------------------------------------------------

    train_exog = (
        exog_contexto
        .iloc[
            offset_train:
            offset_train
            + TRAIN_LAST_HOURS
        ]
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # FUTURE EXOG
    #
    # train_end_contexto es la posicion
    # donde comienza el test dentro
    # de exog_contexto.
    # -----------------------------------------------------

    train_end_contexto = (
        offset_train
        + TRAIN_LAST_HOURS
    )

    test_exog = (
        construir_future_exog(
            exog_serie=(
                exog_contexto
            ),
            train_end=(
                train_end_contexto
            ),
            horizon=(
                FORECAST_HORIZON
            )
        )
    )

    # -----------------------------------------------------
    # DIAGNOSTICOS
    # -----------------------------------------------------

    print(
        "\n   Split general fijo"
    )

    print(
        f"      Train: "
        f"{len(train):,} obs "
        "(2 semanas)"
    )

    print(
        f"      Test: "
        f"{len(test):,} obs "
        "(1 semana)"
    )

    print(
        f"      Horizonte: "
        f"{FORECAST_HORIZON} horas"
    )

    print(
        "\n   Exogenas activas:"
    )

    for col in EXOG_COLS:

        if col in (
            EXOG_CONOCIDAS_FUTURO
        ):

            print(
                f"      {col}: "
                "valor del horizonte"
            )

        else:

            print(
                f"      {col}: "
                "estimada con "
                "t-168 y t-336"
            )

    # -----------------------------------------------------
    # VALIDACION
    #
    # Con solo 336 horas no tiene sentido
    # conservar el esquema anterior de
    # 3 grandes splits.
    #
    # Usamos:
    #   primeras 168 h para train
    #   ultimas 168 h para validacion
    #
    # Esto mantiene orden temporal.
    # -----------------------------------------------------

    val_size = (
        FORECAST_HORIZON
    )

    tune_train_y = (
        train[
            :-val_size
        ]
    )

    tune_val_y = (
        train[
            -val_size:
        ]
    )

    tune_train_exog = (
        train_exog
        .iloc[
            :-val_size
        ]
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # EXOG VALIDACION SIN LEAKAGE
    #
    # Para la validacion:
    # las variables operativas tampoco
    # deben ver los valores reales.
    # -----------------------------------------------------

    val_start_contexto = (
        train_end_contexto
        - val_size
    )

    tune_val_exog = (
        construir_future_exog(
            exog_serie=(
                exog_contexto
            ),
            train_end=(
                val_start_contexto
            ),
            horizon=(
                val_size
            )
        )
    )

    return {

        "train":
            train,

        "test":
            test,

        "train_exog":
            train_exog,

        "test_exog":
            test_exog,

        "fechas_test":
            fechas_test,

        "horizon":
            FORECAST_HORIZON,

        "horizonte_usado":
            (
                f"{FORECAST_HORIZON}"
                "_horas"
            ),

        "tune_train_y":
            tune_train_y,

        "tune_val_y":
            tune_val_y,

        "window_tune":
            WINDOW_DEFAULT,

        "train_exog_tune":
            tune_train_exog,

        "val_exog_tune":
            tune_val_exog,
    }


# =========================================================
# EVALUAR SERIE
# =========================================================

def evaluar_serie(
    nombre_serie,
    serie,
    fechas,
    exogenas_df=None
):

    contexto = (
        _preparar_serie(
            nombre_serie,
            serie,
            fechas,
            exogenas_df
        )
    )

    if contexto is None:
        return

    try:

        print(
            "\n      XGBoost tuning..."
        )

        best_params, trials_df = (
            tune_xgboost(

                contexto[
                    "tune_train_y"
                ],

                contexto[
                    "tune_val_y"
                ],

                nombre_serie,

                contexto[
                    "window_tune"
                ],

                contexto[
                    "train_exog_tune"
                ],

                contexto[
                    "val_exog_tune"
                ],
            )
        )

        trials_df[
            "serie"
        ] = nombre_serie

        trials_df[
            "modelo"
        ] = "XGBoost"

        RESULTS_TRIALS.append(
            trials_df
        )

        RESULTS_CONFIG_USADA.append({

            "serie":
                nombre_serie,

            "modelo":
                "XGBoost",

            "parametros":
                str(
                    best_params
                ),

            "horizonte_usado":
                contexto[
                    "horizonte_usado"
                ],

            "train_horas":
                TRAIN_LAST_HOURS,

            "exogenas":
                str(
                    EXOG_COLS
                ),
        })

        # -------------------------------------------------
        # FORECAST FINAL
        # -------------------------------------------------

        pred = forecast_xgboost_tuned(

            contexto[
                "train"
            ],

            contexto[
                "horizon"
            ],

            best_params,

            contexto[
                "window_tune"
            ],

            contexto[
                "train_exog"
            ],

            contexto[
                "test_exog"
            ]
        )

        validar_horizonte(
            "XGBoost_Tuned_2Semanas",
            pred
        )

        metricas = calcular_metricas(
            contexto[
                "test"
            ],
            pred
        )

        if metricas:

            guardar_metricas(

                nombre_serie,

                "XGBoost_Tuned_2Semanas",

                True,

                metricas,

                contexto[
                    "horizonte_usado"
                ]
            )

            guardar_predicciones(

                nombre_serie,

                contexto[
                    "fechas_test"
                ],

                pred,

                "XGBoost_Tuned_2Semanas"
            )

            print(
                "      "
                "XGBoost_Tuned_2Semanas: "
                f"MAPE="
                f"{metricas['MAPE']:.2f}%"
            )

    except Exception as e:

        print(
            f"      Error evaluando "
            f"{nombre_serie}: "
            f"{type(e).__name__}: "
            f"{e}"
        )

    finally:

        cleanup()


# =========================================================
# PIPELINE PRINCIPAL
# =========================================================

def ejecutar_pipeline():

    global RESULTS_SERIES
    global RESULTS_METRICS
    global RESULTS_TRIALS
    global RESULTS_CONFIG_USADA

    RESULTS_SERIES = []
    RESULTS_METRICS = []
    RESULTS_TRIALS = []
    RESULTS_CONFIG_USADA = []

    os.makedirs(
        DRIVE_OUTPUT_DIR,
        exist_ok=True
    )

    print(
        "=" * 80
    )

    print(
        PIPELINE_NAME
    )

    print(
        "=" * 80
    )

    print(
        f"Directorio de salida: "
        f"{DRIVE_OUTPUT_DIR}"
    )

    print(
        f"Train usado: "
        f"{TRAIN_LAST_HOURS} horas"
    )

    print(
        f"Test usado: "
        f"{FORECAST_HORIZON} horas"
    )

    print(
        "\nExogenas activas:"
    )

    for exog in EXOG_COLS:

        print(
            f"   - {exog}"
        )

    # -----------------------------------------------------
    # CARGAR REGIONES
    # -----------------------------------------------------

    regiones = cargar_regiones()

    # -----------------------------------------------------
    # PROCESAR
    # -----------------------------------------------------

    for (
        region,
        df
    ) in regiones.items():

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"Serie: "
            f"{region}_DEMANDA"
        )

        print(
            "=" * 80
        )

        # Cada region necesita
        # GEN / IMP / EXP propios.

        exogenas_df = (
            merge_exogenas(
                region
            )
        )

        print(
            f"Exogenas construidas: "
            f"{len(exogenas_df):,} "
            "filas"
        )

        nombre_serie = (
            f"{region}_DEMANDA"
        )

        serie, fechas = (
            extraer_serie_horaria(
                df,
                COL_DEMANDA,
                nombre_serie
            )
        )

        print(
            f"Serie completa: "
            f"{len(serie):,} "
            "observaciones"
        )

        print(
            f"Rango: "
            f"{fechas[0]} "
            f"a "
            f"{fechas[-1]}"
        )

        evaluar_serie(
            nombre_serie,
            serie,
            fechas,
            exogenas_df
        )

        print(
            "\nGuardando avance..."
        )

        guardar_todos_csv()

    # -----------------------------------------------------
    # FINAL
    # -----------------------------------------------------

    df_metricas = (
        guardar_todos_csv()
    )

    imprimir_resultados(
        df_metricas
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        "PIPELINE COMPLETADO"
    )

    print(
        "=" * 80
    )

    print(
        os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}"
            "_series.csv"
        )
    )

    print(
        os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}"
            "_metricas.csv"
        )
    )

    print(
        os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}"
            "_optuna_trials.csv"
        )
    )

    print(
        os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}"
            "_config_usada.csv"
        )
    )


# =========================================================
# EJECUTAR
# =========================================================

ejecutar_pipeline()

PIPELINE XGBOOST DEMANDA - TRAIN 2 SEMANAS
Directorio de salida: /content/drive/MyDrive/Pipeline_Resultados
Train usado: 336 horas
Test usado: 168 horas

Exogenas activas:
   - Temperatura
   - Primarias
   - Secundarias
   - Terciarias
   - IGAE
   - Generacion
   - Importacion
   - Exportacion
OK BCA: BCA_long.csv cargado con shape (64792, 11)
OK CEN: CEN_long.csv cargado con shape (64796, 11)
OK NES: NES_long.csv cargado con shape (64796, 11)
OK NOR: NOR_long.csv cargado con shape (64796, 11)
OK NTE: NTE_long.csv cargado con shape (64796, 11)
OK OCC: OCC_long.csv cargado con shape (64796, 11)
OK ORI: ORI_long.csv cargado con shape (64796, 11)
OK PEN: PEN_long.csv cargado con shape (64796, 11)

Serie: BCA_DEMANDA

Exogenas cargadas para BCA:
   Temperatura     | 64,800 valores
   Primarias       | 64,800 valores
   Secundarias     | 64,800 valores
   Terciarias      | 64,800 valores
   IGAE            | 64,800 valores
   Generacion      | 64,800 valores
   Importacion     | 64,800 va

[I 2026-08-06 14:20:22,438] A new study created in RDB with name: BCA_DEMANDA_xgboost_2semanas
[I 2026-08-06 14:20:22,602] Trial 0 finished with value: inf and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:22,737] Trial 1 finished with value: inf and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:22,874] Trial 2 finished with value: inf and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'subsample': 0.7873687420594125, 'colsample_bytree': 0

      XGBoost_Tuned_2Semanas: MAPE=5.85%

Guardando avance...

OK Series guardadas: 168 registros
OK Metricas guardadas: 1 registros
OK Trials Optuna guardados: 10 registros
OK Configuracion usada guardada: 1 registros

Serie: CEN_DEMANDA

Exogenas cargadas para CEN:
   Temperatura     | 64,800 valores
   Primarias       | 64,800 valores
   Secundarias     | 64,800 valores
   Terciarias      | 64,800 valores
   IGAE            | 64,800 valores
   Generacion      | 64,800 valores
   Importacion     | 64,800 valores
   Exportacion     | 64,800 valores
Exogenas construidas: 64,800 filas
Serie completa: 64,796 observaciones
Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000

   Split general fijo
      Train: 336 obs (2 semanas)
      Test: 168 obs (1 semana)
      Horizonte: 168 horas

   Exogenas activas:
      Temperatura: valor del horizonte
      Primarias: estimada con t-168 y t-336
      Secundarias: estimada con t-168 y t-336
      Terciarias: estimada con t-168 y

[I 2026-08-06 14:20:27,852] A new study created in RDB with name: CEN_DEMANDA_xgboost_2semanas
[I 2026-08-06 14:20:28,021] Trial 0 finished with value: inf and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:28,171] Trial 1 finished with value: inf and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:28,314] Trial 2 finished with value: inf and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'subsample': 0.7873687420594125, 'colsample_bytree': 0

      XGBoost_Tuned_2Semanas: MAPE=3.36%

Guardando avance...

OK Series guardadas: 336 registros
OK Metricas guardadas: 2 registros
OK Trials Optuna guardados: 20 registros
OK Configuracion usada guardada: 2 registros

Serie: NES_DEMANDA

Exogenas cargadas para NES:
   Temperatura     | 64,800 valores
   Primarias       | 64,800 valores
   Secundarias     | 64,800 valores
   Terciarias      | 64,800 valores
   IGAE            | 64,800 valores
   Generacion      | 64,800 valores
   Importacion     | 64,800 valores
   Exportacion     | 64,800 valores
Exogenas construidas: 64,800 filas
Serie completa: 64,796 observaciones
Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000

   Split general fijo
      Train: 336 obs (2 semanas)
      Test: 168 obs (1 semana)
      Horizonte: 168 horas

   Exogenas activas:
      Temperatura: valor del horizonte
      Primarias: estimada con t-168 y t-336
      Secundarias: estimada con t-168 y t-336
      Terciarias: estimada con t-168 y

...


      XGBoost tuning...


[I 2026-08-06 14:20:56,519] Trial 0 finished with value: inf and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:56,710] Trial 1 finished with value: inf and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: inf.
[I 2026-08-06 14:20:56,892] Trial 2 finished with value: inf and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'subsample': 0.7873687420594125, 'colsample_bytree': 0.8835558684167139, 'min_child_weight': 2, 'reg_alpha': 1.4607232426760908, 'reg_lambda': 1.8318

      XGBoost_Tuned_2Semanas: MAPE=5.53%

Guardando avance...

OK Series guardadas: 1,176 registros
OK Metricas guardadas: 7 registros
OK Trials Optuna guardados: 70 registros
OK Configuracion usada guardada: 7 registros

Serie: PEN_DEMANDA

Exogenas cargadas para PEN:
   Temperatura     | 64,800 valores
   Primarias       | 64,800 valores
   Secundarias     | 64,800 valores
   Terciarias      | 64,800 valores
   IGAE            | 64,800 valores
   Generacion      | 64,800 valores
   Importacion     | 64,800 valores
   Exportacion     | 64,800 valores
Exogenas construidas: 64,800 filas
Serie completa: 64,796 observaciones
Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000

   Split general fijo
      Train: 336 obs (2 semanas)
      Test: 168 obs (1 semana)
      Horizonte: 168 horas

   Exogenas activas:
      Temperatura: valor del horizonte
      Primarias: estimada con t-168 y t-336
      Secundarias: estimada con t-168 y t-336
      Terciarias: estimada con t-168

[I 2026-08-06 14:21:01,815] A new study created in RDB with name: PEN_DEMANDA_xgboost_2semanas
[I 2026-08-06 14:21:01,971] Trial 0 finished with value: inf and parameters: {'n_estimators': 106, 'max_depth': 8, 'learning_rate': 0.19103866719850912, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: inf.
[I 2026-08-06 14:21:02,116] Trial 1 finished with value: inf and parameters: {'n_estimators': 140, 'max_depth': 6, 'learning_rate': 0.03452858874507654, 'subsample': 0.9909729556485982, 'colsample_bytree': 0.9497327922401265, 'min_child_weight': 2, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: inf.
[I 2026-08-06 14:21:02,259] Trial 2 finished with value: inf and parameters: {'n_estimators': 95, 'max_depth': 5, 'learning_rate': 0.12502790410126546, 'subsample': 0.7873687420594125, 'colsample_bytree': 0

      XGBoost_Tuned_2Semanas: MAPE=7.76%

Guardando avance...

OK Series guardadas: 1,344 registros
OK Metricas guardadas: 8 registros
OK Trials Optuna guardados: 80 registros
OK Configuracion usada guardada: 8 registros

OK Series guardadas: 1,344 registros
OK Metricas guardadas: 8 registros
OK Trials Optuna guardados: 80 registros
OK Configuracion usada guardada: 8 registros

MEJOR MODELO POR SERIE
      serie                 modelo horizonte_usado     MAPE    sMAPE        MAE       RMSE
BCA_DEMANDA XGBoost_Tuned_2Semanas       168_horas 5.849112 5.645501 110.970848 141.051566
CEN_DEMANDA XGBoost_Tuned_2Semanas       168_horas 3.364265 3.427817 243.668040 283.799776
NES_DEMANDA XGBoost_Tuned_2Semanas       168_horas 7.319997 7.346681 593.284054 726.806758
NOR_DEMANDA XGBoost_Tuned_2Semanas       168_horas 7.467657 7.038781 246.568772 316.632453
NTE_DEMANDA XGBoost_Tuned_2Semanas       168_horas 4.123019 4.015921 167.602892 207.738228
OCC_DEMANDA XGBoost_Tuned_2Semanas       168_horas

##Meter resultados a Drive

In [39]:
import os
import shutil
from google.colab import drive

# ==========================================
# 1. RESPALDAR LA CARPETA FALSA /content/drive
# ==========================================

ORIGEN_FALSO = "/content/drive"
RESPALDO = "/content/drive_temporal_respaldo"

if os.path.exists(ORIGEN_FALSO):

    if os.path.exists(RESPALDO):
        shutil.rmtree(RESPALDO)

    shutil.move(
        ORIGEN_FALSO,
        RESPALDO
    )

    print(
        f"✅ Carpeta temporal movida a: "
        f"{RESPALDO}"
    )


# ==========================================
# 2. MONTAR GOOGLE DRIVE REAL
# ==========================================

drive.mount(
    "/content/drive"
)

print(
    "✅ Google Drive montado correctamente"
)


# ==========================================
# 3. CREAR DESTINO REAL EN MYDRIVE
# ==========================================

DESTINO_REAL = (
    "/content/drive/MyDrive/Pipeline_Resultados"
)

os.makedirs(
    DESTINO_REAL,
    exist_ok=True
)


# ==========================================
# 4. BUSCAR RESULTADOS EN EL RESPALDO
# ==========================================

ORIGEN_RESULTADOS = (
    "/content/drive_temporal_respaldo/"
    "MyDrive/Pipeline_Resultados"
)

if not os.path.exists(
    ORIGEN_RESULTADOS
):

    raise FileNotFoundError(
        f"No encontré los resultados en "
        f"{ORIGEN_RESULTADOS}"
    )


# ==========================================
# 5. COPIAR RESULTADOS AL DRIVE REAL
# ==========================================

for archivo in os.listdir(
    ORIGEN_RESULTADOS
):

    origen = os.path.join(
        ORIGEN_RESULTADOS,
        archivo
    )

    destino = os.path.join(
        DESTINO_REAL,
        archivo
    )

    if os.path.isfile(origen):

        shutil.copy2(
            origen,
            destino
        )

        print(
            f"✅ Copiado: {archivo}"
        )


print(
    "\n🎉 Resultados copiados a tu "
    "Google Drive real:"
)

print(
    DESTINO_REAL
)

✅ Carpeta temporal movida a: /content/drive_temporal_respaldo
Mounted at /content/drive
✅ Google Drive montado correctamente
✅ Copiado: xgboost_demanda_horario_2semanas_optuna_trials.csv
✅ Copiado: xgboost_demanda_horario_2semanas_config_usada.csv
✅ Copiado: optuna_xgboost_demanda_2semanas.db
✅ Copiado: xgboost_demanda_horario_2semanas_metricas.csv
✅ Copiado: xgboost_demanda_horario_2semanas_series.csv

🎉 Resultados copiados a tu Google Drive real:
/content/drive/MyDrive/Pipeline_Resultados


#LSTM: No funcionó!

La red LSTM implementada corresponde a una arquitectura multivariada de una sola capa LSTM seguida de una capa densa de salida (`Dense(1)`), utilizando como variables exógenas la **Temperatura** y el **IGAE**. La optimización de hiperparámetros se realizó mediante **Optuna** con un **TPESampler** (`seed = 42`) durante **10 pruebas**, explorando el número de unidades de la capa LSTM (`16`, `32`), la tasa de *dropout* (`0.1`, `0.2`), el tamaño de lote (*batch size*, `64`, `128`), la longitud de la ventana temporal (`24`, `48`, `168` horas) y la tasa de aprendizaje del optimizador Adam (`0.001`, `0.003`). El modelo se entrenó con una función de pérdida **Mean Squared Error (MSE)**, un máximo de **5 épocas** y **Early Stopping** con una paciencia de **3 épocas**, empleando una ventana de entrenamiento fija de **3600 horas** y un horizonte de predicción de **168 horas (7 días)**. :contentReference[oaicite:0]{index=0} :contentReference[oaicite:1]{index=1}

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 27.4 MB/s eta 0:00:00


In [ ]:
# =========================================================
# PIPELINE LSTM DEMANDA
# Pipeline demanda multivariado final
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import numpy as np
import pandas as pd
import optuna

from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive

drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Pipeline_Resultados"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# =========================================================
# CONFIG
# =========================================================

MIN_OBS = 24 * 30
FORECAST_HORIZON = 24 * 7
LSTM_TRAIN_HOURS = 3600
WINDOW_DEFAULT = 168
VAL_GAP = 24 * 7
VAL_SPLITS = 3



ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}

COL_FECHA = "fecha"
COL_HORA = "Hora"
COL_DEMANDA = "Estimacion de Demanda por Balance (MWh)"

EXOG_COLS = [
    "Temperatura",
    "IGAE",
]

EXOG_SOURCE_MAP = {
    "Temperatura": "Temperaturas_H",
    "IGAE": "IGAE_H",
}

N_TRIALS_LSTM = 10
EPOCHS_LSTM = 5
PATIENCE_LSTM = 3
OPTUNA_DB = os.path.join(DRIVE_OUTPUT_DIR, "optuna_lstm_demanda.db")
SAVE_PREFIX = "lstm_demanda_horario"
PIPELINE_NAME = "PIPELINE LSTM DEMANDA"


# =========================================================
# UTILIDADES
# =========================================================

def cleanup():
    K.clear_session()
    gc.collect()


def mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100


def calcular_metricas(y_true, y_pred):
    n = min(len(y_true), len(y_pred))
    y_c = np.array(y_true[:n], dtype=float)
    p_c = np.array(y_pred[:n], dtype=float)
    mask = (~np.isnan(y_c)) & (~np.isnan(p_c))
    if mask.sum() == 0:
        return None
    y_c = y_c[mask]
    p_c = p_c[mask]
    return {
        "MAE": mean_absolute_error(y_c, p_c),
        "RMSE": np.sqrt(mean_squared_error(y_c, p_c)),
        "MAPE": mape(y_c, p_c),
        "sMAPE": smape(y_c, p_c),
    }


def _hora_a_0_23(hora):
    hora = pd.to_numeric(hora, errors="coerce")
    if hora.dropna().empty:
        return hora
    if hora.min() >= 1 and hora.max() <= 24:
        return hora.astype(float) - 1
    return hora.astype(float)


def _normalizar_exogena(df, nombre_variable):
    aux = df.copy()
    aux.columns = aux.columns.astype(str).str.strip()
    cols = {c.lower(): c for c in aux.columns}
    for requerida in ["fecha", "hora", "valor"]:
        if requerida not in cols:
            raise ValueError(f"No existe columna {requerida} en {nombre_variable}")

    aux = aux[[cols["fecha"], cols["hora"], cols["valor"]]].copy()
    aux[cols["fecha"]] = pd.to_datetime(aux[cols["fecha"]], errors="coerce")
    aux["hora_0_23"] = _hora_a_0_23(aux[cols["hora"]])
    aux[nombre_variable] = pd.to_numeric(aux[cols["valor"]], errors="coerce")
    aux = aux.dropna(subset=[cols["fecha"], "hora_0_23", nombre_variable])
    aux["hora_0_23"] = aux["hora_0_23"].astype(int)
    if not aux["hora_0_23"].between(0, 23).all():
        raise ValueError(f"La columna hora de {nombre_variable} debe estar en 0-23 o 1-24")

    aux["datetime"] = aux[cols["fecha"]] + pd.to_timedelta(aux["hora_0_23"], unit="h")
    return (
        aux[["datetime", nombre_variable]]
        .groupby("datetime", as_index=False)
        .mean()
        .sort_values("datetime")
    )


def merge_exogenas():
    exog_df = None
    for nombre_variable, nombre_df in EXOG_SOURCE_MAP.items():
        if nombre_df not in globals():
            raise ValueError(
                f"No existe el DataFrame global {nombre_df}. "
                "Debe estar construido antes de ejecutar_pipeline()."
            )
        aux = _normalizar_exogena(globals()[nombre_df], nombre_variable)
        exog_df = aux if exog_df is None else exog_df.merge(aux, on="datetime", how="outer")

    exog_df = exog_df.sort_values("datetime").reset_index(drop=True)
    return exog_df[["datetime"] + EXOG_COLS]


def alinear_exogenas_a_fechas(fechas, exogenas_df):
    if exogenas_df is None:
        raise ValueError("exogenas_df no puede ser None")

    base = pd.DataFrame({"datetime": pd.to_datetime(fechas, errors="coerce")})
    aux = exogenas_df.copy()
    aux["datetime"] = pd.to_datetime(aux["datetime"], errors="coerce")
    aux = aux.dropna(subset=["datetime"]).sort_values("datetime")
    out = base.merge(aux[["datetime"] + EXOG_COLS], on="datetime", how="left")

    # Solo las exogenas se adaptan a las fechas de la serie objetivo.
    # La variable objetivo nunca se rellena, interpola ni modifica aqui.
    out[EXOG_COLS] = out[EXOG_COLS].ffill().bfill()

    if out[EXOG_COLS].isna().any().any():
        raise ValueError("No hay exogenas suficientes para las fechas de la serie")
    return out[EXOG_COLS].astype(float).reset_index(drop=True)


def validar_horizonte(modelo, pred):
    if len(pred) != FORECAST_HORIZON:
        raise ValueError(f"{modelo}: prediccion con {len(pred)} horas; se esperaban {FORECAST_HORIZON}")


# =========================================================
# LECTURA
# =========================================================

def cargar_regiones():
    regiones = {}
    for region, archivo in ARCHIVOS_REGIONES.items():
        if not os.path.exists(archivo):
            print(f"No encontre {archivo}, salto {region}")
            continue
        df = pd.read_csv(archivo)
        df.columns = df.columns.astype(str).str.strip()
        regiones[region] = df
        print(f"OK {region}: {archivo} cargado con shape {df.shape}")
    return regiones


def extraer_serie_horaria(df, columna, nombre_serie):
    if columna not in df.columns:
        raise ValueError(f"No existe columna {columna} en {nombre_serie}. Columnas: {list(df.columns)}")
    if COL_FECHA not in df.columns:
        raise ValueError(f"No existe columna '{COL_FECHA}' en {nombre_serie}")
    if COL_HORA not in df.columns:
        raise ValueError(f"No existe columna '{COL_HORA}' en {nombre_serie}")

    aux = df[[COL_FECHA, COL_HORA, columna]].copy()
    aux[COL_FECHA] = pd.to_datetime(aux[COL_FECHA], errors="coerce")
    aux[COL_HORA] = pd.to_numeric(aux[COL_HORA], errors="coerce")
    aux[columna] = pd.to_numeric(aux[columna], errors="coerce")
    aux = aux.dropna(subset=[COL_FECHA, COL_HORA])
    aux["hora_0_23"] = aux[COL_HORA].astype(int) - 1
    aux["datetime"] = aux[COL_FECHA] + pd.to_timedelta(aux["hora_0_23"], unit="h")
    aux = aux.sort_values("datetime")
    return aux[columna].values.astype(float), aux["datetime"].values


# =========================================================
# SPLIT TEMPORAL
# =========================================================

def temporal_validation_split(train_y, n_splits=3, gap=168):
    splits = []
    total_len = len(train_y)
    val_size = max(24 * 7, total_len // (n_splits + 2))

    for i in range(n_splits):
        val_end = total_len - (i * val_size) - gap
        val_start = max(val_end - val_size, gap)
        train_end = val_start - gap
        if train_end < MIN_OBS or val_end - val_start < 24:
            continue
        splits.append({
            "train_y": train_y[:train_end],
            "val_y": train_y[val_start:val_end],
            "name": f"split_{i}"
        })
    return splits


# =========================================================
# LSTM MULTIVARIADA
# =========================================================

def crear_secuencias(y, window):
    arr = np.asarray(y, dtype=float)
    Xs = []
    ys = []
    for i in range(window, len(arr)):
        Xs.append(arr[i-window:i])
        ys.append(arr[i] if arr.ndim == 1 else arr[i, 0])
    return np.array(Xs), np.array(ys)


def _armar_matriz_lstm(y, exog):
    exog_df = pd.DataFrame(exog).reset_index(drop=True)
    if len(exog_df) != len(y):
        raise ValueError("La longitud de exog debe coincidir con y para LSTM")
    for col in EXOG_COLS:
        if col not in exog_df.columns:
            raise ValueError(f"Falta variable exogena {col} para LSTM")
    return np.column_stack([np.asarray(y, dtype=float), exog_df[EXOG_COLS].astype(float).values])


def _escalar_exog_lstm(exog, scaler):
    exog_df = pd.DataFrame(exog).reset_index(drop=True)
    return (exog_df[EXOG_COLS].astype(float).values - scaler.mean_[1:]) / scaler.scale_[1:]


def _predecir_lstm_autoregresivo(model, train_scaled, future_exog_scaled, window, scaler):
    preds = []
    hist_rows = [row.copy() for row in train_scaled]
    for step in range(len(future_exog_scaled)):
        seq = np.array(hist_rows[-window:]).reshape(1, window, train_scaled.shape[1])
        pred_s = model.predict(seq, verbose=0)[0][0]
        hist_rows.append(np.concatenate([np.array([pred_s]), future_exog_scaled[step]]))
        preds.append(pred_s)
    return (np.array(preds) * scaler.scale_[0]) + scaler.mean_[0]


def objective_lstm(trial, train_y, val_y, train_exog=None, val_exog=None):
    params = {
        "units": trial.suggest_categorical("units", [16, 32]),
        "dropout": trial.suggest_categorical("dropout", [0.1, 0.2]),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "window": trial.suggest_categorical("window", [24, 48, 168]),
        "learning_rate": trial.suggest_categorical("learning_rate", [0.001, 0.003]),
    }
    window = params["window"]
    if len(train_y) < window + 200:
        return float("inf")

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(_armar_matriz_lstm(train_y, train_exog))
    X_seq, y_seq = crear_secuencias(train_scaled, window)
    if len(X_seq) < 200:
        return float("inf")

    split = int(len(X_seq) * 0.8)
    X_train, X_val = X_seq[:split], X_seq[split:]
    y_train, y_val = y_seq[:split], y_seq[split:]

    model = Sequential([
        Input(shape=(window, train_scaled.shape[1])),
        LSTM(params["units"], dropout=params["dropout"]),
        Dense(1),
    ])
    model.compile(loss="mse", optimizer=tf.keras.optimizers.Adam(learning_rate=params["learning_rate"]))
    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE_LSTM, restore_best_weights=True)
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_LSTM,
        batch_size=params["batch_size"],
        callbacks=[early_stop],
        verbose=0,
    )

    val_exog_scaled = _escalar_exog_lstm(val_exog, scaler)
    preds_original = _predecir_lstm_autoregresivo(model, train_scaled, val_exog_scaled, window, scaler)
    score = smape(val_y, preds_original)
    cleanup()
    return score if not np.isnan(score) else float("inf")


def tune_lstm(train_y, val_y, nombre_serie, train_exog=None, val_exog=None):
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42),
        study_name=f"{nombre_serie}_lstm_fast_1y",
        storage=f"sqlite:///{OPTUNA_DB}",
        load_if_exists=True,
    )
    study.optimize(
        lambda trial: objective_lstm(trial, train_y, val_y, train_exog, val_exog),
        n_trials=N_TRIALS_LSTM,
        show_progress_bar=False,
    )
    return study.best_params, study.trials_dataframe()


def forecast_lstm_tuned(train_y, horizon, best_params, train_exog=None, future_exog=None):
    try:
        if future_exog is None or len(future_exog) != horizon:
            raise ValueError("future_exog debe tener exactamente horizon filas")
        window = best_params.get("window", 24)
        scaler = StandardScaler()
        train_scaled = scaler.fit_transform(_armar_matriz_lstm(train_y, train_exog))
        X_seq, y_seq = crear_secuencias(train_scaled, window)
        if len(X_seq) < 200:
            return np.full(horizon, np.nan)

        split = int(len(X_seq) * 0.8)
        X_train, X_val = X_seq[:split], X_seq[split:]
        y_train, y_val = y_seq[:split], y_seq[split:]

        model = Sequential([
            Input(shape=(window, train_scaled.shape[1])),
            LSTM(best_params["units"], dropout=best_params["dropout"]),
            Dense(1),
        ])
        model.compile(loss="mse", optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["learning_rate"]))
        early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE_LSTM, restore_best_weights=True)
        model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS_LSTM,
            batch_size=best_params["batch_size"],
            callbacks=[early_stop],
            verbose=0,
        )

        future_exog_scaled = _escalar_exog_lstm(future_exog, scaler)
        return _predecir_lstm_autoregresivo(model, train_scaled, future_exog_scaled, window, scaler)
    except Exception as e:
        print(f"         Error forecast LSTM: {str(e)[:80]}")
        return np.full(horizon, np.nan)
    finally:
        cleanup()


# =========================================================
# RESULTADOS
# =========================================================

RESULTS_SERIES = []
RESULTS_METRICS = []
RESULTS_TRIALS = []
RESULTS_CONFIG_USADA = []


def guardar_predicciones(nombre_serie, fechas_test, pred, modelo):
    for j, pred_val in enumerate(pred):
        if j < len(fechas_test):
            RESULTS_SERIES.append({
                "serie": nombre_serie,
                "fecha": fechas_test[j],
                "tipo": "prediccion",
                "subset": "test",
                "modelo": modelo,
                "valor": pred_val,
            })


def guardar_metricas(nombre_serie, modelo, tuneado, metricas, horizonte_usado):
    RESULTS_METRICS.append({
        "serie": nombre_serie,
        "modelo": modelo,
        "tuneado": tuneado,
        "horizonte_usado": horizonte_usado,
        "MAPE": metricas["MAPE"],
        "sMAPE": metricas["sMAPE"],
        "MAE": metricas["MAE"],
        "RMSE": metricas["RMSE"],
    })


def guardar_todos_csv():
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

    rutas = {
        "series": os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}_series.csv"
        ),
        "metricas": os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}_metricas.csv"
        ),
        "trials": os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}_optuna_trials.csv"
        ),
        "config": os.path.join(
            DRIVE_OUTPUT_DIR,
            f"{SAVE_PREFIX}_config_usada.csv"
        ),
    }

    # Series
    if RESULTS_SERIES:
        df_series = pd.DataFrame(RESULTS_SERIES)
        df_series["fecha"] = pd.to_datetime(
            df_series["fecha"],
            errors="coerce"
        )
        df_series = df_series.sort_values(
            ["serie", "fecha", "modelo"]
        )
    else:
        df_series = pd.DataFrame(columns=[
            "serie",
            "fecha",
            "tipo",
            "subset",
            "modelo",
            "valor",
        ])

    df_series.to_csv(
        rutas["series"],
        index=False,
        encoding="utf-8-sig"
    )

    # Métricas
    columnas_metricas = [
        "serie",
        "modelo",
        "tuneado",
        "horizonte_usado",
        "MAPE",
        "sMAPE",
        "MAE",
        "RMSE",
    ]

    if RESULTS_METRICS:
        df_metrics = pd.DataFrame(RESULTS_METRICS)
        df_metrics = df_metrics.sort_values(
            ["serie", "MAPE"]
        )
    else:
        df_metrics = pd.DataFrame(
            columns=columnas_metricas
        )

    df_metrics.to_csv(
        rutas["metricas"],
        index=False,
        encoding="utf-8-sig"
    )

    # Trials
    if RESULTS_TRIALS:
        df_trials = pd.concat(
            RESULTS_TRIALS,
            ignore_index=True
        )
    else:
        df_trials = pd.DataFrame()

    df_trials.to_csv(
        rutas["trials"],
        index=False,
        encoding="utf-8-sig"
    )

    # Configuración
    if RESULTS_CONFIG_USADA:
        df_config = pd.DataFrame(
            RESULTS_CONFIG_USADA
        )
    else:
        df_config = pd.DataFrame(columns=[
            "serie",
            "modelo",
            "parametros",
            "horizonte_usado",
        ])

    df_config.to_csv(
        rutas["config"],
        index=False,
        encoding="utf-8-sig"
    )

    print("\nArchivos guardados:")
    print(f"   Series:  {len(df_series):,} -> {rutas['series']}")
    print(f"   Métricas: {len(df_metrics):,} -> {rutas['metricas']}")
    print(f"   Trials:   {len(df_trials):,} -> {rutas['trials']}")
    print(f"   Config:   {len(df_config):,} -> {rutas['config']}")

    if df_metrics.empty:
        print(
            "\nADVERTENCIA: el archivo de métricas está vacío. "
            "Ninguna serie produjo métricas válidas."
        )

    return df_metrics


def imprimir_resultados(df_metricas):
    if df_metricas is None or len(df_metricas) == 0:
        print("\nNo hay resultados para mostrar")
        return

    print("\n" + "=" * 100)
    print("MEJOR MODELO POR SERIE")
    print("=" * 100)
    mejor_por_serie = df_metricas.loc[df_metricas.groupby("serie")["MAPE"].idxmin()]
    print(
        mejor_por_serie[
            ["serie", "modelo", "horizonte_usado", "MAPE", "sMAPE", "MAE", "RMSE"]
        ].to_string(index=False)
    )

    print("\n" + "=" * 100)
    print("RANKING GLOBAL DE MODELOS")
    print("=" * 100)
    ranking = (
        df_metricas
        .groupby("modelo")["MAPE"]
        .agg(["mean", "std"])
        .round(2)
        .sort_values("mean")
    )
    print(ranking.to_string())


def _preparar_serie(nombre_serie, serie, fechas, exogenas_df):
    required_hours = LSTM_TRAIN_HOURS + FORECAST_HORIZON

    if len(serie) < required_hours:
        print(f"   Serie insuficiente para {LSTM_TRAIN_HOURS}h train + {FORECAST_HORIZON}h test: {nombre_serie}")
        return None

    exog_serie = alinear_exogenas_a_fechas(fechas, exogenas_df)
    na_objetivo = int(pd.Series(serie).isna().sum())
    na_exogenas = exog_serie[EXOG_COLS].isna().sum()

    print("\n   Verificaciones antes del entrenamiento:")
    print(f"      Longitud serie objetivo: {len(serie):,}")
    print(f"      Longitud exogenas alineadas: {len(exog_serie):,}")
    print("      Valores faltantes por exogena:")
    print(na_exogenas.to_string())
    print(f"      Valores faltantes serie objetivo: {na_objetivo:,}")
    print("      Confirmacion objetivo: NO fue rellenada ni interpolada")
    print("      Confirmacion exogenas: alineadas por datetime con ffill()/bfill() solo en exogenas")

    for j, fecha in enumerate(fechas):
        RESULTS_SERIES.append({
            "serie": nombre_serie,
            "fecha": fecha,
            "tipo": "real",
            "subset": "completo",
            "modelo": "real",
            "valor": serie[j],
        })

    test_size = FORECAST_HORIZON
    train_start = len(serie) - test_size - LSTM_TRAIN_HOURS
    train_end = len(serie) - test_size

    train = serie[train_start:train_end]
    test = serie[-test_size:]
    train_exog = exog_serie.iloc[train_start:train_end].reset_index(drop=True)
    test_exog = exog_serie.iloc[-test_size:].reset_index(drop=True)
    fechas_test = fechas[-test_size:]
    horizon = FORECAST_HORIZON
    horizonte_usado = f"{FORECAST_HORIZON}_horas"

    print("\n   Split general fijo")
    print(f"      Train: {len(train)} obs")
    print(f"      Test:  {len(test)} obs")
    print(f"      Horizonte: {horizon} horas")
    print(f"      LSTM usa solo las {LSTM_TRAIN_HOURS} horas previas al test")

    splits = temporal_validation_split(train, n_splits=VAL_SPLITS, gap=VAL_GAP)
    fallback_split = False

    if len(splits) == 0:
        fallback_split = True
        val_size = max(24 * 30, len(train) // 5)
        splits = [{
            "train_y": train[:-val_size],
            "val_y": train[-val_size:],
            "name": "split_0"
        }]

    tune_split = splits[0]
    window_tune = WINDOW_DEFAULT
    train_end_tune = len(tune_split["train_y"])

    if fallback_split:
        val_start_tune = train_end_tune
    else:
        val_start_tune = train_end_tune + VAL_GAP

    val_end_tune = val_start_tune + len(tune_split["val_y"])
    train_exog_tune = train_exog.iloc[:train_end_tune].reset_index(drop=True)
    val_exog_tune = train_exog.iloc[val_start_tune:val_end_tune].reset_index(drop=True)

    return {
        "train": train,
        "test": test,
        "train_exog": train_exog,
        "test_exog": test_exog,
        "fechas_test": fechas_test,
        "horizon": horizon,
        "horizonte_usado": horizonte_usado,
        "tune_split": tune_split,
        "window_tune": window_tune,
        "train_exog_tune": train_exog_tune,
        "val_exog_tune": val_exog_tune,
    }

def evaluar_serie(nombre_serie, serie, fechas, exogenas_df=None):
    contexto = _preparar_serie(
        nombre_serie,
        serie,
        fechas,
        exogenas_df
    )

    if contexto is None:
        return

    try:
        print("      LSTM multivariada tuning...")
        print(f"         LSTM train: {len(contexto['train'])} obs")
        print(f"         LSTM test:  {len(contexto['test'])} obs")

        tune_split = contexto["tune_split"]

        best_params, trials_df = tune_lstm(
            tune_split["train_y"],
            tune_split["val_y"],
            nombre_serie,
            contexto["train_exog_tune"],
            contexto["val_exog_tune"]
        )

        trials_df["serie"] = nombre_serie
        trials_df["modelo"] = "LSTM"
        RESULTS_TRIALS.append(trials_df)

        RESULTS_CONFIG_USADA.append({
            "serie": nombre_serie,
            "modelo": "LSTM",
            "parametros": str(best_params),
            "horizonte_usado": contexto["horizonte_usado"],
        })

        pred = forecast_lstm_tuned(
            contexto["train"],
            contexto["horizon"],
            best_params,
            contexto["train_exog"],
            contexto["test_exog"]
        )

        validar_horizonte(
            "LSTM_Tuned_1Y",
            pred
        )

        n_pred_validas = int(
            np.isfinite(pred).sum()
        )

        n_test_validas = int(
            np.isfinite(contexto["test"]).sum()
        )

        print(
            f"      Predicciones válidas: "
            f"{n_pred_validas}/{len(pred)}"
        )

        print(
            f"      Valores test válidos: "
            f"{n_test_validas}/{len(contexto['test'])}"
        )

        metricas = calcular_metricas(
            contexto["test"],
            pred
        )

        if metricas is None:
            print(
                "      ADVERTENCIA: no fue posible calcular "
                "métricas; no existen pares "
                "real-predicción válidos."
            )
            return

        guardar_metricas(
            nombre_serie,
            "LSTM_Tuned_1Y",
            True,
            metricas,
            contexto["horizonte_usado"]
        )

        guardar_predicciones(
            nombre_serie,
            contexto["fechas_test"],
            pred,
            "LSTM_Tuned_1Y"
        )

        print(
            f"      LSTM_Tuned_1Y: "
            f"MAPE={metricas['MAPE']:.2f}%"
        )

    except Exception as e:
        print(
            f"      Error evaluando {nombre_serie}: "
            f"{type(e).__name__}: {e}"
        )

# =========================================================
# PIPELINE PRINCIPAL
# =========================================================

def ejecutar_pipeline():
    global RESULTS_SERIES, RESULTS_METRICS, RESULTS_TRIALS, RESULTS_CONFIG_USADA

    RESULTS_SERIES = []
    RESULTS_METRICS = []
    RESULTS_TRIALS = []
    RESULTS_CONFIG_USADA = []

    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

    print("=" * 80)
    print(PIPELINE_NAME)
    print("=" * 80)
    print(f"Directorio de salida: {DRIVE_OUTPUT_DIR}")

    exogenas_df = merge_exogenas()
    print(f"Exogenas cargadas: {len(exogenas_df):,} filas")

    regiones = cargar_regiones()
    for region, df in regiones.items():
        nombre_serie = f"{region}_DEMANDA"
        columna = COL_DEMANDA

        print(f"\n{'=' * 80}")
        print(f"Serie: {nombre_serie}")
        print(f"{'=' * 80}")

        serie, fechas = extraer_serie_horaria(df, columna, nombre_serie)
        print("\n   Serie horaria:")
        print(f"      Longitud: {len(serie):,} observaciones")
        print(f"      Rango: {fechas[0]} a {fechas[-1]}")
        print(f"      Minimo: {np.nanmin(serie):.2f}")
        print(f"      Maximo: {np.nanmax(serie):.2f}")
        print(f"      Media: {np.nanmean(serie):.2f}")

        evaluar_serie(nombre_serie, serie, fechas, exogenas_df)

        print("\n   Guardando avance acumulado en Drive...")
        guardar_todos_csv()

    df_metricas = guardar_todos_csv()
    imprimir_resultados(df_metricas)

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETADO")
    print(os.path.join(DRIVE_OUTPUT_DIR, f"{SAVE_PREFIX}_series.csv"))
    print(os.path.join(DRIVE_OUTPUT_DIR, f"{SAVE_PREFIX}_metricas.csv"))
    print(os.path.join(DRIVE_OUTPUT_DIR, f"{SAVE_PREFIX}_optuna_trials.csv"))
    print(os.path.join(DRIVE_OUTPUT_DIR, f"{SAVE_PREFIX}_config_usada.csv"))
    print("=" * 80)
    print("\nGuardando resultados acumulados...")

    guardar_todos_csv()


# =========================================================
# EJECUTAR
# =========================================================

ejecutar_pipeline()


Mounted at /content/drive
PIPELINE LSTM DEMANDA
Directorio de salida: /content/drive/MyDrive/Pipeline_Resultados
Exogenas cargadas: 63,528 filas
OK BCA: BCA_long.csv cargado con shape (64799, 11)
OK CEN: CEN_long.csv cargado con shape (64800, 11)
OK NES: NES_long.csv cargado con shape (64800, 11)
OK NOR: NOR_long.csv cargado con shape (64800, 11)
OK NTE: NTE_long.csv cargado con shape (64800, 11)
OK OCC: OCC_long.csv cargado con shape (64800, 11)
OK ORI: ORI_long.csv cargado con shape (64800, 11)
OK PEN: PEN_long.csv cargado con shape (64800, 11)

Serie: BCA_DEMANDA

   Serie horaria:
      Longitud: 64,799 observaciones
      Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000
      Minimo: 821.54
      Maximo: 3572.45
      Media: 1789.23

   Verificaciones antes del entrenamiento:
      Longitud serie objetivo: 64,799
      Longitud exogenas alineadas: 64,799
      Valores faltantes por exogena:
Temperatura    0
IGAE           0
      Valores faltantes serie objetiv

[I 2026-07-28 17:03:39,897] A new study created in RDB with name: BCA_DEMANDA_lstm_fast_1y
[I 2026-07-28 17:04:51,241] Trial 0 finished with value: 14.4985206207073 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 14.4985206207073.
[I 2026-07-28 17:05:52,038] Trial 1 finished with value: 12.913132883877537 and parameters: {'units': 16, 'dropout': 0.1, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 12.913132883877537.
[I 2026-07-28 17:06:54,924] Trial 2 finished with value: 13.40438323437917 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 128, 'window': 168, 'learning_rate': 0.001}. Best is trial 1 with value: 12.913132883877537.
[I 2026-07-28 17:07:59,442] Trial 3 finished with value: 13.502371392721175 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 128, 'window': 168, 'learning_rate': 0.003}. Best is trial 1 with value: 12.913132883877537.
[

      Predicciones válidas: 168/168
      Valores test válidos: 168/168
      LSTM_Tuned_1Y: MAPE=14.15%

   Guardando avance acumulado en Drive...


[I 2026-07-28 17:14:34,228] A new study created in RDB with name: CEN_DEMANDA_lstm_fast_1y



Archivos guardados:
   Series:  64,967 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_series.csv
   Métricas: 1 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_metricas.csv
   Trials:   10 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_optuna_trials.csv
   Config:   1 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_config_usada.csv

Serie: CEN_DEMANDA

   Serie horaria:
      Longitud: 64,800 observaciones
      Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000
      Minimo: 1431.15
      Maximo: 8918.56
      Media: 6231.83

   Verificaciones antes del entrenamiento:
      Longitud serie objetivo: 64,800
      Longitud exogenas alineadas: 64,800
      Valores faltantes por exogena:
Temperatura    0
IGAE           0
      Valores faltantes serie objetivo: 0
      Confirmacion objetivo: NO fue rellenada ni interpolada
      Confirmacion exogenas: alineadas por datetime con ffill()/bfill() s

...

      Predicciones válidas: 168/168
      Valores test válidos: 168/168
      LSTM_Tuned_1Y: MAPE=9.33%

   Guardando avance acumulado en Drive...

Archivos guardados:
   Series:  389,807 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_series.csv
   Métricas: 6 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_metricas.csv
   Trials:   60 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_optuna_trials.csv
   Config:   6 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_config_usada.csv

Serie: ORI_DEMANDA

   Serie horaria:
      Longitud: 64,800 observaciones
      Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000
      Minimo: 3368.58
      Maximo: 11692.71
      Media: 6337.85

   Verificaciones antes del entrenamiento:
      Longitud serie objetivo: 64,800
      Longitud exogenas alineadas: 64,800
      Valores faltantes por exogena:
Temperatura    0
IGAE           0
      Valores faltantes ser

[I 2026-07-28 18:11:53,140] A new study created in RDB with name: ORI_DEMANDA_lstm_fast_1y
[I 2026-07-28 18:13:02,523] Trial 0 finished with value: 12.842658855507366 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 12.842658855507366.
[I 2026-07-28 18:14:10,362] Trial 1 finished with value: 12.602354069079135 and parameters: {'units': 16, 'dropout': 0.1, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 12.602354069079135.
[I 2026-07-28 18:15:18,510] Trial 2 finished with value: 10.859806197016121 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 128, 'window': 168, 'learning_rate': 0.001}. Best is trial 2 with value: 10.859806197016121.
[I 2026-07-28 18:16:26,997] Trial 3 finished with value: 12.537670154889533 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 128, 'window': 168, 'learning_rate': 0.003}. Best is trial 2 with value: 10.8598061970161

      Predicciones válidas: 168/168
      Valores test válidos: 168/168
      LSTM_Tuned_1Y: MAPE=17.63%

   Guardando avance acumulado en Drive...

Archivos guardados:
   Series:  454,775 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_series.csv
   Métricas: 7 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_metricas.csv
   Trials:   70 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_optuna_trials.csv
   Config:   7 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_config_usada.csv

Serie: PEN_DEMANDA

   Serie horaria:
      Longitud: 64,800 observaciones
      Rango: 2019-01-01T00:00:00.000000000 a 2026-05-23T23:00:00.000000000
      Minimo: 62.74
      Maximo: 2974.17
      Media: 1586.82

   Verificaciones antes del entrenamiento:
      Longitud serie objetivo: 64,800
      Longitud exogenas alineadas: 64,800
      Valores faltantes por exogena:
Temperatura    0
IGAE           0
      Valores faltantes serie

[I 2026-07-28 18:23:42,442] A new study created in RDB with name: PEN_DEMANDA_lstm_fast_1y
[I 2026-07-28 18:24:49,213] Trial 0 finished with value: 13.666340676488359 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 13.666340676488359.
[I 2026-07-28 18:25:54,871] Trial 1 finished with value: 20.76349139042892 and parameters: {'units': 16, 'dropout': 0.1, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 0 with value: 13.666340676488359.
[I 2026-07-28 18:27:00,089] Trial 2 finished with value: 20.468338537853235 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 128, 'window': 168, 'learning_rate': 0.001}. Best is trial 0 with value: 13.666340676488359.
[I 2026-07-28 18:28:07,314] Trial 3 finished with value: 13.64243049266446 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 128, 'window': 168, 'learning_rate': 0.003}. Best is trial 3 with value: 13.64243049266446.

      Predicciones válidas: 168/168
      Valores test válidos: 168/168
      LSTM_Tuned_1Y: MAPE=20.25%

   Guardando avance acumulado en Drive...

Archivos guardados:
   Series:  519,743 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_series.csv
   Métricas: 8 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_metricas.csv
   Trials:   80 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_optuna_trials.csv
   Config:   8 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_config_usada.csv

Archivos guardados:
   Series:  519,743 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_series.csv
   Métricas: 8 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_metricas.csv
   Trials:   80 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_optuna_trials.csv
   Config:   8 -> /content/drive/MyDrive/Pipeline_Resultados/lstm_demanda_horario_config_usada.csv

MEJOR MODELO POR SERIE


#LSTM pero con 60 épocas y más Optuna

In [32]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 27.5 MB/s eta 0:00:00


In [33]:
# =========================================================
# PIPELINE LSTM DIRECTA MULTI-HORIZONTE
#
# OBJETIVO:
#   Historia -> 168 horas de demanda de una sola vez
#
# TRAIN:
#   3 meses = 2160 horas
#
# FORECAST:
#   168 horas = 1 semana
#
# EXOGENAS:
#   - Temperatura
#   - IGAE
#   - Generacion
#   - Importacion
#   - Exportacion
#
# Durante el horizonte:
#
#   Temperatura / IGAE:
#       valor correspondiente a cada hora futura
#
#   Generacion / Importacion / Exportacion:
#       valor de la misma hora de la semana anterior
#       (lag 168)
#
# Así NO usamos GEN / IMP / EXP reales del test.
#
# Puedes comentar cualquier exogena en EXOG_COLS.
# =========================================================


# =========================================================
# IMPORTS
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import gc
import numpy as np
import pandas as pd
import optuna

from optuna.samplers import TPESampler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from sklearn.preprocessing import StandardScaler

import tensorflow as tf

from tensorflow.keras import backend as K

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Flatten,
    Concatenate
)

from tensorflow.keras.callbacks import EarlyStopping

from google.colab import drive


# =========================================================
# DRIVE
# =========================================================

drive.mount("/content/drive")

DRIVE_OUTPUT_DIR = (
    "/content/drive/MyDrive/Pipeline_Resultados"
)

os.makedirs(
    DRIVE_OUTPUT_DIR,
    exist_ok=True
)


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = "/content"

MIN_OBS = 24 * 30

FORECAST_HORIZON = 24 * 7      # 168 h

# 3 meses
LSTM_TRAIN_HOURS = 24 * 30 * 3 # 2160 h

# Si quieres regresar a 3600:
# LSTM_TRAIN_HOURS = 3600


# Ventanas históricas que probará Optuna
WINDOW_OPTIONS = [
    24,
    48,
    168
]


# =========================================================
# REGIONES
# =========================================================

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}


# =========================================================
# COLUMNAS OBJETIVO
# =========================================================

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = (
    "Estimacion de Demanda por Balance (MWh)"
)


# =========================================================
# EXOGENAS
#
# COMENTA CUALQUIER LINEA PARA QUITARLA.
# POR DEFECTO ENTRAN TODAS.
# =========================================================

EXOG_COLS = [
    "Temperatura",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]


# =========================================================
# EXOGENAS CONOCIDAS EN EL HORIZONTE
# =========================================================

EXOG_CONOCIDAS_FUTURO = [
    "Temperatura",
    "IGAE",
]


# =========================================================
# EXOGENAS ELECTRICAS
#
# Para el horizonte se usan con lag 168:
#
# GEN_t futuro <- GEN_(t-168)
# IMP_t futuro <- IMP_(t-168)
# EXP_t futuro <- EXP_(t-168)
# =========================================================

EXOG_LAG_SEMANAL = [
    "Generacion",
    "Importacion",
    "Exportacion",
]

LAG_EXOG_FUTURO = 168


# =========================================================
# DATAFRAMES GLOBALES
# =========================================================

EXOG_SOURCE_MAP = {
    "Temperatura": "Temperaturas_H",
    "IGAE": "IGAE_H",
}


# =========================================================
# OPTUNA / LSTM
# =========================================================

N_TRIALS_LSTM = 10

EPOCHS_LSTM = 60

PATIENCE_LSTM = 8


OPTUNA_DB = os.path.join(
    DRIVE_OUTPUT_DIR,
    "optuna_lstm_directa_168.db"
)


SAVE_PREFIX = (
    "lstm_directa_168_demanda_horario"
)

PIPELINE_NAME = (
    "PIPELINE LSTM DIRECTA 168H"
)


# =========================================================
# UTILIDADES
# =========================================================

def cleanup():

    K.clear_session()

    gc.collect()


def mape(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                / y_true[mask]
            )
        )
        * 100
    )


def smape(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (denominator != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                y_true[mask]
                - y_pred[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def calcular_metricas(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    n = min(
        len(y_true),
        len(y_pred)
    )

    y_true = y_true[:n]
    y_pred = y_pred[:n]

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    if mask.sum() == 0:
        return None

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return {

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAPE":
            mape(
                y_true,
                y_pred
            ),

        "sMAPE":
            smape(
                y_true,
                y_pred
            )
    }


# =========================================================
# HORAS
# =========================================================

def _hora_a_0_23(hora):

    hora = pd.to_numeric(
        hora,
        errors="coerce"
    )

    if hora.dropna().empty:
        return hora

    if (
        hora.min() >= 1
        and hora.max() <= 24
    ):

        return (
            hora.astype(float)
            - 1
        )

    return hora.astype(float)


# =========================================================
# NORMALIZAR EXOGENA
# =========================================================

def _normalizar_exogena(
    df,
    nombre_variable
):

    aux = df.copy()

    aux.columns = (
        aux.columns
        .astype(str)
        .str.strip()
    )

    cols = {
        c.lower(): c
        for c in aux.columns
    }

    for requerida in [
        "fecha",
        "hora",
        "valor"
    ]:

        if requerida not in cols:

            raise ValueError(
                f"No existe columna "
                f"{requerida} "
                f"en {nombre_variable}"
            )

    aux = aux[
        [
            cols["fecha"],
            cols["hora"],
            cols["valor"]
        ]
    ].copy()

    aux[
        cols["fecha"]
    ] = pd.to_datetime(
        aux[
            cols["fecha"]
        ],
        errors="coerce"
    )

    aux[
        "hora_0_23"
    ] = _hora_a_0_23(
        aux[
            cols["hora"]
        ]
    )

    aux[
        nombre_variable
    ] = pd.to_numeric(
        aux[
            cols["valor"]
        ],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            cols["fecha"],
            "hora_0_23",
            nombre_variable
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            "hora_0_23"
        ].astype(int)
    )

    aux[
        "datetime"
    ] = (
        aux[
            cols["fecha"]
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    return (
        aux[
            [
                "datetime",
                nombre_variable
            ]
        ]
        .groupby(
            "datetime",
            as_index=False
        )
        .mean()
        .sort_values(
            "datetime"
        )
    )


# =========================================================
# EXOGENAS DE CADA REGION
# =========================================================

def merge_exogenas(
    region
):

    dfs = []

    # -----------------------------------------------------
    # TEMPERATURA / IGAE
    # -----------------------------------------------------

    for (
        nombre_variable,
        nombre_df
    ) in EXOG_SOURCE_MAP.items():

        if (
            nombre_variable
            not in EXOG_COLS
        ):
            continue

        if (
            nombre_df
            not in globals()
        ):

            raise ValueError(
                f"No existe "
                f"{nombre_df}"
            )

        aux = _normalizar_exogena(
            globals()[
                nombre_df
            ],
            nombre_variable
        )

        dfs.append(
            aux
        )

    # -----------------------------------------------------
    # GEN / IMP / EXP REGIONALES
    # -----------------------------------------------------

    REGION_EXOG_MAP = {

        "Generacion":
            f"{region}_GEN",

        "Importacion":
            f"{region}_IMP",

        "Exportacion":
            f"{region}_EXP",
    }

    for (
        variable,
        nombre_serie
    ) in REGION_EXOG_MAP.items():

        if (
            variable
            not in EXOG_COLS
        ):
            continue

        if (
            "series"
            not in globals()
        ):

            raise ValueError(
                "No existe el "
                "diccionario global "
                "'series'."
            )

        if (
            nombre_serie
            not in series
        ):

            raise ValueError(
                f"No existe "
                f"{nombre_serie} "
                "dentro de series."
            )

        aux = _normalizar_exogena(
            series[
                nombre_serie
            ],
            variable
        )

        dfs.append(
            aux
        )

    if len(dfs) == 0:

        raise ValueError(
            "No hay exogenas activas."
        )

    exog_df = dfs[0]

    for aux in dfs[1:]:

        exog_df = (
            exog_df.merge(
                aux,
                on="datetime",
                how="outer"
            )
        )

    exog_df = (
        exog_df
        .sort_values(
            "datetime"
        )
        .reset_index(
            drop=True
        )
    )

    exog_df[
        EXOG_COLS
    ] = (
        exog_df[
            EXOG_COLS
        ]
        .ffill()
        .bfill()
    )

    return exog_df[
        ["datetime"]
        + EXOG_COLS
    ]


# =========================================================
# ALINEAR EXOGENAS
# =========================================================

def alinear_exogenas_a_fechas(
    fechas,
    exogenas_df
):

    base = pd.DataFrame({

        "datetime":
            pd.to_datetime(
                fechas,
                errors="coerce"
            )
    })

    aux = (
        exogenas_df
        .copy()
    )

    aux[
        "datetime"
    ] = pd.to_datetime(
        aux[
            "datetime"
        ],
        errors="coerce"
    )

    aux = (
        aux
        .dropna(
            subset=[
                "datetime"
            ]
        )
        .sort_values(
            "datetime"
        )
    )

    out = base.merge(

        aux[
            ["datetime"]
            + EXOG_COLS
        ],

        on="datetime",

        how="left"
    )

    out[
        EXOG_COLS
    ] = (
        out[
            EXOG_COLS
        ]
        .ffill()
        .bfill()
    )

    if (
        out[
            EXOG_COLS
        ]
        .isna()
        .any()
        .any()
    ):

        raise ValueError(
            "No hay exogenas "
            "suficientes."
        )

    return (
        out[
            EXOG_COLS
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# CARGAR REGIONES
# =========================================================

def cargar_regiones():

    regiones = {}

    for (
        region,
        archivo
    ) in (
        ARCHIVOS_REGIONES.items()
    ):

        ruta = os.path.join(
            DATA_DIR,
            archivo
        )

        if not os.path.exists(
            ruta
        ):

            print(
                f"No encontre "
                f"{ruta}, "
                f"salto {region}"
            )

            continue

        df = pd.read_csv(
            ruta
        )

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
        )

        regiones[
            region
        ] = df

        print(
            f"OK {region}: "
            f"{archivo} "
            f"{df.shape}"
        )

    return regiones


# =========================================================
# EXTRAER DEMANDA
# =========================================================

def extraer_serie_horaria(
    df,
    columna,
    nombre_serie
):

    if (
        columna
        not in df.columns
    ):

        raise ValueError(
            f"No existe "
            f"{columna} "
            f"en {nombre_serie}"
        )

    aux = df[
        [
            COL_FECHA,
            COL_HORA,
            columna
        ]
    ].copy()

    aux[
        COL_FECHA
    ] = pd.to_datetime(
        aux[
            COL_FECHA
        ],
        errors="coerce"
    )

    aux[
        COL_HORA
    ] = pd.to_numeric(
        aux[
            COL_HORA
        ],
        errors="coerce"
    )

    aux[
        columna
    ] = pd.to_numeric(
        aux[
            columna
        ],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            COL_FECHA,
            COL_HORA
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            COL_HORA
        ].astype(int)
        - 1
    )

    aux[
        "datetime"
    ] = (
        aux[
            COL_FECHA
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux
        .sort_values(
            "datetime"
        )
        .drop_duplicates(
            "datetime",
            keep="last"
        )
    )

    return (

        aux[
            columna
        ].to_numpy(
            dtype=float
        ),

        aux[
            "datetime"
        ].to_numpy()
    )


# =========================================================
# BLOQUE DE EXOGENAS FUTURAS
#
# PARA CADA HORA FUTURA:
#
# Temp / IGAE:
#     valor de esa hora
#
# GEN / IMP / EXP:
#     valor de esa hora - 168 h
# =========================================================

def construir_future_exog_directa(
    exog,
    origin,
    horizon
):

    filas = []

    for h in range(
        horizon
    ):

        idx_future = (
            origin
            + h
        )

        row = {}

        for col in EXOG_COLS:

            # ---------------------------------------------
            # CONOCIDAS EN EL FUTURO
            # ---------------------------------------------

            if (
                col
                in EXOG_CONOCIDAS_FUTURO
            ):

                row[col] = (
                    exog
                    .iloc[
                        idx_future
                    ][col]
                )

            # ---------------------------------------------
            # ELECTRICAS:
            # MISMA HORA SEMANA ANTERIOR
            # ---------------------------------------------

            elif (
                col
                in EXOG_LAG_SEMANAL
            ):

                idx_lag = (
                    idx_future
                    - LAG_EXOG_FUTURO
                )

                if idx_lag < 0:

                    raise ValueError(
                        f"No existe lag "
                        f"168 para {col}"
                    )

                row[col] = (
                    exog
                    .iloc[
                        idx_lag
                    ][col]
                )

            else:

                raise ValueError(
                    f"No se definio "
                    f"tratamiento futuro "
                    f"para {col}"
                )

        filas.append(
            row
        )

    return (
        pd.DataFrame(
            filas
        )[
            EXOG_COLS
        ]
        .astype(float)
    )


# =========================================================
# ESCALADO
# =========================================================

def escalar_historia(
    y,
    exog,
    scaler_y,
    scaler_exog
):

    y_scaled = (
        scaler_y.transform(
            np.asarray(
                y
            ).reshape(
                -1,
                1
            )
        )
    )

    exog_scaled = (
        scaler_exog.transform(
            pd.DataFrame(
                exog
            )[
                EXOG_COLS
            ]
        )
    )

    return (
        np.concatenate(
            [
                y_scaled,
                exog_scaled
            ],
            axis=1
        )
    )


# =========================================================
# CREAR DATASET DIRECTO
#
# INPUT 1:
#   ventana histórica
#
# INPUT 2:
#   exogenas de las siguientes 168h
#
# TARGET:
#   demanda de las siguientes 168h
# =========================================================

def crear_dataset_directo(
    y,
    exog,
    window,
    scaler_y,
    scaler_exog
):

    y = np.asarray(
        y,
        dtype=float
    )

    exog = (
        pd.DataFrame(
            exog
        )
        .reset_index(
            drop=True
        )
    )

    hist_scaled = (
        escalar_historia(
            y,
            exog,
            scaler_y,
            scaler_exog
        )
    )

    X_hist = []
    X_future = []
    Y = []

    inicio = max(
        window,
        LAG_EXOG_FUTURO
    )

    ultimo_origin = (
        len(y)
        - FORECAST_HORIZON
    )

    for origin in range(
        inicio,
        ultimo_origin + 1
    ):

        # ---------------------------------------------
        # HISTORIA
        # ---------------------------------------------

        hist = (
            hist_scaled[
                origin-window:
                origin
            ]
        )

        # ---------------------------------------------
        # EXOGENAS DEL HORIZONTE
        # ---------------------------------------------

        future_exog = (
            construir_future_exog_directa(
                exog,
                origin,
                FORECAST_HORIZON
            )
        )

        future_exog_scaled = (
            scaler_exog.transform(
                future_exog[
                    EXOG_COLS
                ]
            )
        )

        # ---------------------------------------------
        # TARGET 168H
        # ---------------------------------------------

        target = (
            y[
                origin:
                origin
                + FORECAST_HORIZON
            ]
        )

        target_scaled = (
            scaler_y.transform(
                target.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        X_hist.append(
            hist
        )

        X_future.append(
            future_exog_scaled
        )

        Y.append(
            target_scaled
        )

    return (
        np.asarray(
            X_hist
        ),
        np.asarray(
            X_future
        ),
        np.asarray(
            Y
        )
    )


# =========================================================
# MODELO LSTM DIRECTO
# =========================================================

def construir_modelo_lstm_directo(
    window,
    n_features_hist,
    n_exog,
    units,
    dropout,
    learning_rate
):

    # -----------------------------------------------------
    # HISTORIA
    # -----------------------------------------------------

    hist_input = Input(
        shape=(
            window,
            n_features_hist
        ),
        name="historia"
    )

    hist_encoded = LSTM(
        units,
        dropout=dropout,
        name="lstm"
    )(
        hist_input
    )

    # -----------------------------------------------------
    # EXOGENAS FUTURAS
    # -----------------------------------------------------

    future_input = Input(
        shape=(
            FORECAST_HORIZON,
            n_exog
        ),
        name="exogenas_futuras"
    )

    future_flat = Flatten(
        name="flatten_exogenas"
    )(
        future_input
    )

    # -----------------------------------------------------
    # COMBINAR
    # -----------------------------------------------------

    combinado = Concatenate(
        name="fusion"
    )(
        [
            hist_encoded,
            future_flat
        ]
    )

    # -----------------------------------------------------
    # 168 SALIDAS
    # -----------------------------------------------------

    output = Dense(
        FORECAST_HORIZON,
        name="demanda_168h"
    )(
        combinado
    )

    model = Model(
        inputs=[
            hist_input,
            future_input
        ],
        outputs=output
    )

    model.compile(

        loss="mse",

        optimizer=(
            tf.keras.optimizers.Adam(
                learning_rate=(
                    learning_rate
                )
            )
        )
    )

    return model


# =========================================================
# OPTUNA
# =========================================================

def objective_lstm(
    trial,
    train_y,
    train_exog
):

    params = {

        "units":
            trial.suggest_categorical(
                "units",
                [
                    32,
                    64,
                    128
                ]
            ),

        "dropout":
            trial.suggest_categorical(
                "dropout",
                [
                    0.0,
                    0.1,
                    0.2,
                    0.3
                ]
            ),

        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [
                    32,
                    64,
                    128
                ]
            ),

        "window":
            trial.suggest_categorical(
                "window",
                WINDOW_OPTIONS
            ),

        "learning_rate":
            trial.suggest_categorical(
                "learning_rate",
                [
                    0.001,
                    0.003
                ]
            ),
    }

    window = (
        params[
            "window"
        ]
    )

    # -----------------------------------------------------
    # HOLDOUT DE 168H DENTRO DEL TRAIN
    # -----------------------------------------------------

    val_horizon = (
        FORECAST_HORIZON
    )

    core_y = (
        train_y[
            :-val_horizon
        ]
    )

    core_exog = (
        train_exog
        .iloc[
            :-val_horizon
        ]
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # SCALERS SOLO SOBRE CORE TRAIN
    # -----------------------------------------------------

    scaler_y = (
        StandardScaler()
    )

    scaler_exog = (
        StandardScaler()
    )

    scaler_y.fit(
        np.asarray(
            core_y
        ).reshape(
            -1,
            1
        )
    )

    scaler_exog.fit(
        core_exog[
            EXOG_COLS
        ]
    )

    # -----------------------------------------------------
    # DATASET DE TRAINING DIRECTO
    # -----------------------------------------------------

    (
        X_hist,
        X_future,
        Y
    ) = crear_dataset_directo(

        core_y,
        core_exog,
        window,
        scaler_y,
        scaler_exog
    )

    if len(
        X_hist
    ) < 100:

        cleanup()

        return float(
            "inf"
        )

    # -----------------------------------------------------
    # MODELO
    # -----------------------------------------------------

    model = (
        construir_modelo_lstm_directo(

            window=window,

            n_features_hist=(
                1
                + len(
                    EXOG_COLS
                )
            ),

            n_exog=len(
                EXOG_COLS
            ),

            units=(
                params[
                    "units"
                ]
            ),

            dropout=(
                params[
                    "dropout"
                ]
            ),

            learning_rate=(
                params[
                    "learning_rate"
                ]
            )
        )
    )

    early_stop = EarlyStopping(

        monitor="val_loss",

        patience=(
            PATIENCE_LSTM
        ),

        restore_best_weights=True
    )

    model.fit(

        [
            X_hist,
            X_future
        ],

        Y,

        validation_split=0.2,

        epochs=(
            EPOCHS_LSTM
        ),

        batch_size=(
            params[
                "batch_size"
            ]
        ),

        callbacks=[
            early_stop
        ],

        verbose=0,

        shuffle=False
    )

    # -----------------------------------------------------
    # VALIDACION REAL DE 168H
    #
    # Origin = inicio de las ultimas 168h
    # -----------------------------------------------------

    origin = len(
        core_y
    )

    # Necesitamos core + val para obtener
    # Temp/IGAE del horizonte de validacion.

    all_y = np.asarray(
        train_y,
        dtype=float
    )

    all_exog = (
        train_exog
        .reset_index(
            drop=True
        )
    )

    hist_raw = (
        all_y[
            origin-window:
            origin
        ]
    )

    hist_exog_raw = (
        all_exog
        .iloc[
            origin-window:
            origin
        ]
        .reset_index(
            drop=True
        )
    )

    hist_scaled = escalar_historia(

        hist_raw,

        hist_exog_raw,

        scaler_y,

        scaler_exog
    )

    future_exog = (
        construir_future_exog_directa(

            all_exog,

            origin,

            FORECAST_HORIZON
        )
    )

    future_scaled = (
        scaler_exog.transform(
            future_exog[
                EXOG_COLS
            ]
        )
    )

    pred_scaled = (
        model.predict(
            [
                hist_scaled.reshape(
                    1,
                    window,
                    -1
                ),

                future_scaled.reshape(
                    1,
                    FORECAST_HORIZON,
                    len(
                        EXOG_COLS
                    )
                )
            ],

            verbose=0
        )[0]
    )

    pred = (
        scaler_y.inverse_transform(
            pred_scaled.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    val_real = (
        all_y[
            origin:
            origin
            + FORECAST_HORIZON
        ]
    )

    score = smape(
        val_real,
        pred
    )

    cleanup()

    if np.isnan(
        score
    ):

        return float(
            "inf"
        )

    return score


# =========================================================
# TUNING
# =========================================================

def tune_lstm(
    train_y,
    train_exog,
    nombre_serie
):

    study = optuna.create_study(

        direction="minimize",

        sampler=(
            TPESampler(
                seed=42
            )
        ),

        study_name=(
            f"{nombre_serie}"
            "_lstm_directa_168"
        ),

        storage=(
            f"sqlite:///"
            f"{OPTUNA_DB}"
        ),

        load_if_exists=True
    )

    study.optimize(

        lambda trial:
            objective_lstm(
                trial,
                train_y,
                train_exog
            ),

        n_trials=(
            N_TRIALS_LSTM
        ),

        show_progress_bar=False
    )

    return (
        study.best_params,
        study.trials_dataframe()
    )


# =========================================================
# FORECAST FINAL DIRECTO
# =========================================================

def forecast_lstm_directo(
    train_y,
    train_exog,
    full_exog,
    test_origin,
    best_params
):

    try:

        window = (
            best_params[
                "window"
            ]
        )

        scaler_y = (
            StandardScaler()
        )

        scaler_exog = (
            StandardScaler()
        )

        scaler_y.fit(
            np.asarray(
                train_y
            ).reshape(
                -1,
                1
            )
        )

        scaler_exog.fit(
            train_exog[
                EXOG_COLS
            ]
        )

        # -------------------------------------------------
        # DATASET COMPLETO DE TRAIN
        # -------------------------------------------------

        (
            X_hist,
            X_future,
            Y
        ) = crear_dataset_directo(

            train_y,

            train_exog,

            window,

            scaler_y,

            scaler_exog
        )

        if len(
            X_hist
        ) < 100:

            raise ValueError(
                "Muy pocas secuencias "
                "para entrenamiento directo"
            )

        # -------------------------------------------------
        # MODELO
        # -------------------------------------------------

        model = (
            construir_modelo_lstm_directo(

                window=window,

                n_features_hist=(
                    1
                    + len(
                        EXOG_COLS
                    )
                ),

                n_exog=len(
                    EXOG_COLS
                ),

                units=(
                    best_params[
                        "units"
                    ]
                ),

                dropout=(
                    best_params[
                        "dropout"
                    ]
                ),

                learning_rate=(
                    best_params[
                        "learning_rate"
                    ]
                )
            )
        )

        early_stop = EarlyStopping(

            monitor="val_loss",

            patience=(
                PATIENCE_LSTM
            ),

            restore_best_weights=True
        )

        model.fit(

            [
                X_hist,
                X_future
            ],

            Y,

            validation_split=0.2,

            epochs=(
                EPOCHS_LSTM
            ),

            batch_size=(
                best_params[
                    "batch_size"
                ]
            ),

            callbacks=[
                early_stop
            ],

            verbose=0,

            shuffle=False
        )

        # -------------------------------------------------
        # HISTORIA INMEDIATA ANTES DEL TEST
        # -------------------------------------------------

        hist_y = (
            train_y[
                -window:
            ]
        )

        hist_exog = (
            train_exog
            .iloc[
                -window:
            ]
            .reset_index(
                drop=True
            )
        )

        hist_scaled = (
            escalar_historia(

                hist_y,

                hist_exog,

                scaler_y,

                scaler_exog
            )
        )

        # -------------------------------------------------
        # EXOGENAS PARA LAS 168H FUTURAS
        # -------------------------------------------------

        future_exog = (
            construir_future_exog_directa(

                full_exog,

                test_origin,

                FORECAST_HORIZON
            )
        )

        future_scaled = (
            scaler_exog.transform(
                future_exog[
                    EXOG_COLS
                ]
            )
        )

        # -------------------------------------------------
        # UNA SOLA PREDICCION
        # -------------------------------------------------

        pred_scaled = (
            model.predict(

                [
                    hist_scaled.reshape(
                        1,
                        window,
                        -1
                    ),

                    future_scaled.reshape(
                        1,
                        FORECAST_HORIZON,
                        len(
                            EXOG_COLS
                        )
                    )
                ],

                verbose=0
            )[0]
        )

        pred = (
            scaler_y.inverse_transform(
                pred_scaled.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        return pred

    except Exception as e:

        print(
            "         Error forecast "
            "LSTM directa: "
            f"{type(e).__name__}: "
            f"{str(e)[:150]}"
        )

        return np.full(
            FORECAST_HORIZON,
            np.nan
        )

    finally:

        cleanup()


# =========================================================
# RESULTADOS
# =========================================================

RESULTS_SERIES = []
RESULTS_METRICS = []
RESULTS_TRIALS = []
RESULTS_CONFIG_USADA = []


def guardar_predicciones(
    nombre_serie,
    fechas_test,
    pred,
    modelo
):

    for j, pred_val in enumerate(
        pred
    ):

        RESULTS_SERIES.append({

            "serie":
                nombre_serie,

            "fecha":
                fechas_test[j],

            "tipo":
                "prediccion",

            "subset":
                "test",

            "modelo":
                modelo,

            "valor":
                pred_val
        })


def guardar_metricas(
    nombre_serie,
    modelo,
    metricas
):

    RESULTS_METRICS.append({

        "serie":
            nombre_serie,

        "modelo":
            modelo,

        "tuneado":
            True,

        "horizonte_usado":
            "168_horas_directas",

        "MAPE":
            metricas[
                "MAPE"
            ],

        "sMAPE":
            metricas[
                "sMAPE"
            ],

        "MAE":
            metricas[
                "MAE"
            ],

        "RMSE":
            metricas[
                "RMSE"
            ]
    })


# =========================================================
# GUARDAR CSV
# =========================================================

def guardar_todos_csv():

    os.makedirs(
        DRIVE_OUTPUT_DIR,
        exist_ok=True
    )

    rutas = {

        "series":
            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_series.csv"
            ),

        "metricas":
            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_metricas.csv"
            ),

        "trials":
            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_optuna_trials.csv"
            ),

        "config":
            os.path.join(
                DRIVE_OUTPUT_DIR,
                f"{SAVE_PREFIX}"
                "_config_usada.csv"
            )
    }

    # -----------------------------------------------------
    # SERIES
    # -----------------------------------------------------

    df_series = pd.DataFrame(
        RESULTS_SERIES
    )

    if not df_series.empty:

        df_series[
            "fecha"
        ] = pd.to_datetime(
            df_series[
                "fecha"
            ],
            errors="coerce"
        )

        df_series = (
            df_series
            .sort_values(
                [
                    "serie",
                    "fecha",
                    "modelo"
                ]
            )
        )

    df_series.to_csv(
        rutas[
            "series"
        ],
        index=False,
        encoding="utf-8-sig"
    )

    # -----------------------------------------------------
    # METRICAS
    # -----------------------------------------------------

    df_metrics = pd.DataFrame(
        RESULTS_METRICS
    )

    if not df_metrics.empty:

        df_metrics = (
            df_metrics
            .sort_values(
                [
                    "serie",
                    "MAPE"
                ]
            )
        )

    df_metrics.to_csv(
        rutas[
            "metricas"
        ],
        index=False,
        encoding="utf-8-sig"
    )

    # -----------------------------------------------------
    # TRIALS
    # -----------------------------------------------------

    if RESULTS_TRIALS:

        df_trials = pd.concat(
            RESULTS_TRIALS,
            ignore_index=True
        )

    else:

        df_trials = (
            pd.DataFrame()
        )

    df_trials.to_csv(
        rutas[
            "trials"
        ],
        index=False,
        encoding="utf-8-sig"
    )

    # -----------------------------------------------------
    # CONFIG
    # -----------------------------------------------------

    df_config = pd.DataFrame(
        RESULTS_CONFIG_USADA
    )

    df_config.to_csv(
        rutas[
            "config"
        ],
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "\nArchivos guardados:"
    )

    print(
        f"Series:   "
        f"{rutas['series']}"
    )

    print(
        f"Metricas: "
        f"{rutas['metricas']}"
    )

    print(
        f"Trials:   "
        f"{rutas['trials']}"
    )

    print(
        f"Config:   "
        f"{rutas['config']}"
    )

    return df_metrics


# =========================================================
# EVALUAR UNA SERIE
# =========================================================

def evaluar_serie(
    nombre_serie,
    serie,
    fechas,
    exogenas_df
):

    required_hours = (
        LSTM_TRAIN_HOURS
        + FORECAST_HORIZON
    )

    if len(
        serie
    ) < required_hours:

        print(
            f"Serie insuficiente: "
            f"{nombre_serie}"
        )

        return

    # -----------------------------------------------------
    # EXOGENAS COMPLETAS
    # -----------------------------------------------------

    exog_serie = (
        alinear_exogenas_a_fechas(
            fechas,
            exogenas_df
        )
    )

    # -----------------------------------------------------
    # TRAIN / TEST
    # -----------------------------------------------------

    train_start = (
        len(serie)
        - FORECAST_HORIZON
        - LSTM_TRAIN_HOURS
    )

    test_origin = (
        len(serie)
        - FORECAST_HORIZON
    )

    train = (
        serie[
            train_start:
            test_origin
        ]
    )

    test = (
        serie[
            test_origin:
        ]
    )

    fechas_test = (
        fechas[
            test_origin:
        ]
    )

    train_exog = (
        exog_serie
        .iloc[
            train_start:
            test_origin
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        "\nSplit:"
    )

    print(
        f"   Train: "
        f"{len(train):,} h"
    )

    print(
        f"   Test:  "
        f"{len(test):,} h"
    )

    print(
        f"   Estrategia: "
        "DIRECTA 168h"
    )

    print(
        "\nExogenas:"
    )

    for col in (
        EXOG_COLS
    ):

        if (
            col
            in EXOG_CONOCIDAS_FUTURO
        ):

            print(
                f"   {col}: "
                "valor del horizonte"
            )

        else:

            print(
                f"   {col}: "
                "lag 168h"
            )

    # -----------------------------------------------------
    # OPTUNA
    # -----------------------------------------------------

    print(
        "\nLSTM directa tuning..."
    )

    best_params, trials_df = (
        tune_lstm(
            train,
            train_exog,
            nombre_serie
        )
    )

    print(
        "\nMejores parametros:"
    )

    print(
        best_params
    )

    trials_df[
        "serie"
    ] = nombre_serie

    trials_df[
        "modelo"
    ] = (
        "LSTM_Directa_168"
    )

    RESULTS_TRIALS.append(
        trials_df
    )

    RESULTS_CONFIG_USADA.append({

        "serie":
            nombre_serie,

        "modelo":
            "LSTM_Directa_168",

        "parametros":
            str(
                best_params
            ),

        "horizonte_usado":
            "168_horas_directas",

        "train_horas":
            LSTM_TRAIN_HOURS,

        "exogenas":
            str(
                EXOG_COLS
            )
    })

    # -----------------------------------------------------
    # FORECAST FINAL
    # -----------------------------------------------------

    pred = (
        forecast_lstm_directo(

            train_y=train,

            train_exog=(
                train_exog
            ),

            full_exog=(
                exog_serie
            ),

            test_origin=(
                test_origin
            ),

            best_params=(
                best_params
            )
        )
    )

    print(
        f"\nPredicciones validas: "
        f"{np.isfinite(pred).sum()}"
        f"/{len(pred)}"
    )

    metricas = (
        calcular_metricas(
            test,
            pred
        )
    )

    if metricas is None:

        print(
            "No fue posible "
            "calcular metricas."
        )

        return

    guardar_metricas(
        nombre_serie,
        "LSTM_Directa_168",
        metricas
    )

    guardar_predicciones(
        nombre_serie,
        fechas_test,
        pred,
        "LSTM_Directa_168"
    )

    print(
        "\nLSTM_Directa_168:"
    )

    print(
        f"   MAPE:  "
        f"{metricas['MAPE']:.2f}%"
    )

    print(
        f"   sMAPE: "
        f"{metricas['sMAPE']:.2f}%"
    )

    print(
        f"   MAE:   "
        f"{metricas['MAE']:.2f}"
    )

    print(
        f"   RMSE:  "
        f"{metricas['RMSE']:.2f}"
    )


# =========================================================
# PIPELINE PRINCIPAL
# =========================================================

def ejecutar_pipeline():

    global RESULTS_SERIES
    global RESULTS_METRICS
    global RESULTS_TRIALS
    global RESULTS_CONFIG_USADA

    RESULTS_SERIES = []
    RESULTS_METRICS = []
    RESULTS_TRIALS = []
    RESULTS_CONFIG_USADA = []

    print(
        "=" * 80
    )

    print(
        PIPELINE_NAME
    )

    print(
        "=" * 80
    )

    print(
        f"Train: "
        f"{LSTM_TRAIN_HOURS} h"
    )

    print(
        f"Forecast: "
        f"{FORECAST_HORIZON} h"
    )

    print(
        "Estrategia: "
        "DIRECTA MULTI-HORIZONTE"
    )

    print(
        "\nExogenas activas:"
    )

    for col in (
        EXOG_COLS
    ):

        print(
            f"   - {col}"
        )

    regiones = (
        cargar_regiones()
    )

    for (
        region,
        df
    ) in regiones.items():

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"Serie: "
            f"{region}_DEMANDA"
        )

        print(
            "=" * 80
        )

        try:

            exogenas_df = (
                merge_exogenas(
                    region
                )
            )

            nombre_serie = (
                f"{region}_DEMANDA"
            )

            serie, fechas = (
                extraer_serie_horaria(
                    df,
                    COL_DEMANDA,
                    nombre_serie
                )
            )

            print(
                f"Serie completa: "
                f"{len(serie):,} "
                "observaciones"
            )

            evaluar_serie(
                nombre_serie,
                serie,
                fechas,
                exogenas_df
            )

            print(
                "\nGuardando avance..."
            )

            guardar_todos_csv()

        except Exception as e:

            print(
                f"Error {region}: "
                f"{type(e).__name__}: "
                f"{e}"
            )

        finally:

            cleanup()

    # -----------------------------------------------------
    # FINAL
    # -----------------------------------------------------

    df_metricas = (
        guardar_todos_csv()
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        "PIPELINE COMPLETADO"
    )

    print(
        "=" * 80
    )

    if (
        df_metricas
        is not None
        and not df_metricas.empty
    ):

        print(
            "\nResultados:"
        )

        print(
            df_metricas
            .sort_values(
                "MAPE"
            )
            .to_string(
                index=False
            )
        )


# =========================================================
# EJECUTAR
# =========================================================

ejecutar_pipeline()

Mounted at /content/drive
PIPELINE LSTM DIRECTA 168H
Train: 2160 h
Forecast: 168 h
Estrategia: DIRECTA MULTI-HORIZONTE

Exogenas activas:
   - Temperatura
   - IGAE
   - Generacion
   - Importacion
   - Exportacion
OK BCA: BCA_long.csv (64792, 11)
OK CEN: CEN_long.csv (64796, 11)
OK NES: NES_long.csv (64796, 11)
OK NOR: NOR_long.csv (64796, 11)
OK NTE: NTE_long.csv (64796, 11)
OK OCC: OCC_long.csv (64796, 11)
OK ORI: ORI_long.csv (64796, 11)
OK PEN: PEN_long.csv (64796, 11)

Serie: BCA_DEMANDA
Serie completa: 64,792 observaciones

Split:
   Train: 2,160 h
   Test:  168 h
   Estrategia: DIRECTA 168h

Exogenas:
   Temperatura: valor del horizonte
   IGAE: valor del horizonte
   Generacion: lag 168h
   Importacion: lag 168h
   Exportacion: lag 168h

LSTM directa tuning...


[I 2026-08-06 15:53:15,668] A new study created in RDB with name: BCA_DEMANDA_lstm_directa_168
[I 2026-08-06 15:54:00,586] Trial 0 finished with value: 12.718832039206912 and parameters: {'units': 64, 'dropout': 0.0, 'batch_size': 32, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 12.718832039206912.
[I 2026-08-06 15:54:42,464] Trial 1 finished with value: 10.687272815942093 and parameters: {'units': 128, 'dropout': 0.2, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 10.687272815942093.
[I 2026-08-06 15:55:24,024] Trial 2 finished with value: 10.638897545414462 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 2 with value: 10.638897545414462.
[I 2026-08-06 15:56:01,732] Trial 3 finished with value: 28.761338641713763 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 32, 'window': 24, 'learning_rate': 0.003}. Best is trial 2 with value: 10.638897545414


Mejores parametros:
{'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}

Predicciones validas: 168/168

LSTM_Directa_168:
   MAPE:  9.21%
   sMAPE: 9.34%
   MAE:   179.35
   RMSE:  202.77

Guardando avance...

Archivos guardados:
Series:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_series.csv
Metricas: /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_metricas.csv
Trials:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_optuna_trials.csv
Config:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_config_usada.csv

Serie: CEN_DEMANDA
Serie completa: 64,796 observaciones

Split:
   Train: 2,160 h
   Test:  168 h
   Estrategia: DIRECTA 168h

Exogenas:
   Temperatura: valor del horizonte
   IGAE: valor del horizonte
   Generacion: lag 168h
   Importacion: lag 168h
   Exportacion: lag 168h

LSTM directa tuning...


[I 2026-08-06 16:00:44,807] A new study created in RDB with name: CEN_DEMANDA_lstm_directa_168
[I 2026-08-06 16:01:30,055] Trial 0 finished with value: 5.9114993979282975 and parameters: {'units': 64, 'dropout': 0.0, 'batch_size': 32, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 5.9114993979282975.
[I 2026-08-06 16:02:09,293] Trial 1 finished with value: 5.408011734509738 and parameters: {'units': 128, 'dropout': 0.2, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 5.408011734509738.
[I 2026-08-06 16:02:49,609] Trial 2 finished with value: 5.019417741120058 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 2 with value: 5.019417741120058.
[I 2026-08-06 16:03:27,064] Trial 3 finished with value: 7.774470253726184 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 32, 'window': 24, 'learning_rate': 0.003}. Best is trial 2 with value: 5.019417741120058.
[


Mejores parametros:
{'units': 64, 'dropout': 0.3, 'batch_size': 32, 'window': 24, 'learning_rate': 0.003}

Predicciones validas: 168/168

LSTM_Directa_168:
   MAPE:  5.16%
   sMAPE: 5.18%
   MAE:   374.14
   RMSE:  453.70

Guardando avance...

Archivos guardados:
Series:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_series.csv
Metricas: /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_metricas.csv
Trials:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_optuna_trials.csv
Config:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_config_usada.csv

Serie: NES_DEMANDA


...


Mejores parametros:
{'units': 64, 'dropout': 0.2, 'batch_size': 32, 'window': 48, 'learning_rate': 0.001}

Predicciones validas: 168/168

LSTM_Directa_168:
   MAPE:  6.47%
   sMAPE: 6.74%
   MAE:   523.12
   RMSE:  590.52

Guardando avance...

Archivos guardados:
Series:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_series.csv
Metricas: /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_metricas.csv
Trials:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_optuna_trials.csv
Config:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_config_usada.csv

Serie: PEN_DEMANDA
Serie completa: 64,796 observaciones


[I 2026-08-06 16:47:24,003] A new study created in RDB with name: PEN_DEMANDA_lstm_directa_168



Split:
   Train: 2,160 h
   Test:  168 h
   Estrategia: DIRECTA 168h

Exogenas:
   Temperatura: valor del horizonte
   IGAE: valor del horizonte
   Generacion: lag 168h
   Importacion: lag 168h
   Exportacion: lag 168h

LSTM directa tuning...


[I 2026-08-06 16:48:07,956] Trial 0 finished with value: 15.170561223019222 and parameters: {'units': 64, 'dropout': 0.0, 'batch_size': 32, 'window': 48, 'learning_rate': 0.001}. Best is trial 0 with value: 15.170561223019222.
[I 2026-08-06 16:48:51,299] Trial 1 finished with value: 6.333497222665247 and parameters: {'units': 128, 'dropout': 0.2, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}. Best is trial 1 with value: 6.333497222665247.
[I 2026-08-06 16:49:31,399] Trial 2 finished with value: 8.956011705269306 and parameters: {'units': 32, 'dropout': 0.1, 'batch_size': 64, 'window': 48, 'learning_rate': 0.001}. Best is trial 1 with value: 6.333497222665247.
[I 2026-08-06 16:50:11,354] Trial 3 finished with value: 26.994458649487964 and parameters: {'units': 32, 'dropout': 0.2, 'batch_size': 32, 'window': 24, 'learning_rate': 0.003}. Best is trial 1 with value: 6.333497222665247.
[I 2026-08-06 16:50:55,898] Trial 4 finished with value: 6.1524588235972235 and parameters: {'u


Mejores parametros:
{'units': 128, 'dropout': 0.2, 'batch_size': 128, 'window': 24, 'learning_rate': 0.001}

Predicciones validas: 168/168

LSTM_Directa_168:
   MAPE:  5.88%
   sMAPE: 6.13%
   MAE:   145.52
   RMSE:  175.89

Guardando avance...

Archivos guardados:
Series:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_series.csv
Metricas: /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_metricas.csv
Trials:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_optuna_trials.csv
Config:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_config_usada.csv

Archivos guardados:
Series:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_series.csv
Metricas: /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_metricas.csv
Trials:   /content/drive/MyDrive/Pipeline_Resultados/lstm_directa_168_demanda_horario_optuna_trials.csv
Config:  

#Ensamble

In [ ]:
# =========================================================
# ENSEMBLE POR COMPONENTES CON EXOGENAS
#
# STL
# +
# LSTM(TENDENCIA + EXOG)
# +
# FCNN(ESTACIONALIDAD + EXOG)
# +
# AR(RESIDUO)
#
# SOLO DEMANDA
#
# Train: 3600 h
# Test:  168 h
# Window: 168 h
#
# EXOGENAS:
#
# Contemporaneas:
#   - Temperaturas
#   - IGAE
#
# Rezagadas 168 h:
#   - Generacion_lag168
#   - Importacion_lag168
#   - Exportacion_lag168
#
# Para predecir una hora t:
#
#   Temperaturas[t]
#   IGAE[t]
#
#   Generacion_lag168[t]  = Generacion[t-168]
#   Importacion_lag168[t] = Importacion[t-168]
#   Exportacion_lag168[t] = Exportacion[t-168]
#
# Esto evita utilizar GEN / IMP / EXP reales
# correspondientes al horizonte futuro.
#
# El AR del residuo permanece UNIVARIADO.
# =========================================================


# =========================================================
# INSTALL
# =========================================================

!pip install -q optuna


# =========================================================
# DRIVE
# =========================================================

from google.colab import drive
drive.mount("/content/drive")


# =========================================================
# IMPORTS
# =========================================================

import os
import gc
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
import tensorflow as tf

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.ar_model import AutoReg

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, LSTM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = "/content"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Tesis/Resultados/"
    "ENSEMBLE_STL_LSTM_FCNN_AR_EXOG_ALL_demanda_5m_1w"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

INCREMENTAL_DIR = os.path.join(
    OUTPUT_DIR,
    "por_serie"
)

os.makedirs(
    INCREMENTAL_DIR,
    exist_ok=True
)


# =========================================================
# REGIONES
# =========================================================

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}


# =========================================================
# COLUMNAS
# =========================================================

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = (
    "Estimacion de Demanda por Balance (MWh)"
)


# =========================================================
# TRAIN / TEST
# =========================================================

TRAIN_LAST_HOURS = 24 * 30 * 5   # 3600 h
TEST_HORIZON = 24 * 7            # 168 h

WINDOW = 168
STL_PERIOD = 168

MAX_LAG_AR = 168

N_TRIALS = 5
EPOCHS = 60
SEED = 42


# =========================================================
# LAG ELECTRICO
# =========================================================

ELECTRIC_LAG = 168


# =========================================================
# EXOGENAS
#
# Puedes comentar cualquiera.
# =========================================================

EXOG_NAMES = [
    "Temperaturas",
    "IGAE",
    "Generacion_lag168",
    "Importacion_lag168",
    "Exportacion_lag168",
]


# =========================================================
# SEEDS
# =========================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)


# =========================================================
# METRICAS
# =========================================================

def mape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                / y_true[mask]
            )
        )
        * 100
    )


def smape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denom = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (denom != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                y_true[mask]
                - y_pred[mask]
            )
            / denom[mask]
        )
        * 100
    )


def calcular_metricas(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return {
        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAPE":
            mape(
                y_true,
                y_pred
            ),

        "sMAPE":
            smape(
                y_true,
                y_pred
            ),
    }


# =========================================================
# CONVERTIR HORA A 0-23
# =========================================================

def convertir_hora_0_23(
    serie_hora
):

    hora = pd.to_numeric(
        serie_hora,
        errors="coerce"
    )

    if hora.dropna().empty:
        return hora

    if (
        hora.min() >= 1
        and hora.max() <= 24
    ):

        return (
            hora.astype(float)
            - 1
        )

    return hora.astype(float)


# =========================================================
# LECTURA DEMANDA
# =========================================================

def extraer_serie_horaria(
    df,
    columna
):

    aux = df[
        [
            COL_FECHA,
            COL_HORA,
            columna
        ]
    ].copy()

    aux[
        COL_FECHA
    ] = pd.to_datetime(
        aux[
            COL_FECHA
        ],
        errors="coerce"
    )

    aux[
        COL_HORA
    ] = pd.to_numeric(
        aux[
            COL_HORA
        ],
        errors="coerce"
    )

    aux[
        columna
    ] = pd.to_numeric(
        aux[
            columna
        ],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            COL_FECHA,
            COL_HORA,
            columna
        ]
    )

    aux[
        "hora_0_23"
    ] = convertir_hora_0_23(
        aux[
            COL_HORA
        ]
    )

    aux = aux.dropna(
        subset=[
            "hora_0_23"
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            "hora_0_23"
        ]
        .astype(int)
    )

    aux[
        "datetime"
    ] = (
        aux[
            COL_FECHA
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux
        .sort_values(
            "datetime"
        )
        .drop_duplicates(
            "datetime",
            keep="last"
        )
    )

    return (
        aux[
            columna
        ].to_numpy(
            dtype=float
        ),

        aux[
            "datetime"
        ].to_numpy()
    )


# =========================================================
# PREPARAR EXOGENA HORARIA
# =========================================================

def preparar_exogena_horaria(
    df,
    nombre
):

    aux = df.copy()

    aux.columns = (
        aux.columns
        .astype(str)
        .str.strip()
    )

    cols_lower = {
        c.lower(): c
        for c in aux.columns
    }

    if "fecha" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no tiene columna fecha"
        )

    if "hora" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no tiene columna hora/Hora"
        )

    if "valor" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no tiene columna valor"
        )

    fecha_col = cols_lower[
        "fecha"
    ]

    hora_col = cols_lower[
        "hora"
    ]

    valor_col = cols_lower[
        "valor"
    ]

    aux[
        fecha_col
    ] = pd.to_datetime(
        aux[
            fecha_col
        ],
        errors="coerce"
    )

    aux[
        hora_col
    ] = pd.to_numeric(
        aux[
            hora_col
        ],
        errors="coerce"
    )

    aux[
        valor_col
    ] = pd.to_numeric(
        aux[
            valor_col
        ],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            fecha_col,
            hora_col,
            valor_col
        ]
    )

    aux[
        "hora_0_23"
    ] = convertir_hora_0_23(
        aux[
            hora_col
        ]
    )

    aux = aux.dropna(
        subset=[
            "hora_0_23"
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            "hora_0_23"
        ]
        .astype(int)
    )

    aux[
        "datetime"
    ] = (
        aux[
            fecha_col
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux[
            [
                "datetime",
                valor_col
            ]
        ]
        .rename(
            columns={
                valor_col:
                    nombre
            }
        )
        .groupby(
            "datetime",
            as_index=False
        )
        .mean()
        .sort_values(
            "datetime"
        )
    )

    return aux


# =========================================================
# PREPARAR EXOGENA ELECTRICA LAG 168
# =========================================================

def preparar_exogena_lag168(
    df,
    nombre
):

    """
    Construye:

        X_lag168[t] = X[t-168]

    La forma más limpia de hacerlo es
    desplazar el timestamp 168 horas
    hacia adelante.
    """

    aux = (
        preparar_exogena_horaria(
            df,
            nombre
        )
    )

    aux[
        "datetime"
    ] = (
        aux[
            "datetime"
        ]
        + pd.to_timedelta(
            ELECTRIC_LAG,
            unit="h"
        )
    )

    return aux


# =========================================================
# MATRIZ EXOGENA POR REGION
# =========================================================

def construir_matriz_exogena_region(
    region
):

    dfs = []

    # -----------------------------------------------------
    # TEMPERATURA CONTEMPORANEA
    # -----------------------------------------------------

    if (
        "Temperaturas"
        in EXOG_NAMES
    ):

        if (
            "Temperaturas_H"
            not in globals()
        ):

            raise ValueError(
                "No existe "
                "Temperaturas_H"
            )

        temp = (
            preparar_exogena_horaria(
                Temperaturas_H,
                "Temperaturas"
            )
        )

        dfs.append(
            temp
        )

    # -----------------------------------------------------
    # IGAE CONTEMPORANEO
    # -----------------------------------------------------

    if (
        "IGAE"
        in EXOG_NAMES
    ):

        if (
            "IGAE_H"
            not in globals()
        ):

            raise ValueError(
                "No existe "
                "IGAE_H"
            )

        igae = (
            preparar_exogena_horaria(
                IGAE_H,
                "IGAE"
            )
        )

        dfs.append(
            igae
        )

    # -----------------------------------------------------
    # VALIDAR DICCIONARIO SERIES
    # -----------------------------------------------------

    electricas_activas = [
        x
        for x in [
            "Generacion_lag168",
            "Importacion_lag168",
            "Exportacion_lag168",
        ]
        if x in EXOG_NAMES
    ]

    if (
        len(
            electricas_activas
        )
        > 0
    ):

        if (
            "series"
            not in globals()
        ):

            raise ValueError(
                "No existe el "
                "diccionario global "
                "'series'."
            )

    # -----------------------------------------------------
    # GENERACION
    # -----------------------------------------------------

    if (
        "Generacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_GEN"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        gen = (
            preparar_exogena_lag168(
                series[
                    key
                ],
                "Generacion_lag168"
            )
        )

        dfs.append(
            gen
        )

    # -----------------------------------------------------
    # IMPORTACION
    # -----------------------------------------------------

    if (
        "Importacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_IMP"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        imp = (
            preparar_exogena_lag168(
                series[
                    key
                ],
                "Importacion_lag168"
            )
        )

        dfs.append(
            imp
        )

    # -----------------------------------------------------
    # EXPORTACION
    # -----------------------------------------------------

    if (
        "Exportacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_EXP"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        exp = (
            preparar_exogena_lag168(
                series[
                    key
                ],
                "Exportacion_lag168"
            )
        )

        dfs.append(
            exp
        )

    # -----------------------------------------------------
    # VALIDAR
    # -----------------------------------------------------

    if len(
        dfs
    ) == 0:

        raise ValueError(
            "No hay exogenas "
            "activas."
        )

    # -----------------------------------------------------
    # MERGE
    # -----------------------------------------------------

    exog = dfs[0]

    for df_next in (
        dfs[1:]
    ):

        exog = exog.merge(
            df_next,
            on="datetime",
            how="outer"
        )

    exog = (
        exog
        .sort_values(
            "datetime"
        )
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # RELLENO
    #
    # Se conserva la filosofia
    # del pipeline original.
    # -----------------------------------------------------

    exog[
        EXOG_NAMES
    ] = (
        exog[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    # -----------------------------------------------------
    # RESUMEN
    # -----------------------------------------------------

    print(
        f"\nExogenas ensemble "
        f"{region}:"
    )

    for col in (
        EXOG_NAMES
    ):

        print(
            f"   {col:25s} | "
            f"{exog[col].notna().sum():,}"
        )

    print(
        f"Rango: "
        f"{exog['datetime'].min()} "
        f"-> "
        f"{exog['datetime'].max()}"
    )

    return exog


# =========================================================
# ALINEAR EXOGENAS CON FECHAS
# =========================================================

def alinear_exogenas_con_fechas(
    fechas,
    exog_region
):

    base = pd.DataFrame({

        "datetime":
            pd.to_datetime(
                fechas
            )
    })

    X = base.merge(

        exog_region[
            [
                "datetime"
            ]
            + EXOG_NAMES
        ],

        on="datetime",

        how="left"
    )

    X[
        EXOG_NAMES
    ] = (
        X[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    if (
        X[
            EXOG_NAMES
        ]
        .isna()
        .any()
        .any()
    ):

        faltantes = (
            X[
                EXOG_NAMES
            ]
            .isna()
            .sum()
            .to_dict()
        )

        raise ValueError(
            "Exogenas faltantes: "
            f"{faltantes}"
        )

    return (
        X[
            EXOG_NAMES
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# VENTANAS MULTIVARIADAS
# =========================================================

def crear_ventanas(
    y,
    window,
    exog=None
):

    """
    Para predecir componente[t]:

        componente[t-window:t]

    Si hay exogenas:

        exog[t-window:t]

        +

        exog[t]

    IMPORTANTE:

    Las electricas ya estan construidas
    como lag168.

    Por tanto exog[t] significa:

        Temperaturas[t]
        IGAE[t]
        Generacion[t-168]
        Importacion[t-168]
        Exportacion[t-168]
    """

    X = []
    Y = []

    y = np.asarray(
        y,
        dtype=float
    )

    if (
        exog
        is not None
    ):

        exog_values = (
            np.asarray(
                exog,
                dtype=float
            )
        )

    else:

        exog_values = None

    for i in range(
        window,
        len(y)
    ):

        y_window = (
            y[
                i-window:
                i
            ]
        )

        if (
            exog_values
            is not None
        ):

            exog_window = (
                exog_values[
                    i-window:
                    i
                ]
                .reshape(-1)
            )

            exog_actual = (
                exog_values[
                    i
                ]
                .reshape(-1)
            )

            features = (
                np.concatenate(
                    [
                        y_window,
                        exog_window,
                        exog_actual
                    ]
                )
            )

        else:

            features = (
                y_window
            )

        X.append(
            features
        )

        Y.append(
            y[i]
        )

    return (
        np.asarray(X),
        np.asarray(Y)
    )


# =========================================================
# RESHAPE LSTM
# =========================================================

def reshape_lstm_features(
    X
):

    return (
        X.reshape(
            X.shape[0],
            1,
            X.shape[1]
        )
    )


# =========================================================
# LSTM PARA TENDENCIA
# =========================================================

def construir_lstm(
    params,
    input_dim
):

    model = Sequential()

    model.add(
        Input(
            shape=(
                1,
                input_dim
            )
        )
    )

    if (
        params[
            "n_layers"
        ]
        == 1
    ):

        model.add(
            LSTM(
                params[
                    "units"
                ]
            )
        )

    else:

        model.add(
            LSTM(
                params[
                    "units"
                ],
                return_sequences=True
            )
        )

        model.add(
            Dropout(
                params[
                    "dropout"
                ]
            )
        )

        model.add(
            LSTM(
                params[
                    "units_2"
                ]
            )
        )

    model.add(
        Dropout(
            params[
                "dropout"
            ]
        )
    )

    model.add(
        Dense(1)
    )

    model.compile(

        optimizer=Adam(
            learning_rate=(
                params[
                    "learning_rate"
                ]
            )
        ),

        loss="mse"
    )

    return model


# =========================================================
# OPTUNA LSTM TREND
# =========================================================

def tunear_lstm(
    train_component,
    nombre_serie,
    exog_train=None
):

    X, y = crear_ventanas(
        train_component,
        WINDOW,
        exog=(
            exog_train
        )
    )

    val_size = (
        TEST_HORIZON
    )

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        reshape_lstm_features(
            scaler_x.fit_transform(
                X_train
            )
        )
    )

    X_val_s = (
        reshape_lstm_features(
            scaler_x.transform(
                X_val
            )
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    def objective(
        trial
    ):

        tf.keras.backend.clear_session()

        params = {

            "n_layers":
                trial.suggest_categorical(
                    "n_layers",
                    [
                        1,
                        2
                    ]
                ),

            "units":
                trial.suggest_categorical(
                    "units",
                    [
                        32,
                        64,
                        128
                    ]
                ),

            "dropout":
                trial.suggest_float(
                    "dropout",
                    0.0,
                    0.4
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    1e-4,
                    3e-3,
                    log=True
                ),

            "batch_size":
                trial.suggest_categorical(
                    "batch_size",
                    [
                        32,
                        64,
                        128
                    ]
                ),
        }

        if (
            params[
                "n_layers"
            ]
            == 2
        ):

            params[
                "units_2"
            ] = (
                trial.suggest_categorical(
                    "units_2",
                    [
                        32,
                        64,
                        128
                    ]
                )
            )

        else:

            params[
                "units_2"
            ] = 0

        model = (
            construir_lstm(
                params,
                input_dim=(
                    X_train_s.shape[
                        2
                    ]
                )
            )
        )

        early = (
            EarlyStopping(

                monitor="val_loss",

                patience=8,

                restore_best_weights=True
            )
        )

        model.fit(

            X_train_s,

            y_train_s,

            validation_data=(
                X_val_s,
                y_val_s
            ),

            epochs=EPOCHS,

            batch_size=(
                params[
                    "batch_size"
                ]
            ),

            verbose=0,

            callbacks=[
                early
            ]
        )

        pred_s = (
            model.predict(
                X_val_s,
                verbose=0
            )
        )

        pred = (
            scaler_y
            .inverse_transform(
                pred_s.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        return (
            mean_absolute_error(
                y_val,
                pred
            )
        )

    db_path = (
        os.path.join(
            OUTPUT_DIR,
            f"{nombre_serie}"
            "_LSTM_trend_EXOG_ALL_optuna.db"
        )
    )

    study = (
        optuna.create_study(

            direction="minimize",

            study_name=(
                f"{nombre_serie}"
                "_LSTM_trend_EXOG_ALL"
            ),

            storage=(
                f"sqlite:///"
                f"{db_path}"
            ),

            load_if_exists=True
        )
    )

    completed = len([

        t

        for t in study.trials

        if (
            t.state
            == optuna.trial.TrialState.COMPLETE
        )
    ])

    remaining = max(
        0,
        N_TRIALS
        - completed
    )

    print(
        f"      LSTM trend "
        f"trials completos: "
        f"{completed}"
    )

    print(
        f"      LSTM trend "
        f"trials restantes: "
        f"{remaining}"
    )

    if remaining > 0:

        study.optimize(
            objective,
            n_trials=(
                remaining
            )
        )

    return (
        study.best_params,
        study.best_value
    )


# =========================================================
# ENTRENAR LSTM FINAL
# =========================================================

def entrenar_lstm_final(
    train_component,
    params,
    exog_train=None
):

    X, y = crear_ventanas(
        train_component,
        WINDOW,
        exog=(
            exog_train
        )
    )

    val_size = (
        TEST_HORIZON
    )

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        reshape_lstm_features(
            scaler_x.fit_transform(
                X_train
            )
        )
    )

    X_val_s = (
        reshape_lstm_features(
            scaler_x.transform(
                X_val
            )
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    params = (
        params.copy()
    )

    params[
        "units_2"
    ] = (
        params.get(
            "units_2",
            0
        )
    )

    model = (
        construir_lstm(
            params,
            input_dim=(
                X_train_s.shape[
                    2
                ]
            )
        )
    )

    early = (
        EarlyStopping(

            monitor="val_loss",

            patience=10,

            restore_best_weights=True
        )
    )

    model.fit(

        X_train_s,

        y_train_s,

        validation_data=(
            X_val_s,
            y_val_s
        ),

        epochs=EPOCHS,

        batch_size=(
            params[
                "batch_size"
            ]
        ),

        verbose=0,

        callbacks=[
            early
        ]
    )

    del (
        X,
        y,
        X_train,
        y_train,
        X_val,
        y_val,
        X_train_s,
        X_val_s,
        y_train_s,
        y_val_s
    )

    gc.collect()

    return (
        model,
        scaler_x,
        scaler_y
    )


# =========================================================
# FORECAST RECURSIVO LSTM
# =========================================================

def forecast_recursivo_lstm(
    model,
    train_component,
    horizon,
    scaler_x,
    scaler_y,
    exog_hist=None,
    exog_future=None
):

    historial = list(
        np.asarray(
            train_component,
            dtype=float
        )
    )

    preds = []

    if (
        exog_hist
        is not None
    ):

        exog_total = (
            pd.concat(
                [
                    exog_hist
                    .reset_index(
                        drop=True
                    ),

                    exog_future
                    .reset_index(
                        drop=True
                    )
                ],

                ignore_index=True
            )
        )

        exog_values = (
            np.asarray(
                exog_total,
                dtype=float
            )
        )

    else:

        exog_values = None

    for step in range(
        horizon
    ):

        y_window = (
            np.asarray(
                historial[
                    -WINDOW:
                ],
                dtype=float
            )
        )

        if (
            exog_values
            is not None
        ):

            end_idx = (
                len(
                    train_component
                )
                + step
            )

            exog_window = (
                exog_values[
                    end_idx-WINDOW:
                    end_idx
                ]
                .reshape(-1)
            )

            exog_actual = (
                exog_values[
                    end_idx
                ]
                .reshape(-1)
            )

            x = (
                np.concatenate(
                    [
                        y_window,
                        exog_window,
                        exog_actual
                    ]
                )
                .reshape(
                    1,
                    -1
                )
            )

        else:

            x = (
                y_window.reshape(
                    1,
                    -1
                )
            )

        x_s = (
            reshape_lstm_features(
                scaler_x.transform(
                    x
                )
            )
        )

        pred_s = (
            model.predict(
                x_s,
                verbose=0
            )
        )

        pred = (
            scaler_y
            .inverse_transform(
                pred_s.reshape(
                    -1,
                    1
                )
            )[0, 0]
        )

        preds.append(
            pred
        )

        historial.append(
            pred
        )

    return np.asarray(
        preds
    )


# =========================================================
# FCNN ESTACIONALIDAD
# =========================================================

def construir_fcnn(
    params,
    input_dim
):

    model = (
        Sequential()
    )

    model.add(
        Input(
            shape=(
                input_dim,
            )
        )
    )

    model.add(
        Dense(
            params[
                "units_1"
            ],
            activation="relu"
        )
    )

    model.add(
        Dropout(
            params[
                "dropout"
            ]
        )
    )

    if (
        params[
            "n_layers"
        ]
        == 2
    ):

        model.add(
            Dense(
                params[
                    "units_2"
                ],
                activation="relu"
            )
        )

        model.add(
            Dropout(
                params[
                    "dropout"
                ]
            )
        )

    model.add(
        Dense(1)
    )

    model.compile(

        optimizer=Adam(
            learning_rate=(
                params[
                    "learning_rate"
                ]
            )
        ),

        loss="mse"
    )

    return model


# =========================================================
# OPTUNA FCNN
# =========================================================

def tunear_fcnn(
    train_component,
    nombre_serie,
    exog_train=None
):

    X, y = crear_ventanas(
        train_component,
        WINDOW,
        exog=(
            exog_train
        )
    )

    val_size = (
        TEST_HORIZON
    )

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        scaler_x.fit_transform(
            X_train
        )
    )

    X_val_s = (
        scaler_x.transform(
            X_val
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    def objective(
        trial
    ):

        tf.keras.backend.clear_session()

        params = {

            "n_layers":
                trial.suggest_categorical(
                    "n_layers",
                    [
                        1,
                        2
                    ]
                ),

            "units_1":
                trial.suggest_categorical(
                    "units_1",
                    [
                        32,
                        64,
                        128,
                        256
                    ]
                ),

            "dropout":
                trial.suggest_float(
                    "dropout",
                    0.0,
                    0.4
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    1e-4,
                    3e-3,
                    log=True
                ),

            "batch_size":
                trial.suggest_categorical(
                    "batch_size",
                    [
                        32,
                        64,
                        128
                    ]
                ),
        }

        if (
            params[
                "n_layers"
            ]
            == 2
        ):

            params[
                "units_2"
            ] = (
                trial.suggest_categorical(
                    "units_2",
                    [
                        16,
                        32,
                        64,
                        128
                    ]
                )
            )

        else:

            params[
                "units_2"
            ] = 0

        model = (
            construir_fcnn(
                params,
                input_dim=(
                    X_train_s.shape[
                        1
                    ]
                )
            )
        )

        early = (
            EarlyStopping(

                monitor="val_loss",

                patience=8,

                restore_best_weights=True
            )
        )

        model.fit(

            X_train_s,

            y_train_s,

            validation_data=(
                X_val_s,
                y_val_s
            ),

            epochs=EPOCHS,

            batch_size=(
                params[
                    "batch_size"
                ]
            ),

            verbose=0,

            callbacks=[
                early
            ]
        )

        pred_s = (
            model.predict(
                X_val_s,
                verbose=0
            )
        )

        pred = (
            scaler_y
            .inverse_transform(
                pred_s.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        return (
            mean_absolute_error(
                y_val,
                pred
            )
        )

    db_path = (
        os.path.join(
            OUTPUT_DIR,
            f"{nombre_serie}"
            "_FCNN_seasonal_EXOG_ALL_optuna.db"
        )
    )

    study = (
        optuna.create_study(

            direction="minimize",

            study_name=(
                f"{nombre_serie}"
                "_FCNN_seasonal_EXOG_ALL"
            ),

            storage=(
                f"sqlite:///"
                f"{db_path}"
            ),

            load_if_exists=True
        )
    )

    completed = len([

        t

        for t in (
            study.trials
        )

        if (
            t.state
            == optuna.trial.TrialState.COMPLETE
        )
    ])

    remaining = max(
        0,
        N_TRIALS
        - completed
    )

    print(
        f"      FCNN seasonal "
        f"trials completos: "
        f"{completed}"
    )

    print(
        f"      FCNN seasonal "
        f"trials restantes: "
        f"{remaining}"
    )

    if remaining > 0:

        study.optimize(
            objective,
            n_trials=(
                remaining
            )
        )

    return (
        study.best_params,
        study.best_value
    )


# =========================================================
# ENTRENAR FCNN FINAL
# =========================================================

def entrenar_fcnn_final(
    train_component,
    params,
    exog_train=None
):

    X, y = crear_ventanas(
        train_component,
        WINDOW,
        exog=(
            exog_train
        )
    )

    val_size = (
        TEST_HORIZON
    )

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        scaler_x.fit_transform(
            X_train
        )
    )

    X_val_s = (
        scaler_x.transform(
            X_val
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    params = (
        params.copy()
    )

    params[
        "units_2"
    ] = (
        params.get(
            "units_2",
            0
        )
    )

    model = (
        construir_fcnn(
            params,
            input_dim=(
                X_train_s.shape[
                    1
                ]
            )
        )
    )

    early = (
        EarlyStopping(

            monitor="val_loss",

            patience=10,

            restore_best_weights=True
        )
    )

    model.fit(

        X_train_s,

        y_train_s,

        validation_data=(
            X_val_s,
            y_val_s
        ),

        epochs=EPOCHS,

        batch_size=(
            params[
                "batch_size"
            ]
        ),

        verbose=0,

        callbacks=[
            early
        ]
    )

    del (
        X,
        y,
        X_train,
        y_train,
        X_val,
        y_val,
        X_train_s,
        X_val_s,
        y_train_s,
        y_val_s
    )

    gc.collect()

    return (
        model,
        scaler_x,
        scaler_y
    )


# =========================================================
# FORECAST RECURSIVO FCNN
# =========================================================

def forecast_recursivo_fcnn(
    model,
    train_component,
    horizon,
    scaler_x,
    scaler_y,
    exog_hist=None,
    exog_future=None
):

    historial = list(
        np.asarray(
            train_component,
            dtype=float
        )
    )

    preds = []

    if (
        exog_hist
        is not None
    ):

        exog_total = (
            pd.concat(
                [
                    exog_hist
                    .reset_index(
                        drop=True
                    ),

                    exog_future
                    .reset_index(
                        drop=True
                    )
                ],

                ignore_index=True
            )
        )

        exog_values = (
            np.asarray(
                exog_total,
                dtype=float
            )
        )

    else:

        exog_values = None

    for step in range(
        horizon
    ):

        y_window = (
            np.asarray(
                historial[
                    -WINDOW:
                ],
                dtype=float
            )
        )

        if (
            exog_values
            is not None
        ):

            end_idx = (
                len(
                    train_component
                )
                + step
            )

            exog_window = (
                exog_values[
                    end_idx-WINDOW:
                    end_idx
                ]
                .reshape(-1)
            )

            exog_actual = (
                exog_values[
                    end_idx
                ]
                .reshape(-1)
            )

            x = (
                np.concatenate(
                    [
                        y_window,
                        exog_window,
                        exog_actual
                    ]
                )
                .reshape(
                    1,
                    -1
                )
            )

        else:

            x = (
                y_window.reshape(
                    1,
                    -1
                )
            )

        x_s = (
            scaler_x.transform(
                x
            )
        )

        pred_s = (
            model.predict(
                x_s,
                verbose=0
            )
        )

        pred = (
            scaler_y
            .inverse_transform(
                pred_s.reshape(
                    -1,
                    1
                )
            )[0, 0]
        )

        preds.append(
            pred
        )

        historial.append(
            pred
        )

    return np.asarray(
        preds
    )


# =========================================================
# AR RESIDUO
# =========================================================

def seleccionar_ar_por_aic(
    y,
    max_lag=168
):

    y = (
        pd.Series(
            y
        )
        .astype(float)
        .dropna()
        .to_numpy()
    )

    resultados = []

    mejor_modelo = None
    mejor_lag = None
    mejor_aic = np.inf

    for lag in range(
        1,
        max_lag + 1
    ):

        try:

            model = AutoReg(
                y,
                lags=lag,
                trend="c",
                old_names=False
            )

            fitted = (
                model.fit()
            )

            resultados.append({
                "lag":
                    lag,

                "AIC":
                    fitted.aic,

                "BIC":
                    fitted.bic
            })

            if (
                np.isfinite(
                    fitted.aic
                )
                and fitted.aic
                < mejor_aic
            ):

                mejor_aic = (
                    fitted.aic
                )

                mejor_lag = (
                    lag
                )

                mejor_modelo = (
                    fitted
                )

        except Exception as e:

            resultados.append({

                "lag":
                    lag,

                "AIC":
                    np.nan,

                "BIC":
                    np.nan,

                "error":
                    str(e)[:150]
            })

    if (
        mejor_modelo
        is None
    ):

        raise RuntimeError(
            "No se pudo ajustar "
            "ningun AR valido."
        )

    return (
        mejor_modelo,
        mejor_lag,
        pd.DataFrame(
            resultados
        )
    )


def forecast_ar_resid(
    resid,
    horizon
):

    (
        modelo,
        lag_optimo,
        df_lags
    ) = (
        seleccionar_ar_por_aic(
            resid,
            max_lag=(
                MAX_LAG_AR
            )
        )
    )

    pred = (
        modelo.predict(

            start=len(
                resid
            ),

            end=(
                len(
                    resid
                )
                + horizon
                - 1
            ),

            dynamic=False
        )
    )

    return (
        np.asarray(
            pred
        ),
        modelo,
        lag_optimo,
        df_lags
    )


# =========================================================
# STL
# =========================================================

def descomponer_stl(
    train
):

    stl = STL(

        pd.Series(
            train
        )
        .astype(float),

        period=(
            STL_PERIOD
        ),

        robust=True
    )

    res = stl.fit()

    return (
        np.asarray(
            res.trend
        ),

        np.asarray(
            res.seasonal
        ),

        np.asarray(
            res.resid
        ),

        res
    )


# =========================================================
# GUARDADO
# =========================================================

def guardar_resultados(
    nombre_serie,
    serie,
    fechas,
    fechas_test,
    pred_final,
    trend_pred,
    seasonal_pred,
    resid_pred,
    metricas,
    params_trend,
    params_seasonal,
    lag_resid,
    df_lags_resid,
    best_mae_trend,
    best_mae_seasonal
):

    serie_dir = os.path.join(
        INCREMENTAL_DIR,
        nombre_serie
    )

    os.makedirs(
        serie_dir,
        exist_ok=True
    )

    nombre_modelo = (
        "ENSEMBLE_STL_"
        "LSTMtrend_"
        "FCNNseason_"
        "ARresid_"
        "EXOG_ALL_Lag168"
    )

    df_series = (
        pd.concat(
            [

                # -----------------------------------------
                # REAL
                # -----------------------------------------

                pd.DataFrame({

                    "serie":
                        nombre_serie,

                    "fecha":
                        pd.to_datetime(
                            fechas
                        ),

                    "tipo":
                        "real",

                    "subset":
                        "completo",

                    "modelo":
                        "real",

                    "valor":
                        serie
                }),

                # -----------------------------------------
                # FINAL
                # -----------------------------------------

                pd.DataFrame({

                    "serie":
                        nombre_serie,

                    "fecha":
                        pd.to_datetime(
                            fechas_test
                        ),

                    "tipo":
                        "prediccion",

                    "subset":
                        "test",

                    "modelo":
                        nombre_modelo,

                    "valor":
                        pred_final
                }),

                # -----------------------------------------
                # TREND
                # -----------------------------------------

                pd.DataFrame({

                    "serie":
                        nombre_serie,

                    "fecha":
                        pd.to_datetime(
                            fechas_test
                        ),

                    "tipo":
                        "componente_pred",

                    "subset":
                        "test",

                    "modelo":
                        "LSTM_trend_EXOG_ALL_Lag168",

                    "valor":
                        trend_pred
                }),

                # -----------------------------------------
                # SEASONAL
                # -----------------------------------------

                pd.DataFrame({

                    "serie":
                        nombre_serie,

                    "fecha":
                        pd.to_datetime(
                            fechas_test
                        ),

                    "tipo":
                        "componente_pred",

                    "subset":
                        "test",

                    "modelo":
                        "FCNN_seasonal_EXOG_ALL_Lag168",

                    "valor":
                        seasonal_pred
                }),

                # -----------------------------------------
                # RESID
                # -----------------------------------------

                pd.DataFrame({

                    "serie":
                        nombre_serie,

                    "fecha":
                        pd.to_datetime(
                            fechas_test
                        ),

                    "tipo":
                        "componente_pred",

                    "subset":
                        "test",

                    "modelo":
                        "AR_resid",

                    "valor":
                        resid_pred
                }),
            ],

            ignore_index=True
        )
    )

    df_metricas = pd.DataFrame(
        [
            {
                "serie":
                    nombre_serie,

                "modelo":
                    nombre_modelo,

                "MAE":
                    metricas[
                        "MAE"
                    ],

                "RMSE":
                    metricas[
                        "RMSE"
                    ],

                "MAPE":
                    metricas[
                        "MAPE"
                    ],

                "sMAPE":
                    metricas[
                        "sMAPE"
                    ],

                "trend_model":
                    "LSTM_EXOG",

                "seasonal_model":
                    "FCNN_EXOG",

                "resid_model":
                    "AR_AIC",

                "exogenas":
                    ",".join(
                        EXOG_NAMES
                    ),

                "lag_electricas":
                    ELECTRIC_LAG,

                "resid_lag_optimo":
                    lag_resid,

                "trend_best_val_MAE":
                    best_mae_trend,

                "seasonal_best_val_MAE":
                    best_mae_seasonal,

                "trend_params":
                    json.dumps(
                        params_trend
                    ),

                "seasonal_params":
                    json.dumps(
                        params_seasonal
                    ),

                "train_horas":
                    TRAIN_LAST_HOURS,

                "test_horas":
                    TEST_HORIZON,

                "window":
                    WINDOW,

                "trials":
                    N_TRIALS
            }
        ]
    )

    path_series = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_ENSEMBLE_componentes_EXOG_series.csv"
    )

    path_metricas = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_ENSEMBLE_componentes_EXOG_metricas.csv"
    )

    path_lags = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_AR_resid_lags_AIC.csv"
    )

    df_series.to_csv(
        path_series,
        index=False,
        encoding="utf-8-sig"
    )

    df_metricas.to_csv(
        path_metricas,
        index=False,
        encoding="utf-8-sig"
    )

    df_lags_resid.to_csv(
        path_lags,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Guardado: "
        f"{serie_dir}"
    )

    return (
        df_series,
        df_metricas
    )


# =========================================================
# EVALUAR REGION
# =========================================================

def evaluar_region(
    region,
    archivo,
    exog_region
):

    path = os.path.join(
        DATA_DIR,
        archivo
    )

    if not os.path.exists(
        path
    ):

        print(
            f"AVISO: "
            f"No encontre {path}"
        )

        return (
            None,
            None
        )

    df = pd.read_csv(
        path
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    if (
        COL_DEMANDA
        not in df.columns
    ):

        print(
            f"AVISO: "
            f"No existe "
            f"{COL_DEMANDA} "
            f"en {region}"
        )

        return (
            None,
            None
        )

    nombre_serie = (
        f"{region}_DEMANDA"
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Evaluando ensemble "
        f"por componentes EXOG: "
        f"{nombre_serie}"
    )

    print(
        "=" * 80
    )

    # -----------------------------------------------------
    # DEMANDA
    # -----------------------------------------------------

    serie, fechas = (
        extraer_serie_horaria(
            df,
            COL_DEMANDA
        )
    )

    # -----------------------------------------------------
    # EXOGENAS
    # -----------------------------------------------------

    X_exog = (
        alinear_exogenas_con_fechas(
            fechas,
            exog_region
        )
    )

    requeridas = (
        TRAIN_LAST_HOURS
        + TEST_HORIZON
    )

    if len(
        serie
    ) < requeridas:

        print(
            f"AVISO: "
            f"Serie insuficiente: "
            f"{nombre_serie}"
        )

        return (
            None,
            None
        )

    # -----------------------------------------------------
    # PERIODO RECIENTE
    # -----------------------------------------------------

    serie_reciente = (
        serie[
            -requeridas:
        ]
    )

    fechas_recientes = (
        fechas[
            -requeridas:
        ]
    )

    X_reciente = (
        X_exog
        .iloc[
            -requeridas:
        ]
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # SPLIT
    # -----------------------------------------------------

    train = (
        serie_reciente[
            :-TEST_HORIZON
        ]
    )

    test = (
        serie_reciente[
            -TEST_HORIZON:
        ]
    )

    fechas_test = (
        fechas_recientes[
            -TEST_HORIZON:
        ]
    )

    X_train = (
        X_reciente
        .iloc[
            :-TEST_HORIZON
        ]
        .reset_index(
            drop=True
        )
    )

    X_test = (
        X_reciente
        .iloc[
            -TEST_HORIZON:
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        f"Train: "
        f"{len(train):,}"
    )

    print(
        f"Test:  "
        f"{len(test):,}"
    )

    print(
        f"Exog train: "
        f"{X_train.shape}"
    )

    print(
        f"Exog test:  "
        f"{X_test.shape}"
    )

    print(
        "\nDefinicion exogenas:"
    )

    for col in (
        EXOG_NAMES
    ):

        if col in [
            "Temperaturas",
            "IGAE"
        ]:

            print(
                f"   {col}: "
                "contemporanea"
            )

        else:

            print(
                f"   {col}: "
                f"lag "
                f"{ELECTRIC_LAG}h"
            )

    # =====================================================
    # STL
    # =====================================================

    (
        trend,
        seasonal,
        resid,
        stl_res
    ) = (
        descomponer_stl(
            train
        )
    )

    del stl_res
    gc.collect()


    # =====================================================
    # LSTM TENDENCIA + EXOG
    # =====================================================

    print(
        "\n   Optuna LSTM "
        "para tendencia "
        "con exogenas..."
    )

    (
        params_trend,
        best_mae_trend
    ) = (
        tunear_lstm(

            trend,

            nombre_serie,

            exog_train=(
                X_train
            )
        )
    )

    print(
        f"      Mejor MAE trend "
        f"validacion: "
        f"{best_mae_trend:.4f}"
    )

    print(
        f"      Params trend: "
        f"{params_trend}"
    )

    (
        lstm_trend,
        sx_trend,
        sy_trend
    ) = (
        entrenar_lstm_final(

            trend,

            params_trend,

            exog_train=(
                X_train
            )
        )
    )

    trend_pred = (
        forecast_recursivo_lstm(

            lstm_trend,

            trend,

            horizon=len(
                test
            ),

            scaler_x=(
                sx_trend
            ),

            scaler_y=(
                sy_trend
            ),

            exog_hist=(
                X_train
            ),

            exog_future=(
                X_test
            )
        )
    )

    del (
        lstm_trend,
        sx_trend,
        sy_trend
    )

    tf.keras.backend.clear_session()
    gc.collect()


    # =====================================================
    # FCNN ESTACIONALIDAD + EXOG
    # =====================================================

    print(
        "\n   Optuna FCNN "
        "para estacionalidad "
        "con exogenas..."
    )

    (
        params_seasonal,
        best_mae_seasonal
    ) = (
        tunear_fcnn(

            seasonal,

            nombre_serie,

            exog_train=(
                X_train
            )
        )
    )

    print(
        f"      Mejor MAE seasonal "
        f"validacion: "
        f"{best_mae_seasonal:.4f}"
    )

    print(
        f"      Params seasonal: "
        f"{params_seasonal}"
    )

    (
        fcnn_season,
        sx_season,
        sy_season
    ) = (
        entrenar_fcnn_final(

            seasonal,

            params_seasonal,

            exog_train=(
                X_train
            )
        )
    )

    seasonal_pred = (
        forecast_recursivo_fcnn(

            fcnn_season,

            seasonal,

            horizon=len(
                test
            ),

            scaler_x=(
                sx_season
            ),

            scaler_y=(
                sy_season
            ),

            exog_hist=(
                X_train
            ),

            exog_future=(
                X_test
            )
        )
    )

    del (
        fcnn_season,
        sx_season,
        sy_season
    )

    tf.keras.backend.clear_session()
    gc.collect()


    # =====================================================
    # AR RESIDUO
    # =====================================================

    print(
        "\n   Ajustando AR "
        "sobre residuo por AIC..."
    )

    (
        resid_pred,
        ar_resid,
        lag_resid,
        df_lags_resid
    ) = (
        forecast_ar_resid(

            resid,

            horizon=len(
                test
            )
        )
    )

    del ar_resid
    gc.collect()

    print(
        f"      AR resid "
        f"lag optimo: "
        f"{lag_resid}"
    )


    # =====================================================
    # RECONSTRUCCION FINAL
    # =====================================================

    pred_final = (
        trend_pred
        + seasonal_pred
        + resid_pred
    )

    metricas = (
        calcular_metricas(
            test,
            pred_final
        )
    )

    print(
        f"\n   ENSEMBLE EXOG final: "
        f"MAPE="
        f"{metricas['MAPE']:.2f}% | "
        f"sMAPE="
        f"{metricas['sMAPE']:.2f}% | "
        f"MAE="
        f"{metricas['MAE']:.2f} | "
        f"RMSE="
        f"{metricas['RMSE']:.2f}"
    )

    # =====================================================
    # GUARDAR
    # =====================================================

    (
        df_series,
        df_metricas
    ) = (
        guardar_resultados(

            nombre_serie=(
                nombre_serie
            ),

            serie=(
                serie
            ),

            fechas=(
                fechas
            ),

            fechas_test=(
                fechas_test
            ),

            pred_final=(
                pred_final
            ),

            trend_pred=(
                trend_pred
            ),

            seasonal_pred=(
                seasonal_pred
            ),

            resid_pred=(
                resid_pred
            ),

            metricas=(
                metricas
            ),

            params_trend=(
                params_trend
            ),

            params_seasonal=(
                params_seasonal
            ),

            lag_resid=(
                lag_resid
            ),

            df_lags_resid=(
                df_lags_resid
            ),

            best_mae_trend=(
                best_mae_trend
            ),

            best_mae_seasonal=(
                best_mae_seasonal
            )
        )
    )

    tf.keras.backend.clear_session()
    gc.collect()

    return (
        df_series,
        df_metricas
    )


# =========================================================
# EJECUTAR TODAS
# =========================================================

todas_series = []
todas_metricas = []

print(
    "=" * 80
)

print(
    "ENSEMBLE STL + LSTM + FCNN + AR "
    "CON EXOGENAS GENERALIZADAS"
)

print(
    "=" * 80
)

print(
    f"Train: "
    f"{TRAIN_LAST_HOURS} h"
)

print(
    f"Test: "
    f"{TEST_HORIZON} h"
)

print(
    f"Window: "
    f"{WINDOW} h"
)

print(
    "\nExogenas activas:"
)

for exog in (
    EXOG_NAMES
):

    print(
        f"   - {exog}"
    )


# =========================================================
# LOOP REGIONES
# =========================================================

for (
    region,
    archivo
) in (
    ARCHIVOS_REGIONES.items()
):

    try:

        # -------------------------------------------------
        # EXOGENAS ESPECIFICAS DE LA REGION
        # -------------------------------------------------

        exog_region = (
            construir_matriz_exogena_region(
                region
            )
        )

        (
            df_series,
            df_metricas
        ) = (
            evaluar_region(

                region=(
                    region
                ),

                archivo=(
                    archivo
                ),

                exog_region=(
                    exog_region
                )
            )
        )

        if (
            df_series
            is not None
        ):

            todas_series.append(
                df_series
            )

        if (
            df_metricas
            is not None
        ):

            todas_metricas.append(
                df_metricas
            )

    except Exception as e:

        print(
            f"Error general "
            f"en {region}: "
            f"{type(e).__name__}: "
            f"{e}"
        )

        continue


# =========================================================
# GUARDAR GLOBALES
# =========================================================

if len(
    todas_series
) > 0:

    df_series_global = (
        pd.concat(
            todas_series,
            ignore_index=True
        )
    )

    path_series_global = (
        os.path.join(

            OUTPUT_DIR,

            "ENSEMBLE_componentes_"
            "EXOG_ALL_Lag168_"
            "series_global.csv"
        )
    )

    df_series_global.to_csv(

        path_series_global,

        index=False,

        encoding="utf-8-sig"
    )

    print(
        "\nSeries globales guardadas:"
    )

    print(
        path_series_global
    )


if len(
    todas_metricas
) > 0:

    df_metricas_global = (
        pd.concat(
            todas_metricas,
            ignore_index=True
        )
    )

    path_metricas_global = (
        os.path.join(

            OUTPUT_DIR,

            "ENSEMBLE_componentes_"
            "EXOG_ALL_Lag168_"
            "metricas_global.csv"
        )
    )

    df_metricas_global.to_csv(

        path_metricas_global,

        index=False,

        encoding="utf-8-sig"
    )

    print(
        "\nMetricas globales guardadas:"
    )

    print(
        path_metricas_global
    )

    display(
        df_metricas_global
        .sort_values(
            "MAPE"
        )
    )

    print(
        "\nPromedio MAPE ensemble:"
    )

    display(
        df_metricas_global
        .groupby(
            "modelo"
        )[
            "MAPE"
        ]
        .agg(
            [
                "mean",
                "std"
            ]
        )
    )

#SARIMAX

In [ ]:
# =========================================================
# PIPELINE SARIMAX DEMANDA
#
# TRAIN:
#   2 meses = 1440 horas
#
# TEST:
#   1 semana = 168 horas
#
# EXOGENAS:
#   Temperaturas
#   IGAE
#   Generacion
#   Importacion
#   Exportacion
#
# Temperatura e IGAE:
#   usan valores disponibles del horizonte.
#
# Generacion / Importacion / Exportacion:
#   NO usan valores reales del test.
#   Se estiman con:
#
#       promedio(t-168, t-336)
#
# SARIMAX:
#   (1,1,1)(1,0,1,168)
# =========================================================


# =========================================================
# IMPORTS
# =========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import gc
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = "/content"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Tesis/Resultados/"
    "SARIMAX_exogenas_demanda_2m_1w"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

INCREMENTAL_DIR = os.path.join(
    OUTPUT_DIR,
    "por_serie"
)

os.makedirs(
    INCREMENTAL_DIR,
    exist_ok=True
)


# =========================================================
# REGIONES
# =========================================================

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}


# =========================================================
# COLUMNAS
# =========================================================

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = (
    "Estimacion de Demanda por Balance (MWh)"
)


# =========================================================
# TRAIN / TEST
# =========================================================

# 2 meses de 30 dias
TRAIN_LAST_HOURS = 24 * 30 * 2       # 1440 h

# 1 semana
TEST_HORIZON = 24 * 7                # 168 h

# Para construir GEN/IMP/EXP futuras
LAG_SEMANA_1 = 24 * 7                # 168 h
LAG_SEMANA_2 = 24 * 14               # 336 h


# =========================================================
# SARIMAX
#
# MISMA ARQUITECTURA
# =========================================================

SARIMA_ORDER = (
    1,
    1,
    1
)

SARIMA_SEASONAL_ORDER = (
    1,
    0,
    1,
    168
)


# =========================================================
# EXOGENAS
#
# Puedes comentar cualquiera.
# Por defecto entran TODAS.
# =========================================================

EXOG_NAMES = [
    "Temperaturas",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]


# =========================================================
# CONOCIDAS EN EL HORIZONTE
# =========================================================

EXOG_CONOCIDAS_FUTURO = [
    "Temperaturas",
    "IGAE",
]


# =========================================================
# NO CONOCIDAS EN EL HORIZONTE
# =========================================================

EXOG_NO_CONOCIDAS_FUTURO = [
    "Generacion",
    "Importacion",
    "Exportacion",
]


# =========================================================
# METRICAS
# =========================================================

def mape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                / y_true[mask]
            )
        )
        * 100
    )


def smape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denom = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (denom != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                y_true[mask]
                - y_pred[mask]
            )
            / denom[mask]
        )
        * 100
    )


def calcular_metricas(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    n = min(
        len(y_true),
        len(y_pred)
    )

    y_true = y_true[:n]
    y_pred = y_pred[:n]

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:

        return {
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE": np.nan,
            "sMAPE": np.nan
        }

    return {

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAPE":
            mape(
                y_true,
                y_pred
            ),

        "sMAPE":
            smape(
                y_true,
                y_pred
            )
    }


# =========================================================
# LEER DEMANDA
# =========================================================

def extraer_serie_horaria(
    df,
    columna
):

    aux = df[
        [
            COL_FECHA,
            COL_HORA,
            columna
        ]
    ].copy()

    aux[
        COL_FECHA
    ] = pd.to_datetime(
        aux[COL_FECHA],
        errors="coerce"
    )

    aux[
        COL_HORA
    ] = pd.to_numeric(
        aux[COL_HORA],
        errors="coerce"
    )

    aux[
        columna
    ] = pd.to_numeric(
        aux[columna],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            COL_FECHA,
            COL_HORA,
            columna
        ]
    )

    # Fuente 1-24
    aux[
        "hora_0_23"
    ] = (
        aux[
            COL_HORA
        ].astype(int)
        - 1
    )

    aux[
        "datetime"
    ] = (
        aux[
            COL_FECHA
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux
        .sort_values(
            "datetime"
        )
        .drop_duplicates(
            "datetime",
            keep="last"
        )
    )

    return (

        aux[
            columna
        ].to_numpy(
            dtype=float
        ),

        aux[
            "datetime"
        ].to_numpy()
    )


# =========================================================
# NORMALIZAR EXOGENA
# =========================================================

def preparar_exogena_horaria(
    df,
    nombre
):

    aux = df.copy()

    aux.columns = (
        aux.columns
        .astype(str)
        .str.strip()
    )

    col_hora = (
        "hora"
        if "hora" in aux.columns
        else "Hora"
    )

    if "fecha" not in aux.columns:

        raise ValueError(
            f"{nombre} no tiene "
            "columna fecha"
        )

    if col_hora not in aux.columns:

        raise ValueError(
            f"{nombre} no tiene "
            "columna hora/Hora"
        )

    if "valor" not in aux.columns:

        raise ValueError(
            f"{nombre} no tiene "
            "columna valor"
        )

    aux[
        "fecha"
    ] = pd.to_datetime(
        aux["fecha"],
        errors="coerce"
    )

    aux[
        col_hora
    ] = pd.to_numeric(
        aux[col_hora],
        errors="coerce"
    )

    aux[
        "valor"
    ] = pd.to_numeric(
        aux["valor"],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            "fecha",
            col_hora,
            "valor"
        ]
    )

    hora = aux[
        col_hora
    ]

    # Acepta 1-24 o 0-23
    if (
        hora.min() >= 1
        and hora.max() <= 24
    ):

        aux[
            "hora_0_23"
        ] = (
            hora.astype(int)
            - 1
        )

    else:

        aux[
            "hora_0_23"
        ] = hora.astype(int)

    aux[
        "datetime"
    ] = (
        aux[
            "fecha"
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux[
            [
                "datetime",
                "valor"
            ]
        ]
        .rename(
            columns={
                "valor":
                    nombre
            }
        )
        .sort_values(
            "datetime"
        )
        .drop_duplicates(
            "datetime",
            keep="last"
        )
    )

    return aux


# =========================================================
# MATRIZ EXOGENA POR REGION
# =========================================================

def construir_matriz_exogena_region(
    region
):

    dfs = []

    # -----------------------------------------------------
    # TEMPERATURA
    # -----------------------------------------------------

    if (
        "Temperaturas"
        in EXOG_NAMES
    ):

        if (
            "Temperaturas_H"
            not in globals()
        ):

            raise ValueError(
                "No existe "
                "Temperaturas_H"
            )

        tmp = (
            preparar_exogena_horaria(
                Temperaturas_H,
                "Temperaturas"
            )
        )

        dfs.append(
            tmp
        )

    # -----------------------------------------------------
    # IGAE
    # -----------------------------------------------------

    if (
        "IGAE"
        in EXOG_NAMES
    ):

        if (
            "IGAE_H"
            not in globals()
        ):

            raise ValueError(
                "No existe IGAE_H"
            )

        tmp = (
            preparar_exogena_horaria(
                IGAE_H,
                "IGAE"
            )
        )

        dfs.append(
            tmp
        )

    # -----------------------------------------------------
    # GEN / IMP / EXP
    # -----------------------------------------------------

    REGION_EXOG_MAP = {

        "Generacion":
            f"{region}_GEN",

        "Importacion":
            f"{region}_IMP",

        "Exportacion":
            f"{region}_EXP",
    }

    for (
        variable,
        nombre_serie
    ) in REGION_EXOG_MAP.items():

        if (
            variable
            not in EXOG_NAMES
        ):
            continue

        if (
            "series"
            not in globals()
        ):

            raise ValueError(
                "No existe el diccionario "
                "global 'series'."
            )

        if (
            nombre_serie
            not in series
        ):

            raise ValueError(
                f"No existe "
                f"{nombre_serie} "
                "dentro de series."
            )

        tmp = (
            preparar_exogena_horaria(
                series[
                    nombre_serie
                ],
                variable
            )
        )

        dfs.append(
            tmp
        )

    if len(dfs) == 0:

        raise ValueError(
            "No hay ninguna "
            "exogena activa."
        )

    # -----------------------------------------------------
    # COMBINAR
    # -----------------------------------------------------

    exog = dfs[0]

    for df_next in (
        dfs[1:]
    ):

        exog = exog.merge(
            df_next,
            on="datetime",
            how="outer"
        )

    exog = (
        exog
        .sort_values(
            "datetime"
        )
        .reset_index(
            drop=True
        )
    )

    exog[
        EXOG_NAMES
    ] = (
        exog[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    print(
        f"\nExogenas {region}:"
    )

    print(
        ", ".join(
            EXOG_NAMES
        )
    )

    print(
        f"Filas disponibles: "
        f"{len(exog):,}"
    )

    return exog


# =========================================================
# ALINEAR EXOGENAS CON DEMANDA
# =========================================================

def alinear_exogenas_con_fechas(
    fechas,
    exog_global
):

    base = pd.DataFrame({

        "datetime":
            pd.to_datetime(
                fechas
            )
    })

    X = base.merge(

        exog_global[
            ["datetime"]
            + EXOG_NAMES
        ],

        on="datetime",

        how="left"
    )

    X[
        EXOG_NAMES
    ] = (
        X[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    if (
        X[
            EXOG_NAMES
        ]
        .isna()
        .any()
        .any()
    ):

        raise ValueError(
            "Hay valores faltantes "
            "en las exogenas."
        )

    return (
        X[
            EXOG_NAMES
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# EXOGENAS FUTURAS
# =========================================================

def construir_exogenas_futuras(
    X_completo,
    train_end,
    horizon
):

    """
    Temperatura e IGAE:
        usan el valor correspondiente
        al horizonte.

    Generacion / Importacion / Exportacion:
        NO usan el valor real futuro.

        Para cada hora:

        promedio(
            valor t-168,
            valor t-336
        )
    """

    # Conservamos aquí Temp/IGAE
    # correspondientes al horizonte.

    X_future = (
        X_completo
        .iloc[
            train_end:
            train_end + horizon
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    for variable in (
        EXOG_NO_CONOCIDAS_FUTURO
    ):

        if (
            variable
            not in EXOG_NAMES
        ):
            continue

        estimados = []

        for h in range(
            horizon
        ):

            indice_futuro = (
                train_end
                + h
            )

            idx_1 = (
                indice_futuro
                - LAG_SEMANA_1
            )

            idx_2 = (
                indice_futuro
                - LAG_SEMANA_2
            )

            if idx_2 < 0:

                raise ValueError(
                    f"No existen "
                    f"2 semanas anteriores "
                    f"para estimar "
                    f"{variable}"
                )

            valor_1 = float(
                X_completo
                .iloc[
                    idx_1
                ][variable]
            )

            valor_2 = float(
                X_completo
                .iloc[
                    idx_2
                ][variable]
            )

            estimado = (
                valor_1
                + valor_2
            ) / 2

            estimados.append(
                estimado
            )

        X_future[
            variable
        ] = estimados

    return (
        X_future[
            EXOG_NAMES
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# SARIMAX
# =========================================================

def entrenar_predecir_sarimax(
    train,
    horizon,
    order,
    seasonal_order=(
        0,
        0,
        0,
        0
    ),
    X_train=None,
    X_test=None
):

    y_train = np.asarray(
        train,
        dtype=float
    )

    X_train_arr = (
        np.asarray(
            X_train,
            dtype=float
        )
        if X_train is not None
        else None
    )

    X_test_arr = (
        np.asarray(
            X_test,
            dtype=float
        )
        if X_test is not None
        else None
    )

    model = SARIMAX(

        endog=y_train,

        exog=X_train_arr,

        order=order,

        seasonal_order=(
            seasonal_order
        ),

        enforce_stationarity=False,

        enforce_invertibility=False,

        simple_differencing=False
    )

    try:

        fitted = model.fit(
            disp=False,
            maxiter=100,
            low_memory=True
        )

    except TypeError:

        fitted = model.fit(
            disp=False,
            maxiter=100
        )

    pred = fitted.predict(

        start=len(
            y_train
        ),

        end=(
            len(y_train)
            + horizon
            - 1
        ),

        exog=X_test_arr
    )

    aic = fitted.aic
    bic = fitted.bic

    pred = np.asarray(
        pred
    )

    # -----------------------------------------------------
    # CONVERGENCIA
    # -----------------------------------------------------

    try:

        converged = (
            fitted
            .mle_retvals
            .get(
                "converged",
                None
            )
        )

        iterations = (
            fitted
            .mle_retvals
            .get(
                "iterations",
                None
            )
        )

        print(
            f"      Converged: "
            f"{converged}"
        )

        print(
            f"      Iterations: "
            f"{iterations}"
        )

    except Exception:
        pass

    del fitted
    del model

    gc.collect()

    return (
        pred,
        aic,
        bic
    )


# =========================================================
# GUARDAR RESULTADOS
# =========================================================

def guardar_resultados(
    nombre_serie,
    serie,
    fechas,
    fechas_test,
    resultados_modelos
):

    serie_dir = os.path.join(
        INCREMENTAL_DIR,
        nombre_serie
    )

    os.makedirs(
        serie_dir,
        exist_ok=True
    )

    # -----------------------------------------------------
    # SERIE REAL COMPLETA
    # -----------------------------------------------------

    series_dfs = [

        pd.DataFrame({

            "serie":
                nombre_serie,

            "fecha":
                pd.to_datetime(
                    fechas
                ),

            "tipo":
                "real",

            "subset":
                "completo",

            "modelo":
                "real",

            "valor":
                serie
        })
    ]

    metricas_rows = []

    # -----------------------------------------------------
    # PREDICCIONES
    # -----------------------------------------------------

    for res in (
        resultados_modelos
    ):

        modelo = res[
            "modelo"
        ]

        pred = res[
            "pred"
        ]

        metricas = res[
            "metricas"
        ]

        series_dfs.append(

            pd.DataFrame({

                "serie":
                    nombre_serie,

                "fecha":
                    pd.to_datetime(
                        fechas_test
                    ),

                "tipo":
                    "prediccion",

                "subset":
                    "test",

                "modelo":
                    modelo,

                "valor":
                    pred
            })
        )

        metricas_rows.append({

            "serie":
                nombre_serie,

            "modelo":
                modelo,

            "order":
                str(
                    res[
                        "order"
                    ]
                ),

            "seasonal_order":
                str(
                    res[
                        "seasonal_order"
                    ]
                ),

            "AIC":
                res.get(
                    "AIC",
                    np.nan
                ),

            "BIC":
                res.get(
                    "BIC",
                    np.nan
                ),

            "MAPE":
                metricas[
                    "MAPE"
                ],

            "sMAPE":
                metricas[
                    "sMAPE"
                ],

            "MAE":
                metricas[
                    "MAE"
                ],

            "RMSE":
                metricas[
                    "RMSE"
                ],

            "train_horas":
                TRAIN_LAST_HOURS,

            "test_horas":
                TEST_HORIZON,

            "exogenas":
                str(
                    EXOG_NAMES
                ),

            "metodo_exog_futuras":
                (
                    "Temp/IGAE horizonte; "
                    "GEN/IMP/EXP promedio "
                    "t-168,t-336"
                )
        })

    df_series = pd.concat(
        series_dfs,
        ignore_index=True
    )

    df_metricas = pd.DataFrame(
        metricas_rows
    )

    path_series = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_SARIMAX_series.csv"
    )

    path_metricas = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_SARIMAX_metricas.csv"
    )

    df_series.to_csv(
        path_series,
        index=False,
        encoding="utf-8-sig"
    )

    df_metricas.to_csv(
        path_metricas,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Guardado: "
        f"{serie_dir}"
    )

    return (
        df_series,
        df_metricas
    )


# =========================================================
# EVALUAR SERIE
# =========================================================

def evaluar_serie(
    region,
    df,
    columna,
    etiqueta,
    exog_global
):

    nombre_serie = (
        f"{region}_"
        f"{etiqueta}"
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Evaluando "
        f"{nombre_serie}"
    )

    print(
        "=" * 80
    )

    # -----------------------------------------------------
    # DEMANDA
    # -----------------------------------------------------

    serie, fechas = (
        extraer_serie_horaria(
            df,
            columna
        )
    )

    # -----------------------------------------------------
    # EXOGENAS ALINEADAS
    # -----------------------------------------------------

    X_exog = (
        alinear_exogenas_con_fechas(
            fechas,
            exog_global
        )
    )

    # -----------------------------------------------------
    # TRAIN + TEST
    # -----------------------------------------------------

    requeridas = (
        TRAIN_LAST_HOURS
        + TEST_HORIZON
    )

    if len(
        serie
    ) < requeridas:

        raise ValueError(
            f"{nombre_serie}: "
            "serie insuficiente."
        )

    train_start = (
        len(serie)
        - requeridas
    )

    train_end = (
        len(serie)
        - TEST_HORIZON
    )

    # -----------------------------------------------------
    # DEMANDA
    # -----------------------------------------------------

    train = (
        serie[
            train_start:
            train_end
        ]
    )

    test = (
        serie[
            train_end:
        ]
    )

    fechas_test = (
        fechas[
            train_end:
        ]
    )

    # -----------------------------------------------------
    # EXOGENAS TRAIN
    #
    # Las mismas 1440 horas del train.
    # -----------------------------------------------------

    X_train = (
        X_exog
        .iloc[
            train_start:
            train_end
        ]
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # EXOGENAS TEST
    # -----------------------------------------------------

    X_test = (
        construir_exogenas_futuras(
            X_completo=(
                X_exog
            ),
            train_end=(
                train_end
            ),
            horizon=(
                TEST_HORIZON
            )
        )
    )

    # -----------------------------------------------------
    # DIAGNOSTICO
    # -----------------------------------------------------

    print(
        f"Train demanda: "
        f"{len(train):,}"
    )

    print(
        f"Train exogenas: "
        f"{len(X_train):,}"
    )

    print(
        f"Test demanda: "
        f"{len(test):,}"
    )

    print(
        f"Test exogenas: "
        f"{len(X_test):,}"
    )

    print(
        f"Shape exogenas train: "
        f"{X_train.shape}"
    )

    print(
        f"Shape exogenas test: "
        f"{X_test.shape}"
    )

    print(
        "\nTratamiento "
        "de exogenas:"
    )

    for variable in (
        EXOG_NAMES
    ):

        if (
            variable
            in EXOG_CONOCIDAS_FUTURO
        ):

            print(
                f"   {variable}: "
                "valor del horizonte"
            )

        else:

            print(
                f"   {variable}: "
                "promedio "
                "t-168 y t-336"
            )

    resultados_modelos = []

    # =====================================================
    # SARIMAX(1,1,1)(1,0,1,168)
    # =====================================================

    try:

        print(
            "\n   Ajustando "
            f"SARIMAX"
            f"{SARIMA_ORDER}"
            f"x"
            f"{SARIMA_SEASONAL_ORDER}"
            "..."
        )

        (
            pred_sarimax,
            aic_sarimax,
            bic_sarimax
        ) = (
            entrenar_predecir_sarimax(

                train=train,

                horizon=len(
                    test
                ),

                order=(
                    SARIMA_ORDER
                ),

                seasonal_order=(
                    SARIMA_SEASONAL_ORDER
                ),

                X_train=(
                    X_train
                ),

                X_test=(
                    X_test
                )
            )
        )

        met_sarimax = (
            calcular_metricas(
                test,
                pred_sarimax
            )
        )

        print(
            f"      SARIMAX "
            f"MAPE="
            f"{met_sarimax['MAPE']:.2f}%"
        )

        resultados_modelos.append({

            "modelo":
                (
                    "SARIMAX_1_1_1"
                    "__1_0_1_168"
                    "_EXOG_2M"
                ),

            "order":
                SARIMA_ORDER,

            "seasonal_order":
                SARIMA_SEASONAL_ORDER,

            "pred":
                pred_sarimax,

            "metricas":
                met_sarimax,

            "AIC":
                aic_sarimax,

            "BIC":
                bic_sarimax
        })

    except Exception as e:

        print(
            f"Error SARIMAX "
            f"en {nombre_serie}: "
            f"{type(e).__name__}: "
            f"{e}"
        )

    gc.collect()

    if len(
        resultados_modelos
    ) == 0:

        return (
            None,
            None
        )

    return guardar_resultados(

        nombre_serie=(
            nombre_serie
        ),

        serie=serie,

        fechas=fechas,

        fechas_test=(
            fechas_test
        ),

        resultados_modelos=(
            resultados_modelos
        )
    )


# =========================================================
# EVALUAR REGION
# =========================================================

def evaluar_region(
    region,
    archivo,
    exog_global
):

    path = os.path.join(
        DATA_DIR,
        archivo
    )

    if not os.path.exists(
        path
    ):

        print(
            f"No encontre "
            f"{path}"
        )

        return (
            [],
            []
        )

    df = pd.read_csv(
        path
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    resultados_series = []
    resultados_metricas = []

    if (
        COL_DEMANDA
        not in df.columns
    ):

        print(
            f"No existe columna "
            f"{COL_DEMANDA} "
            f"en {region}"
        )

        return (
            resultados_series,
            resultados_metricas
        )

    try:

        (
            df_series,
            df_metricas
        ) = (
            evaluar_serie(

                region=region,

                df=df,

                columna=(
                    COL_DEMANDA
                ),

                etiqueta="DEMANDA",

                exog_global=(
                    exog_global
                )
            )
        )

        if (
            df_series
            is not None
        ):

            resultados_series.append(
                df_series
            )

        if (
            df_metricas
            is not None
        ):

            resultados_metricas.append(
                df_metricas
            )

    except Exception as e:

        print(
            f"Error general "
            f"en {region}_DEMANDA: "
            f"{type(e).__name__}: "
            f"{e}"
        )

    return (
        resultados_series,
        resultados_metricas
    )


# =========================================================
# EJECUTAR TODAS
# =========================================================

todas_series = []
todas_metricas = []

print(
    "=" * 80
)

print(
    "PIPELINE SARIMAX DEMANDA - 2 MESES"
)

print(
    "=" * 80
)

print(
    f"Train: "
    f"{TRAIN_LAST_HOURS} horas"
)

print(
    f"Horizonte: "
    f"{TEST_HORIZON} horas"
)

print(
    "\nExogenas activas:"
)

for exog in (
    EXOG_NAMES
):

    print(
        f"   - {exog}"
    )


# =========================================================
# LOOP REGIONES
# =========================================================

for (
    region,
    archivo
) in (
    ARCHIVOS_REGIONES.items()
):

    try:

        exog_region = (
            construir_matriz_exogena_region(
                region
            )
        )

        (
            series_region,
            metricas_region
        ) = (
            evaluar_region(

                region=region,

                archivo=archivo,

                exog_global=(
                    exog_region
                )
            )
        )

        todas_series.extend(
            series_region
        )

        todas_metricas.extend(
            metricas_region
        )

    except Exception as e:

        print(
            f"Error en region "
            f"{region}: "
            f"{type(e).__name__}: "
            f"{e}"
        )

        continue


# =========================================================
# GUARDAR GLOBALES
# =========================================================

if len(
    todas_series
) > 0:

    df_series_global = (
        pd.concat(
            todas_series,
            ignore_index=True
        )
    )

    path_series_global = (
        os.path.join(
            OUTPUT_DIR,
            "SARIMAX_series_global.csv"
        )
    )

    df_series_global.to_csv(
        path_series_global,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "\nSeries globales guardadas:"
    )

    print(
        path_series_global
    )


if len(
    todas_metricas
) > 0:

    df_metricas_global = (
        pd.concat(
            todas_metricas,
            ignore_index=True
        )
    )

    path_metricas_global = (
        os.path.join(
            OUTPUT_DIR,
            "SARIMAX_metricas_global.csv"
        )
    )

    df_metricas_global.to_csv(
        path_metricas_global,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "\nMetricas globales guardadas:"
    )

    print(
        path_metricas_global
    )

    print(
        "\nResultados por region:"
    )

    display(
        df_metricas_global
        .sort_values(
            [
                "serie",
                "MAPE"
            ]
        )
    )

    print(
        "\nRanking promedio:"
    )

    display(

        df_metricas_global

        .groupby(
            "modelo"
        )[
            "MAPE"
        ]

        .agg(
            [
                "mean",
                "std"
            ]
        )

        .sort_values(
            "mean"
        )
    )

#Redes multivariadas

In [ ]:
# =========================================================
# PIPELINE FCNN MULTIVARIADA
# + FCNN MULTIVARIADA SOBRE RESIDUOS STL
# SOLO DEMANDA
#
# OBJETIVO:
#   Demanda electrica
#
# EXOGENAS:
#   - Temperaturas contemporanea
#   - IGAE contemporaneo
#   - Generacion lag 168h
#   - Importacion lag 168h
#   - Exportacion lag 168h
#
# TRAIN:
#   3600 horas
#
# TEST:
#   168 horas
#
# WINDOW:
#   168 horas
#
# IMPORTANTE:
#
# Durante entrenamiento se dispone de 3600 h
# de demanda y exogenas.
#
# Cada ejemplo de entrenamiento utiliza una
# ventana movil de 168 h.
#
# Para predecir y[t]:
#
#   y[t-168:t]
#
#   Temperatura[t-168:t] + Temperatura[t]
#   IGAE[t-168:t]        + IGAE[t]
#
#   Generacion_lag168[t-168:t]
#       + Generacion[t-168]
#
#   Importacion_lag168[t-168:t]
#       + Importacion[t-168]
#
#   Exportacion_lag168[t-168:t]
#       + Exportacion[t-168]
#
# No se utilizan GEN / IMP / EXP reales del futuro.
# =========================================================


# =========================================================
# INSTALL
# =========================================================

!pip install -q optuna


# =========================================================
# DRIVE
# =========================================================

from google.colab import drive
drive.mount("/content/drive")


# =========================================================
# IMPORTS
# =========================================================

import os
import gc
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

from statsmodels.tsa.seasonal import STL


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = "/content"

OUTPUT_DIR = (
    "/content/drive/MyDrive/Tesis/Resultados/"
    "FCNN_multivariada_EXOG_electricas_lag168_demanda_5m_1w"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

INCREMENTAL_DIR = os.path.join(
    OUTPUT_DIR,
    "por_serie"
)

os.makedirs(
    INCREMENTAL_DIR,
    exist_ok=True
)


# =========================================================
# REGIONES
# =========================================================

ARCHIVOS_REGIONES = {
    "BCA": "BCA_long.csv",
    "CEN": "CEN_long.csv",
    "NES": "NES_long.csv",
    "NOR": "NOR_long.csv",
    "NTE": "NTE_long.csv",
    "OCC": "OCC_long.csv",
    "ORI": "ORI_long.csv",
    "PEN": "PEN_long.csv",
}


# =========================================================
# COLUMNAS
# =========================================================

COL_FECHA = "fecha"
COL_HORA = "Hora"

COL_DEMANDA = (
    "Estimacion de Demanda por Balance (MWh)"
)


# =========================================================
# TRAIN / TEST
# =========================================================

TRAIN_LAST_HOURS = 24 * 30 * 5   # 3600 h
TEST_HORIZON = 24 * 7            # 168 h

WINDOW = 168
STL_PERIOD = 168

N_TRIALS = 5
EPOCHS = 60

SEED = 42


# =========================================================
# LAG DE EXOGENAS ELECTRICAS
# =========================================================

ELECTRIC_LAG = 168


# =========================================================
# EXOGENAS ACTIVAS
#
# Puedes comentar cualquiera.
#
# OJO:
# Generacion / Importacion / Exportacion ya representan
# internamente las variables desplazadas 168 h.
# =========================================================

EXOG_NAMES = [
    "Temperaturas",
    "IGAE",
    "Generacion_lag168",
    "Importacion_lag168",
    "Exportacion_lag168",
]


# =========================================================
# SEEDS
# =========================================================

np.random.seed(SEED)
tf.random.set_seed(SEED)


# =========================================================
# METRICAS
# =========================================================

def mape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (y_true != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (
                    y_true[mask]
                    - y_pred[mask]
                )
                / y_true[mask]
            )
        )
        * 100
    )


def smape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denom = (
        np.abs(y_true)
        + np.abs(y_pred)
    ) / 2

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & (denom != 0)
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                y_true[mask]
                - y_pred[mask]
            )
            / denom[mask]
        )
        * 100
    )


def calcular_metricas(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    n = min(
        len(y_true),
        len(y_pred)
    )

    y_true = y_true[:n]
    y_pred = y_pred[:n]

    mask = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:

        return {
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE": np.nan,
            "sMAPE": np.nan,
        }

    return {

        "MAE":
            mean_absolute_error(
                y_true,
                y_pred
            ),

        "RMSE":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            ),

        "MAPE":
            mape(
                y_true,
                y_pred
            ),

        "sMAPE":
            smape(
                y_true,
                y_pred
            ),
    }


# =========================================================
# HORAS 0-23
# =========================================================

def convertir_hora_0_23(serie_hora):

    hora = pd.to_numeric(
        serie_hora,
        errors="coerce"
    )

    if hora.dropna().empty:
        return hora

    if (
        hora.min() >= 1
        and hora.max() <= 24
    ):

        return (
            hora.astype(float)
            - 1
        )

    return hora.astype(float)


# =========================================================
# LECTURA DE DEMANDA
# =========================================================

def extraer_serie_horaria(
    df,
    columna
):

    aux = df[
        [
            COL_FECHA,
            COL_HORA,
            columna
        ]
    ].copy()

    aux[
        COL_FECHA
    ] = pd.to_datetime(
        aux[COL_FECHA],
        errors="coerce"
    )

    aux[
        COL_HORA
    ] = pd.to_numeric(
        aux[COL_HORA],
        errors="coerce"
    )

    aux[
        columna
    ] = pd.to_numeric(
        aux[columna],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            COL_FECHA,
            COL_HORA,
            columna
        ]
    )

    aux[
        "hora_0_23"
    ] = convertir_hora_0_23(
        aux[
            COL_HORA
        ]
    )

    aux = aux.dropna(
        subset=[
            "hora_0_23"
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            "hora_0_23"
        ]
        .astype(int)
    )

    aux[
        "datetime"
    ] = (
        aux[
            COL_FECHA
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (
        aux
        .sort_values(
            "datetime"
        )
        .drop_duplicates(
            "datetime",
            keep="last"
        )
    )

    return (

        aux[
            columna
        ].to_numpy(
            dtype=float
        ),

        aux[
            "datetime"
        ].to_numpy()
    )


# =========================================================
# PREPARAR EXOGENA HORARIA
# =========================================================

def preparar_exogena_horaria(
    df,
    nombre
):

    """
    Espera:
        fecha
        hora / Hora
        valor

    Devuelve:
        datetime
        <nombre>
    """

    aux = df.copy()

    aux.columns = (
        aux.columns
        .astype(str)
        .str.strip()
    )

    cols_lower = {
        c.lower(): c
        for c in aux.columns
    }

    if "fecha" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no existe columna fecha"
        )

    if "hora" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no existe columna hora/Hora"
        )

    if "valor" not in cols_lower:

        raise ValueError(
            f"{nombre}: "
            "no existe columna valor"
        )

    fecha_col = cols_lower[
        "fecha"
    ]

    hora_col = cols_lower[
        "hora"
    ]

    valor_col = cols_lower[
        "valor"
    ]

    aux[
        fecha_col
    ] = pd.to_datetime(
        aux[
            fecha_col
        ],
        errors="coerce"
    )

    aux[
        hora_col
    ] = pd.to_numeric(
        aux[
            hora_col
        ],
        errors="coerce"
    )

    aux[
        valor_col
    ] = pd.to_numeric(
        aux[
            valor_col
        ],
        errors="coerce"
    )

    aux = aux.dropna(
        subset=[
            fecha_col,
            hora_col,
            valor_col
        ]
    )

    aux[
        "hora_0_23"
    ] = convertir_hora_0_23(
        aux[
            hora_col
        ]
    )

    aux = aux.dropna(
        subset=[
            "hora_0_23"
        ]
    )

    aux[
        "hora_0_23"
    ] = (
        aux[
            "hora_0_23"
        ]
        .astype(int)
    )

    aux[
        "datetime"
    ] = (
        aux[
            fecha_col
        ]
        + pd.to_timedelta(
            aux[
                "hora_0_23"
            ],
            unit="h"
        )
    )

    aux = (

        aux[
            [
                "datetime",
                valor_col
            ]
        ]

        .rename(
            columns={
                valor_col:
                    nombre
            }
        )

        .groupby(
            "datetime",
            as_index=False
        )

        .mean()

        .sort_values(
            "datetime"
        )
    )

    return aux


# =========================================================
# CREAR EXOGENA ELECTRICA LAG 168
# =========================================================

def preparar_exogena_lag168(
    df,
    nombre_salida
):

    """
    Convierte una serie electrica X_t en:

        X_lag168[t] = X[t-168]

    Se realiza desplazando el timestamp
    168 horas hacia adelante.

    De esta forma la fila correspondiente
    al tiempo t contiene exclusivamente
    informacion conocida en t-168.
    """

    aux = preparar_exogena_horaria(
        df,
        nombre_salida
    )

    aux[
        "datetime"
    ] = (
        aux[
            "datetime"
        ]
        + pd.to_timedelta(
            ELECTRIC_LAG,
            unit="h"
        )
    )

    return aux


# =========================================================
# CONSTRUIR MATRIZ EXOGENA POR REGION
# =========================================================

def construir_matriz_exogena_region(
    region
):

    """
    Temperaturas:
        contemporanea

    IGAE:
        contemporaneo

    Generacion:
        lag 168

    Importacion:
        lag 168

    Exportacion:
        lag 168
    """

    dfs = []

    # -----------------------------------------------------
    # TEMPERATURA
    # -----------------------------------------------------

    if (
        "Temperaturas"
        in EXOG_NAMES
    ):

        if (
            "Temperaturas_H"
            not in globals()
        ):

            raise ValueError(
                "No existe "
                "Temperaturas_H"
            )

        temp = (
            preparar_exogena_horaria(
                Temperaturas_H,
                "Temperaturas"
            )
        )

        dfs.append(
            temp
        )

    # -----------------------------------------------------
    # IGAE
    # -----------------------------------------------------

    if (
        "IGAE"
        in EXOG_NAMES
    ):

        if (
            "IGAE_H"
            not in globals()
        ):

            raise ValueError(
                "No existe IGAE_H"
            )

        igae = (
            preparar_exogena_horaria(
                IGAE_H,
                "IGAE"
            )
        )

        dfs.append(
            igae
        )

    # -----------------------------------------------------
    # SERIES ELECTRICAS REGIONALES
    # -----------------------------------------------------

    if (
        any(
            x in EXOG_NAMES
            for x in [
                "Generacion_lag168",
                "Importacion_lag168",
                "Exportacion_lag168",
            ]
        )
    ):

        if (
            "series"
            not in globals()
        ):

            raise ValueError(
                "No existe el diccionario "
                "global 'series'. "
                "Primero debes construir "
                "las series GEN/IMP/EXP "
                "por region."
            )

    # -----------------------------------------------------
    # GENERACION
    # -----------------------------------------------------

    if (
        "Generacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_GEN"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        gen = (
            preparar_exogena_lag168(
                series[key],
                "Generacion_lag168"
            )
        )

        dfs.append(
            gen
        )

    # -----------------------------------------------------
    # IMPORTACION
    # -----------------------------------------------------

    if (
        "Importacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_IMP"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        imp = (
            preparar_exogena_lag168(
                series[key],
                "Importacion_lag168"
            )
        )

        dfs.append(
            imp
        )

    # -----------------------------------------------------
    # EXPORTACION
    # -----------------------------------------------------

    if (
        "Exportacion_lag168"
        in EXOG_NAMES
    ):

        key = (
            f"{region}_EXP"
        )

        if key not in series:

            raise ValueError(
                f"No existe "
                f"series['{key}']"
            )

        exp = (
            preparar_exogena_lag168(
                series[key],
                "Exportacion_lag168"
            )
        )

        dfs.append(
            exp
        )

    # -----------------------------------------------------
    # VALIDAR
    # -----------------------------------------------------

    if len(dfs) == 0:

        raise ValueError(
            "No hay ninguna "
            "exogena activa."
        )

    # -----------------------------------------------------
    # COMBINAR
    # -----------------------------------------------------

    exog = dfs[0]

    for df_next in (
        dfs[1:]
    ):

        exog = exog.merge(
            df_next,
            on="datetime",
            how="outer"
        )

    exog = (
        exog
        .sort_values(
            "datetime"
        )
        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # COMPLETAR HUECOS
    #
    # Mantiene la misma filosofia
    # del pipeline original.
    # -----------------------------------------------------

    exog[
        EXOG_NAMES
    ] = (
        exog[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    # -----------------------------------------------------
    # RESUMEN
    # -----------------------------------------------------

    print(
        f"\nExogenas {region}:"
    )

    for col in EXOG_NAMES:

        print(
            f"   {col:25s} | "
            f"{exog[col].notna().sum():,} "
            "valores"
        )

    print(
        f"Rango: "
        f"{exog['datetime'].min()} "
        f"-> "
        f"{exog['datetime'].max()}"
    )

    return exog


# =========================================================
# ALINEAR EXOGENAS CON DEMANDA
# =========================================================

def alinear_exogenas_con_fechas(
    fechas,
    exog_region
):

    base = pd.DataFrame({
        "datetime":
            pd.to_datetime(
                fechas
            )
    })

    X = base.merge(

        exog_region[
            ["datetime"]
            + EXOG_NAMES
        ],

        on="datetime",

        how="left"
    )

    X[
        EXOG_NAMES
    ] = (
        X[
            EXOG_NAMES
        ]
        .ffill()
        .bfill()
    )

    if (
        X[
            EXOG_NAMES
        ]
        .isna()
        .any()
        .any()
    ):

        faltantes = (
            X[
                EXOG_NAMES
            ]
            .isna()
            .sum()
            .to_dict()
        )

        raise ValueError(
            "No fue posible completar "
            f"las exogenas: "
            f"{faltantes}"
        )

    return (
        X[
            EXOG_NAMES
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


# =========================================================
# DATASET SUPERVISADO MULTIVARIADO
# =========================================================

def crear_ventanas_multivariadas(
    y,
    exog,
    window
):

    """
    Para predecir y[t]:

        y[t-window:t]

        exog[t-window:t]

        exog[t]

    IMPORTANTE:

    Las columnas electricas ya vienen
    transformadas a lag168.

    Por tanto:

        Generacion_lag168[t]
            = Generacion[t-168]

        Importacion_lag168[t]
            = Importacion[t-168]

        Exportacion_lag168[t]
            = Exportacion[t-168]

    Mientras:

        Temperaturas[t]
            = Temperatura correspondiente
              a la hora objetivo

        IGAE[t]
            = IGAE correspondiente
              a la hora objetivo
    """

    y = np.asarray(
        y,
        dtype=float
    )

    exog = np.asarray(
        exog,
        dtype=float
    )

    if len(y) != len(exog):

        raise ValueError(
            f"Longitudes incompatibles: "
            f"y={len(y)}, "
            f"exog={len(exog)}"
        )

    X = []
    Y = []

    for i in range(
        window,
        len(y)
    ):

        # -------------------------------------------------
        # 168 HORAS DE VARIABLE OBJETIVO
        # -------------------------------------------------

        y_lags = (
            y[
                i-window:
                i
            ]
            .reshape(
                -1,
                1
            )
        )

        # -------------------------------------------------
        # 168 HORAS DE CADA EXOGENA
        # -------------------------------------------------

        exog_lags = (
            exog[
                i-window:
                i
            ]
        )

        # -------------------------------------------------
        # COMBINAR VENTANA
        # -------------------------------------------------

        ventana = (
            np.concatenate(
                [
                    y_lags,
                    exog_lags
                ],
                axis=1
            )
            .ravel()
        )

        # -------------------------------------------------
        # EXOGENAS CORRESPONDIENTES A t
        #
        # Temp[t]
        # IGAE[t]
        # Gen[t-168]
        # Imp[t-168]
        # Exp[t-168]
        # -------------------------------------------------

        exog_actual = (
            exog[i]
            .ravel()
        )

        features = (
            np.concatenate(
                [
                    ventana,
                    exog_actual
                ]
            )
        )

        X.append(
            features
        )

        Y.append(
            y[i]
        )

    return (
        np.asarray(X),
        np.asarray(Y)
    )


# =========================================================
# FORECAST RECURSIVO
# =========================================================

def forecast_recursivo_fcnn_multivariada(
    model,
    y_train,
    exog_train,
    exog_future,
    horizon,
    window,
    scaler_x,
    scaler_y
):

    """
    Pronostico recursivo.

    Cada paso utiliza:

    - ultimas WINDOW horas de y
    - ultimas WINDOW filas de exogenas
    - exogenas de la hora objetivo

    Las electricas ya estan representadas
    como lag168, por lo que no existe
    leakage en las siguientes 168 horas.
    """

    historial_y = list(
        np.asarray(
            y_train,
            dtype=float
        )
    )

    historial_exog = [

        row.copy()

        for row in np.asarray(
            exog_train,
            dtype=float
        )
    ]

    exog_future = np.asarray(
        exog_future,
        dtype=float
    )

    preds = []

    if len(exog_future) < horizon:

        raise ValueError(
            "exog_future tiene "
            "menos filas que el "
            "horizonte solicitado"
        )

    for paso in range(
        horizon
    ):

        # -------------------------------------------------
        # HISTORIA DE Y
        # -------------------------------------------------

        y_lags = np.asarray(

            historial_y[
                -window:
            ],

            dtype=float

        ).reshape(
            -1,
            1
        )

        # -------------------------------------------------
        # HISTORIA DE EXOGENAS
        # -------------------------------------------------

        exog_lags = np.asarray(

            historial_exog[
                -window:
            ],

            dtype=float
        )

        # -------------------------------------------------
        # VENTANA
        # -------------------------------------------------

        ventana = (
            np.concatenate(
                [
                    y_lags,
                    exog_lags
                ],
                axis=1
            )
            .ravel()
        )

        # -------------------------------------------------
        # EXOGENAS DE LA HORA OBJETIVO
        # -------------------------------------------------

        exog_actual = (
            exog_future[
                paso
            ]
            .ravel()
        )

        x = (
            np.concatenate(
                [
                    ventana,
                    exog_actual
                ]
            )
            .reshape(
                1,
                -1
            )
        )

        x_scaled = (
            scaler_x.transform(
                x
            )
        )

        pred_scaled = (
            model.predict(
                x_scaled,
                verbose=0
            )
        )

        pred = (
            scaler_y
            .inverse_transform(
                pred_scaled.reshape(
                    -1,
                    1
                )
            )[0, 0]
        )

        preds.append(
            pred
        )

        # -------------------------------------------------
        # RECURSION
        # -------------------------------------------------

        historial_y.append(
            pred
        )

        historial_exog.append(
            exog_actual.copy()
        )

    return np.asarray(
        preds
    )


# =========================================================
# MODELO FCNN
#
# ARQUITECTURA ORIGINAL
# =========================================================

def construir_fcnn(
    params,
    input_dim
):

    model = Sequential()

    model.add(
        Input(
            shape=(
                input_dim,
            )
        )
    )

    model.add(
        Dense(
            params[
                "units_1"
            ],
            activation="relu"
        )
    )

    model.add(
        Dropout(
            params[
                "dropout"
            ]
        )
    )

    if (
        params[
            "n_layers"
        ]
        == 2
    ):

        model.add(
            Dense(
                params[
                    "units_2"
                ],
                activation="relu"
            )
        )

        model.add(
            Dropout(
                params[
                    "dropout"
                ]
            )
        )

    model.add(
        Dense(1)
    )

    model.compile(

        optimizer=Adam(
            learning_rate=(
                params[
                    "learning_rate"
                ]
            )
        ),

        loss="mse"
    )

    return model


# =========================================================
# OPTUNA
# =========================================================

def tunear_fcnn_optuna(
    train_y,
    train_exog,
    nombre_serie,
    modelo_nombre
):

    X, y = (
        crear_ventanas_multivariadas(
            train_y,
            train_exog,
            WINDOW
        )
    )

    if len(X) <= TEST_HORIZON:

        raise ValueError(
            "No hay suficientes "
            "ventanas para separar "
            "entrenamiento y validacion."
        )

    val_size = TEST_HORIZON

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        scaler_x.fit_transform(
            X_train
        )
    )

    X_val_s = (
        scaler_x.transform(
            X_val
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    input_dim = (
        X_train.shape[1]
    )

    def objective(
        trial
    ):

        tf.keras.backend.clear_session()

        params = {

            "n_layers":
                trial.suggest_categorical(
                    "n_layers",
                    [
                        1,
                        2
                    ]
                ),

            "units_1":
                trial.suggest_categorical(
                    "units_1",
                    [
                        32,
                        64,
                        128,
                        256
                    ]
                ),

            "dropout":
                trial.suggest_float(
                    "dropout",
                    0.0,
                    0.4
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    1e-4,
                    3e-3,
                    log=True
                ),

            "batch_size":
                trial.suggest_categorical(
                    "batch_size",
                    [
                        32,
                        64,
                        128
                    ]
                ),
        }

        if (
            params[
                "n_layers"
            ]
            == 2
        ):

            params[
                "units_2"
            ] = (
                trial.suggest_categorical(
                    "units_2",
                    [
                        16,
                        32,
                        64,
                        128
                    ]
                )
            )

        else:

            params[
                "units_2"
            ] = 0

        model = (
            construir_fcnn(
                params,
                input_dim=(
                    input_dim
                )
            )
        )

        early = EarlyStopping(

            monitor="val_loss",

            patience=8,

            restore_best_weights=True
        )

        model.fit(

            X_train_s,

            y_train_s,

            validation_data=(
                X_val_s,
                y_val_s
            ),

            epochs=EPOCHS,

            batch_size=(
                params[
                    "batch_size"
                ]
            ),

            verbose=0,

            callbacks=[
                early
            ]
        )

        pred_val_s = (
            model.predict(
                X_val_s,
                verbose=0
            )
        )

        pred_val = (
            scaler_y
            .inverse_transform(
                pred_val_s.reshape(
                    -1,
                    1
                )
            )
            .ravel()
        )

        return smape(
            y_val,
            pred_val
        )

    db_path = os.path.join(

        OUTPUT_DIR,

        f"{nombre_serie}_"
        f"{modelo_nombre}_optuna.db"
    )

    study = (
        optuna.create_study(

            direction="minimize",

            study_name=(
                f"{nombre_serie}_"
                f"{modelo_nombre}"
            ),

            storage=(
                f"sqlite:///"
                f"{db_path}"
            ),

            load_if_exists=True
        )
    )

    completed = len([

        t

        for t in (
            study.trials
        )

        if (
            t.state
            == optuna.trial.TrialState.COMPLETE
        )
    ])

    remaining = max(
        0,
        N_TRIALS
        - completed
    )

    print(
        f"      Trials completos: "
        f"{completed}"
    )

    print(
        f"      Trials restantes: "
        f"{remaining}"
    )

    if remaining > 0:

        study.optimize(
            objective,
            n_trials=remaining
        )

    return (
        study.best_params,
        study.best_value
    )


# =========================================================
# ENTRENAR FCNN FINAL
# =========================================================

def entrenar_fcnn_final(
    train_y,
    train_exog,
    params
):

    X, y = (
        crear_ventanas_multivariadas(
            train_y,
            train_exog,
            WINDOW
        )
    )

    if len(X) <= TEST_HORIZON:

        raise ValueError(
            "No hay suficientes "
            "ventanas para separar "
            "entrenamiento y validacion."
        )

    val_size = TEST_HORIZON

    X_train = (
        X[
            :-val_size
        ]
    )

    y_train = (
        y[
            :-val_size
        ]
    )

    X_val = (
        X[
            -val_size:
        ]
    )

    y_val = (
        y[
            -val_size:
        ]
    )

    scaler_x = (
        StandardScaler()
    )

    scaler_y = (
        StandardScaler()
    )

    X_train_s = (
        scaler_x.fit_transform(
            X_train
        )
    )

    X_val_s = (
        scaler_x.transform(
            X_val
        )
    )

    y_train_s = (
        scaler_y
        .fit_transform(
            y_train.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    y_val_s = (
        scaler_y
        .transform(
            y_val.reshape(
                -1,
                1
            )
        )
        .ravel()
    )

    final_params = {

        "n_layers":
            params[
                "n_layers"
            ],

        "units_1":
            params[
                "units_1"
            ],

        "dropout":
            params[
                "dropout"
            ],

        "learning_rate":
            params[
                "learning_rate"
            ],

        "batch_size":
            params[
                "batch_size"
            ],

        "units_2":
            params.get(
                "units_2",
                0
            ),
    }

    model = (
        construir_fcnn(
            final_params,
            input_dim=(
                X_train.shape[1]
            )
        )
    )

    early = EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True
    )

    history = model.fit(

        X_train_s,

        y_train_s,

        validation_data=(
            X_val_s,
            y_val_s
        ),

        epochs=EPOCHS,

        batch_size=(
            final_params[
                "batch_size"
            ]
        ),

        verbose=0,

        callbacks=[
            early
        ]
    )

    return (
        model,
        scaler_x,
        scaler_y,
        history
    )


# =========================================================
# STL
# =========================================================

def forecast_tendencia_lineal(
    trend_train,
    horizon
):

    trend_train = np.asarray(
        trend_train,
        dtype=float
    )

    x = np.arange(
        len(
            trend_train
        )
    ).reshape(
        -1,
        1
    )

    modelo = (
        LinearRegression()
    )

    modelo.fit(
        x,
        trend_train
    )

    x_future = np.arange(

        len(
            trend_train
        ),

        len(
            trend_train
        )
        + horizon

    ).reshape(
        -1,
        1
    )

    return (
        modelo.predict(
            x_future
        )
    )


def forecast_estacionalidad_repetida(
    seasonal_train,
    horizon,
    period=168
):

    ultimos = np.asarray(

        seasonal_train[
            -period:
        ],

        dtype=float
    )

    reps = int(
        np.ceil(
            horizon
            / period
        )
    )

    return (
        np.tile(
            ultimos,
            reps
        )[:horizon]
    )


def descomponer_stl(
    train
):

    stl = STL(

        pd.Series(
            train
        ).astype(float),

        period=STL_PERIOD,

        robust=True
    )

    res = stl.fit()

    return (

        np.asarray(
            res.trend
        ),

        np.asarray(
            res.seasonal
        ),

        np.asarray(
            res.resid
        ),

        res
    )


# =========================================================
# GUARDADO
# =========================================================

def guardar_resultados(
    nombre_serie,
    serie,
    fechas,
    fechas_test,
    resultados
):

    serie_dir = os.path.join(
        INCREMENTAL_DIR,
        nombre_serie
    )

    os.makedirs(
        serie_dir,
        exist_ok=True
    )

    series_dfs = [

        pd.DataFrame({

            "serie":
                nombre_serie,

            "fecha":
                pd.to_datetime(
                    fechas
                ),

            "tipo":
                "real",

            "subset":
                "completo",

            "modelo":
                "real",

            "valor":
                serie,
        })
    ]

    metricas_rows = []

    for res in resultados:

        modelo = res[
            "modelo"
        ]

        pred = res[
            "pred"
        ]

        series_dfs.append(

            pd.DataFrame({

                "serie":
                    nombre_serie,

                "fecha":
                    pd.to_datetime(
                        fechas_test
                    ),

                "tipo":
                    "prediccion",

                "subset":
                    "test",

                "modelo":
                    modelo,

                "valor":
                    pred,
            })
        )

        metricas_rows.append({

            "serie":
                nombre_serie,

            "modelo":
                modelo,

            "MAE":
                res[
                    "metricas"
                ][
                    "MAE"
                ],

            "RMSE":
                res[
                    "metricas"
                ][
                    "RMSE"
                ],

            "MAPE":
                res[
                    "metricas"
                ][
                    "MAPE"
                ],

            "sMAPE":
                res[
                    "metricas"
                ][
                    "sMAPE"
                ],

            "best_val_sMAPE":
                res[
                    "best_val_smape"
                ],

            "params":
                json.dumps(
                    res[
                        "params"
                    ]
                ),

            "exogenas":
                ", ".join(
                    EXOG_NAMES
                ),

            "train_horas":
                TRAIN_LAST_HOURS,

            "test_horas":
                TEST_HORIZON,

            "window":
                WINDOW,

            "stl_period":
                STL_PERIOD,

            "trials":
                N_TRIALS,

            "lag_electricas":
                ELECTRIC_LAG,
        })

    df_series = pd.concat(
        series_dfs,
        ignore_index=True
    )

    df_metricas = (
        pd.DataFrame(
            metricas_rows
        )
    )

    path_series = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_FCNN_multivariada_series.csv"
    )

    path_metricas = os.path.join(

        serie_dir,

        f"{nombre_serie}"
        "_FCNN_multivariada_metricas.csv"
    )

    df_series.to_csv(

        path_series,

        index=False,

        encoding="utf-8-sig"
    )

    df_metricas.to_csv(

        path_metricas,

        index=False,

        encoding="utf-8-sig"
    )

    print(
        f"💾 Guardado: "
        f"{serie_dir}"
    )

    tf.keras.backend.clear_session()
    gc.collect()

    return (
        df_series,
        df_metricas
    )


# =========================================================
# EVALUAR UNA REGION
# =========================================================

def evaluar_region(
    region,
    archivo,
    exog_region
):

    path = os.path.join(
        DATA_DIR,
        archivo
    )

    if not os.path.exists(
        path
    ):

        print(
            f"⚠️ No encontre "
            f"{path}"
        )

        return (
            None,
            None
        )

    df = pd.read_csv(
        path
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    if (
        COL_DEMANDA
        not in df.columns
    ):

        print(
            f"⚠️ No existe "
            f"{COL_DEMANDA} "
            f"en {region}"
        )

        return (
            None,
            None
        )

    nombre_serie = (
        f"{region}_DEMANDA"
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"🔥 Evaluando "
        f"{nombre_serie}"
    )

    print(
        "=" * 80
    )

    # -----------------------------------------------------
    # DEMANDA
    # -----------------------------------------------------

    serie, fechas = (
        extraer_serie_horaria(
            df,
            COL_DEMANDA
        )
    )

    # -----------------------------------------------------
    # EXOGENAS
    # -----------------------------------------------------

    X_exog = (
        alinear_exogenas_con_fechas(
            fechas,
            exog_region
        )
    )

    requeridas = (
        TRAIN_LAST_HOURS
        + TEST_HORIZON
    )

    if len(
        serie
    ) < requeridas:

        print(
            f"⚠️ {nombre_serie} "
            f"tiene {len(serie):,} horas; "
            f"se requieren al menos "
            f"{requeridas:,}."
        )

        return (
            None,
            None
        )

    # -----------------------------------------------------
    # PERIODO RECIENTE
    # -----------------------------------------------------

    serie_reciente = (
        serie[
            -requeridas:
        ]
    )

    fechas_recientes = (
        fechas[
            -requeridas:
        ]
    )

    X_reciente = (

        X_exog

        .iloc[
            -requeridas:
        ]

        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # TRAIN / TEST
    # -----------------------------------------------------

    train = (
        serie_reciente[
            :-TEST_HORIZON
        ]
    )

    test = (
        serie_reciente[
            -TEST_HORIZON:
        ]
    )

    fechas_test = (
        fechas_recientes[
            -TEST_HORIZON:
        ]
    )

    X_train = (

        X_reciente

        .iloc[
            :-TEST_HORIZON
        ]

        .reset_index(
            drop=True
        )
    )

    X_test = (

        X_reciente

        .iloc[
            -TEST_HORIZON:
        ]

        .reset_index(
            drop=True
        )
    )

    # -----------------------------------------------------
    # DIAGNOSTICO
    # -----------------------------------------------------

    print(
        f"Train: "
        f"{len(train):,}"
    )

    print(
        f"Test:  "
        f"{len(test):,}"
    )

    print(
        f"Exogenas train: "
        f"{X_train.shape}"
    )

    print(
        f"Exogenas test:  "
        f"{X_test.shape}"
    )

    print(
        "\nDefinicion exogenas:"
    )

    for col in EXOG_NAMES:

        if col in [
            "Temperaturas",
            "IGAE"
        ]:

            print(
                f"   {col}: "
                "contemporanea"
            )

        else:

            print(
                f"   {col}: "
                f"lag {ELECTRIC_LAG}h"
            )

    resultados = []


    # =====================================================
    # 1. FCNN MULTIVARIADA SOBRE DEMANDA
    # =====================================================

    try:

        print(
            "\n   🔎 Optuna FCNN "
            "multivariada sobre demanda..."
        )

        (
            best_params,
            best_smape
        ) = (
            tunear_fcnn_optuna(

                train_y=train,

                train_exog=(
                    X_train
                ),

                nombre_serie=(
                    nombre_serie
                ),

                modelo_nombre=(
                    "FCNN_multivariada"
                )
            )
        )

        print(
            f"      Mejor sMAPE "
            f"validacion: "
            f"{best_smape:.4f}%"
        )

        print(
            f"      Params: "
            f"{best_params}"
        )

        (
            model,
            scaler_x,
            scaler_y,
            history
        ) = (
            entrenar_fcnn_final(

                train_y=train,

                train_exog=(
                    X_train
                ),

                params=(
                    best_params
                )
            )
        )

        pred = (
            forecast_recursivo_fcnn_multivariada(

                model=model,

                y_train=train,

                exog_train=(
                    X_train
                ),

                exog_future=(
                    X_test
                ),

                horizon=len(
                    test
                ),

                window=WINDOW,

                scaler_x=(
                    scaler_x
                ),

                scaler_y=(
                    scaler_y
                )
            )
        )

        metricas = (
            calcular_metricas(
                test,
                pred
            )
        )

        print(
            f"      FCNN multivariada "
            f"MAPE="
            f"{metricas['MAPE']:.2f}%"
        )

        resultados.append({

            "modelo":
                "FCNN_Multivariada_EXOG_Lag168",

            "pred":
                pred,

            "metricas":
                metricas,

            "params":
                best_params,

            "best_val_smape":
                best_smape,
        })

        del (
            model,
            scaler_x,
            scaler_y,
            history
        )

        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:

        print(
            f"❌ Error FCNN "
            f"multivariada en "
            f"{nombre_serie}: "
            f"{type(e).__name__}: "
            f"{e}"
        )


    # =====================================================
    # 2. STL + FCNN MULTIVARIADA SOBRE RESIDUOS
    # =====================================================

    try:

        print(
            "\n   🔎 STL + Optuna "
            "FCNN multivariada "
            "sobre residuos..."
        )

        # -------------------------------------------------
        # STL SOLO SOBRE TRAIN
        # -------------------------------------------------

        (
            trend,
            seasonal,
            resid,
            stl_res
        ) = (
            descomponer_stl(
                train
            )
        )

        # -------------------------------------------------
        # TENDENCIA FUTURA
        # -------------------------------------------------

        trend_forecast = (
            forecast_tendencia_lineal(
                trend,
                horizon=len(
                    test
                )
            )
        )

        # -------------------------------------------------
        # ESTACIONALIDAD FUTURA
        # -------------------------------------------------

        seasonal_forecast = (
            forecast_estacionalidad_repetida(

                seasonal,

                horizon=len(
                    test
                ),

                period=(
                    STL_PERIOD
                )
            )
        )

        # -------------------------------------------------
        # RESIDUOS + EXOGENAS
        # -------------------------------------------------

        (
            best_params_res,
            best_smape_res
        ) = (
            tunear_fcnn_optuna(

                train_y=resid,

                train_exog=(
                    X_train
                ),

                nombre_serie=(
                    nombre_serie
                ),

                modelo_nombre=(
                    "FCNN_multivariada_"
                    "residuos_STL"
                )
            )
        )

        print(
            f"      Mejor sMAPE "
            f"residuos validacion: "
            f"{best_smape_res:.4f}%"
        )

        print(
            f"      Params residuos: "
            f"{best_params_res}"
        )

        (
            model_res,
            scaler_x_res,
            scaler_y_res,
            history_res
        ) = (
            entrenar_fcnn_final(

                train_y=resid,

                train_exog=(
                    X_train
                ),

                params=(
                    best_params_res
                )
            )
        )

        resid_pred = (
            forecast_recursivo_fcnn_multivariada(

                model=(
                    model_res
                ),

                y_train=resid,

                exog_train=(
                    X_train
                ),

                exog_future=(
                    X_test
                ),

                horizon=len(
                    test
                ),

                window=WINDOW,

                scaler_x=(
                    scaler_x_res
                ),

                scaler_y=(
                    scaler_y_res
                )
            )
        )

        # -------------------------------------------------
        # RECONSTRUCCION
        # -------------------------------------------------

        pred_residuos = (
            trend_forecast
            + seasonal_forecast
            + resid_pred
        )

        metricas_res = (
            calcular_metricas(
                test,
                pred_residuos
            )
        )

        print(
            f"      STL + FCNN "
            f"multivariada residuos "
            f"MAPE="
            f"{metricas_res['MAPE']:.2f}%"
        )

        resultados.append({

            "modelo":
                (
                    "STL_FCNN_Multivariada_"
                    "Residuos_EXOG_Lag168"
                ),

            "pred":
                pred_residuos,

            "metricas":
                metricas_res,

            "params":
                best_params_res,

            "best_val_smape":
                best_smape_res,
        })

        del (
            model_res,
            scaler_x_res,
            scaler_y_res,
            history_res,
            stl_res
        )

        tf.keras.backend.clear_session()
        gc.collect()

    except Exception as e:

        print(
            f"❌ Error STL + FCNN "
            f"multivariada sobre residuos "
            f"en {nombre_serie}: "
            f"{type(e).__name__}: "
            f"{e}"
        )


    # =====================================================
    # GUARDAR REGION
    # =====================================================

    if len(
        resultados
    ) == 0:

        return (
            None,
            None
        )

    return guardar_resultados(

        nombre_serie=(
            nombre_serie
        ),

        serie=serie,

        fechas=fechas,

        fechas_test=(
            fechas_test
        ),

        resultados=(
            resultados
        )
    )


# =========================================================
# EJECUTAR TODAS LAS REGIONES
# =========================================================

todas_series = []
todas_metricas = []

print(
    "=" * 80
)

print(
    "FCNN MULTIVARIADA "
    "+ STL FCNN RESIDUOS"
)

print(
    "=" * 80
)

print(
    f"Train: "
    f"{TRAIN_LAST_HOURS} h"
)

print(
    f"Test: "
    f"{TEST_HORIZON} h"
)

print(
    f"Window: "
    f"{WINDOW} h"
)

print(
    "\nExogenas activas:"
)

for exog in (
    EXOG_NAMES
):

    print(
        f"   - {exog}"
    )


# =========================================================
# LOOP
# =========================================================

for (
    region,
    archivo
) in (
    ARCHIVOS_REGIONES.items()
):

    try:

        # -------------------------------------------------
        # MATRIZ EXOGENA ESPECIFICA
        # DE LA REGION
        # -------------------------------------------------

        exog_region = (
            construir_matriz_exogena_region(
                region
            )
        )

        (
            df_series,
            df_metricas
        ) = (
            evaluar_region(

                region=region,

                archivo=archivo,

                exog_region=(
                    exog_region
                )
            )
        )

        if (
            df_series
            is not None
        ):

            todas_series.append(
                df_series
            )

        if (
            df_metricas
            is not None
        ):

            todas_metricas.append(
                df_metricas
            )

    except Exception as e:

        print(
            f"❌ Error general "
            f"en {region}: "
            f"{type(e).__name__}: "
            f"{e}"
        )

        continue


# =========================================================
# GUARDAR RESULTADOS GLOBALES
# =========================================================

if len(
    todas_series
) > 0:

    df_series_global = (
        pd.concat(
            todas_series,
            ignore_index=True
        )
    )

    path_series_global = (
        os.path.join(

            OUTPUT_DIR,

            "FCNN_multivariada_"
            "EXOG_Lag168_"
            "series_global.csv"
        )
    )

    df_series_global.to_csv(

        path_series_global,

        index=False,

        encoding="utf-8-sig"
    )

    print(
        "\n✅ Series globales "
        "guardadas:"
    )

    print(
        path_series_global
    )


if len(
    todas_metricas
) > 0:

    df_metricas_global = (
        pd.concat(
            todas_metricas,
            ignore_index=True
        )
    )

    path_metricas_global = (
        os.path.join(

            OUTPUT_DIR,

            "FCNN_multivariada_"
            "EXOG_Lag168_"
            "metricas_global.csv"
        )
    )

    df_metricas_global.to_csv(

        path_metricas_global,

        index=False,

        encoding="utf-8-sig"
    )

    print(
        "\n✅ Metricas globales "
        "guardadas:"
    )

    print(
        path_metricas_global
    )

    display(

        df_metricas_global

        .sort_values(
            [
                "serie",
                "MAPE"
            ]
        )
    )

    print(
        "\n📊 Promedio por modelo:"
    )

    display(

        df_metricas_global

        .groupby(
            "modelo"
        )[
            "MAPE"
        ]

        .agg(
            [
                "mean",
                "std"
            ]
        )

        .sort_values(
            "mean"
        )
    )